# ManifoldEM on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatironinstitute/ManifoldEM/blob/main/tutorial/ManifoldEM_Colab.ipynb)

[ManifoldEM](https://github.com/flatironinstitute/ManifoldEM) recovers **continuous conformational changes** of a
macromolecule from single-particle cryo-EM images. It analyses the particles of each projection direction (PD) with
diffusion maps and nonlinear Laplacian spectral analysis (NLSA), stitches the directions together into a
conformational coordinate, and reconstructs a **free-energy profile** and a **3D movie** of the motion.

This notebook runs the complete pipeline in the browser, with no local installation and no GUI. It follows the
[command-line tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md) and adds viewers
for the decisions the GUI normally helps you make: thresholds, ψ movies, anchors. Manual re-embedding of a PD (cutting
clusters out of its 2D embedding) still needs the desktop GUI (`manifold-gui`).

**Quick start:** choose **Runtime ▸ Run all**. With the default settings the notebook builds a small synthetic
dataset with a *known* motion and takes it through every step in about 10–15 minutes on the free tier. The last cell
then downloads a small zip of the results. Afterwards, run it again with your own data.

| Section | What happens | ManifoldEM step | Time (synthetic demo, 2 CPUs) |
|---|---|---|---|
| **0 · Setup** | install ManifoldEM in its own environment | – | 2–4 min |
| **1 · Data** | synthetic demo, RyR1 tutorial data or your own STAR + MRCS | – | ~1 min |
| **2 · Project** | create (or re-open) a project | `init` | seconds |
| **3 · Projection directions** | bin particles by orientation, set occupancy thresholds | `threshold` | seconds |
| **4 · Manifold embedding** | distances, diffusion maps, NLSA, movies for every PD | `calc-distance` … `nlsa-movie` | 3–5 min |
| **5 · Explore** | browse PDs: class averages, eigenvalues, NLSA movies | (GUI *Eigenvectors* tab) | – |
| **6 · Anchors** | pick reference PDs and the direction of the motion | (GUI *Set PD Anchors*) | – |
| **7 · Coordinate & landscape** | belief propagation, free-energy profile, per-state image stacks | `find-ccs`, `probability-landscape`, `trajectory` | ~1 min |
| **8 · Volumes** | reconstruct and view a 3D movie of the conformational change | `utility mrcs2mrc` | ~1 min |
| **9 · Python API** | run your own code against ManifoldEM's `data_store` | – | – |
| **10 · Save** | checkpoint to Google Drive, download results | – | – |

**How the notebook works**
* Every step is a cell with a small form. Change values in the form, not in the code. Each cell can be re-run.
* The notebook keeps track of what is up to date. When you change something (thresholds, parameters, anchors), the
  results that depend on it are marked ⚠️ *out of date* and later steps refuse to use them until you re-run the step
  in between. Cells 7.1–8.1 skip work that is already up to date (tick `RECOMPUTE` to force it); 4.1 skips PDs
  that are finished (untick `ONLY_MISSING_PDS` to redo them); 8.2 takes seconds and always reruns.
* PD numbers and ψ (eigenvector) numbers start at **1**, as in the GUI and in the output folders `topos/PrD_<n>/psi_<k>.gif`.
* **If the session ends.** After *Runtime ▸ Restart session* (or a crash) your files are still there: run
  section 0, then **2.2 Re-open a project** (Run all also works: 2.1 re-opens a project that already exists with the
  same inputs). After a *disconnect* (idle timeout, the 12-hour limit, closing the
  tab for long) Colab deletes the runtime's disk, and only what you saved to Google Drive with **10.1** survives. 10.1
  can be run at any time. On real data, run it after section 4 and again after sections 7–8.
* **Runtime:** ManifoldEM is CPU-only, so a GPU runtime does not help. The free tier has 2 CPUs and ~12 GB of RAM,
  enough for the demo and for small or downsampled datasets. The cost of section 4 grows with the number of PDs ×
  (images per PD)² × (box size)², so real datasets take hours rather than minutes on 2 CPUs. Downsampling by 2 (1.5)
  makes it roughly 4–8× faster. For full-size data use a high-RAM runtime with more CPUs (Colab Pro) or a workstation
  (see the main README), and save checkpoints often.

*Please cite:* A. Ojha *et al.*, *The ManifoldEM method for cryo-EM: a step-by-step breakdown accompanied by a modern
Python implementation*, Acta Cryst. D81, 89–104 (2025), [doi:10.1107/S2059798325001469](https://doi.org/10.1107/S2059798325001469).

## 0 · Setup

In [ ]:
#@title 0.1 · Check the runtime { display-mode: "form" }
import os, sys, shutil, platform, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
ROOT = Path('/content/manifoldem') if IN_COLAB else Path.cwd() / 'manifoldem_colab'
ENV_DIR = ROOT / 'env'          # ManifoldEM's own Python environment
ROOT.mkdir(parents=True, exist_ok=True)

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
except ImportError:
    RAM_GB = float('nan')
print(f"Python {platform.python_version()} · {os.cpu_count()} CPUs · {RAM_GB:.1f} GB RAM · "
      f"{shutil.disk_usage(ROOT).free / 1e9:.0f} GB free disk")
print(f"Working folder: {ROOT}")
if shutil.which('nvidia-smi'):
    print("ℹ️  A GPU is attached, but ManifoldEM only uses CPUs (Runtime ▸ Change runtime type ▸ CPU is fine).")
if (os.cpu_count() or 1) <= 2:
    print("ℹ️  2 CPUs: fine for the demo and small datasets; a high-RAM runtime with more CPUs is much faster for real data.")

In [ ]:
#@title 0.2 · Install ManifoldEM (≈ 2–4 min, once per session) { display-mode: "form" }
#@markdown ManifoldEM is installed from GitHub into its own Python 3.12 environment, because it needs NumPy < 2
#@markdown while Colab runs NumPy 2. Colab's own packages are left untouched, so **no runtime restart is needed**.
MANIFOLDEM_REPO = "https://github.com/flatironinstitute/ManifoldEM"  #@param {type:"string"}
MANIFOLDEM_VERSION = "v0.4.0"  #@param {type:"string"}
#@markdown `MANIFOLDEM_VERSION` is a tag, branch or commit (`main` = latest development version). Use your fork's URL to test your own changes.
REINSTALL = False  #@param {type:"boolean"}
import json, time, importlib.util

SUBPROCESS_ENV = dict(os.environ, MPLBACKEND='Agg')   # the kernel's inline backend does not exist in ManifoldEM's env

def _run(cmd, quiet=True):
    r = subprocess.run([str(c) for c in cmd], capture_output=quiet, text=True, env=SUBPROCESS_ENV)
    if r.returncode:
        print((r.stdout or '')[-3000:], (r.stderr or '')[-3000:])
        raise RuntimeError(f"failed: {' '.join(map(str, cmd))}")
    return r.stdout if quiet else ''

ref = MANIFOLDEM_VERSION.strip()
SPEC = f"manifoldem @ git+{MANIFOLDEM_REPO.strip().rstrip('/')}" + (f"@{ref}" if ref else "")
PY = ENV_DIR / 'bin' / 'python'
marker = ENV_DIR / '.installed.json'
if marker.exists() and json.loads(marker.read_text()).get('spec') == SPEC and not REINSTALL:
    INSTALLED = json.loads(marker.read_text())
    print(f"✔ already installed: ManifoldEM {INSTALLED['manifoldem']} ({SPEC})")
else:
    t0 = time.time()
    uv = [shutil.which('uv')] if shutil.which('uv') else None
    if uv is None:
        _run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
        uv = [sys.executable, '-m', 'uv']
    if ENV_DIR.exists():
        shutil.rmtree(ENV_DIR)
    print('· creating a Python 3.12 environment ...')
    _run(uv + ['venv', '--seed', '--python', '3.12', ENV_DIR])
    print('· installing the scientific stack (NumPy < 2) ...')
    _run(uv + ['pip', 'install', '--python', PY, 'numpy<2', 'scipy', 'numba', 'pandas', 'scikit-learn',
               'scikit-image', 'h5py', 'imageio', 'mrcfile', 'nptyping', 'fasthog', 'toml', 'tqdm',
               'matplotlib', 'threadpoolctl', 'setuptools', 'setuptools-scm'])
    print(f'· installing {SPEC} ...')
    _run([PY, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', '--progress-bar', 'off', 'numpy<2', SPEC])
    check = ("import json, numpy, cv2, ManifoldEM, ManifoldEM.calc_distance, ManifoldEM.find_conformational_coords;"
             "o = cv2.optflow; [getattr(o, n) for n in ('createOptFlow_DeepFlow', 'calcOpticalFlowDenseRLOF')];"
             "print(json.dumps(dict(manifoldem=ManifoldEM.__version__, numpy=numpy.__version__, cv2=cv2.__version__)))")
    try:
        out = _run([PY, '-c', check])
    except RuntimeError:
        # opencv-python-headless and opencv-contrib-python share one cv2/ folder; the contrib build must win
        ver = _run([PY, '-c', 'import importlib.metadata as m; print(m.version("opencv-contrib-python"))']).strip()
        _run([PY, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-contrib-python=={ver}'])
        out = _run([PY, '-c', check])
    INSTALLED = dict(json.loads(out.strip().splitlines()[-1]), spec=SPEC)
    marker.write_text(json.dumps(INSTALLED))
    print(f"✔ ManifoldEM {INSTALLED['manifoldem']} installed in {(time.time() - t0) / 60:.1f} min "
          f"(numpy {INSTALLED['numpy']}, OpenCV {INSTALLED['cv2']})")
if importlib.util.find_spec('mrcfile') is None:   # small, NumPy-2 compatible; used to display volumes
    _run([sys.executable, '-m', 'pip', 'install', '-q', 'mrcfile'])
_ = subprocess.run([str(ENV_DIR / 'bin' / 'manifold-cli')], capture_output=True, env=SUBPROCESS_ENV)  # warm-up: first import compiles bytecode

In [ ]:
#@title 0.3 · Write the ManifoldEM bridge script { display-mode: "form" }
#@markdown Small helper that runs inside ManifoldEM's environment and answers the notebook's questions (PD
#@markdown summaries, anchors, input checks, ...). Expand the cell to read it.
BRIDGE_SOURCE = r'''"""Bridge between the Colab notebook and ManifoldEM.

ManifoldEM needs numpy<2, while Colab's own Python runs numpy 2, so the notebook keeps ManifoldEM in a
separate environment and talks to it through this script:

    <env>/bin/python mem_bridge.py <command> <args.json> <result.json>

Each command reads keyword arguments from args.json and writes a JSON result (plus, for some
commands, an .npz file of arrays next to it).  Nothing here is needed when you use ManifoldEM from a
normal terminal; it only wraps the documented `params` / `data_store` API and works around a few
sharp edges (see the notes on each command).
"""
import glob
import inspect
import json
import os
import pickle
import re
import shutil
import sys
import time

import numpy as np

COMMANDS = {}


class UserError(Exception):
    """A problem with the user's input or project state (reported without a traceback)."""


def command(func):
    COMMANDS[func.__name__.replace('_', '-')] = func
    return func


_RESTORE = 'notebook_restore.json'  # parameters that find_ccs changed temporarily, put back if it was killed
_REBIN = 'notebook_rebin_pending.json'  # an aperture / tessellation change whose re-binning has not finished
_CLEANED = 'notebook_rebin_cleaned'      # a confirmed re-binning has deleted sections 4-8 (reported if it was stopped)


def _load(toml):
    from ManifoldEM.params import params
    params.load(os.path.abspath(toml))  # note: chdir()s into the project folder
    restore = os.path.join(params.out_dir, _RESTORE)
    if os.path.isfile(restore):
        with open(restore) as f:
            for key, value in json.load(f).items():
                setattr(params, key, value)
        params.save()
        os.remove(restore)
    params.create_dir()  # the folder skeleton, in case a stopped clean-up removed part of it
    return params


def _atomic_pd_save(self):
    """data_store's save(), but via a temporary file: a stop while it is written cannot truncate pd_data.pkl."""
    from ManifoldEM.params import params
    tmp = params.pd_file + '.part'
    with open(tmp, 'wb') as f:
        pickle.dump(self.__dict__, f, pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, params.pd_file)


def _prds():
    from ManifoldEM import data_store as ds
    from ManifoldEM.params import params
    ds._ProjectionDirections.save = _atomic_pd_save  # also used by data_store's own rebuild and %%manifoldem
    if os.path.isfile(params.pd_file + '.part'):     # left by a save that was stopped
        os.remove(params.pd_file + '.part')
    pending = os.path.join(params.out_dir, _REBIN)
    if os.path.isfile(pending):  # data_store only notices threshold changes by itself
        os.environ['MANIFOLD_REBUILD_DS'] = '1'
    try:
        prds = ds.data_store.get_prds()
    except (pickle.UnpicklingError, EOFError) as exc:
        raise UserError(f'{params.pd_file} is damaged ({exc}; probably stopped while it was being saved). Delete it '
                        'and run 2.2 again: the binning is rebuilt identically, so the results of section 4 stay '
                        'valid, but the anchors and removed PDs must be entered again in 6.2.')
    if os.path.isfile(pending):
        os.remove(pending)
    return prds


def _cc_bug():
    """True if belief propagation reads Anchor.CC as 0-based (ManifoldEM <= 0.4.0, see find_ccs)."""
    from ManifoldEM.CC import runGlobalOptimization
    return '[anch_list[anchor_id].CC, index]' in inspect.getsource(runGlobalOptimization)


# --------------------------------------------------------------------------------------------- project
_A, _B = 987654, 876543  # sentinels used to turn ManifoldEM's path functions into templates


def _paths(params):
    def tmpl(path):
        return os.path.abspath(path).replace(str(_A), '{p}').replace(str(_B), '{k}')

    topo_dir = os.path.dirname(tmpl(params.get_psi_gif(_A, _B)))
    return dict(
        work=os.getcwd(), toml=os.path.abspath(params.proj_file), out_dir=os.path.abspath(params.out_dir),
        pd_file=os.path.abspath(params.pd_file), CC_dir=os.path.abspath(params.CC_dir),
        CC_file=os.path.abspath(params.CC_file), CC_OF_dir=os.path.abspath(params.CC_OF_dir),
        traj=os.path.abspath(f'{params.traj_file}name{params.traj_name}.pkl'),
        traj_vars=os.path.abspath(f'{params.traj_file}name{params.traj_name}_vars.pkl'),
        OM_file=os.path.abspath(params.OM_file), bin_dir=os.path.abspath(params.bin_dir),
        vols_dir=os.path.abspath(params.postproc_mrcs2mrc_dir),
        denoise_dir=os.path.abspath(params.postproc_denoise_dir),
        # templates: {p} = 0-based PD index unless noted, {k} = 0-based psi index unless noted
        dist=tmpl(params.get_dist_file(_A)), psi=tmpl(params.get_psi_file(_A)),
        psi2=tmpl(params.get_psi2_file(_A, _B)), EL=tmpl(params.get_EL_file(_A)),
        gif=tmpl(params.get_psi_gif(_A, _B)),  # 1-based PD and psi
        topo=tmpl(params.get_topos_path(_A, _B)),  # 1-based PD and psi
        class_avg=os.path.join(topo_dir, 'class_avg.png'),  # 1-based PD
        eig_spec=os.path.join(topo_dir, 'eig_spec.txt'),  # 1-based PD
    )


# ------------------------------------------------------------------------------------ stage ledger
# notebook_ledger.json records, for each stage, the identity of the inputs it was computed from. A stage counts as
# done only while those inputs are unchanged and its files exist. Every stage removes its record before it starts,
# so an interrupted run is never taken for a finished one. Ids are content-based where that is cheap, so an
# identical re-run (e.g. 7.1 giving the same coordinate) keeps everything after it valid.
STAGES = ('embedding', 'cc', 'landscape', 'trajectory', 'volumes', 'denoise')
_UPSTREAM = dict(cc='embedding', landscape='cc', trajectory='landscape', volumes='trajectory', denoise='volumes')
_WHY = dict(binning='the particles were re-binned (3.1)', embedding='section 4 was recomputed',
            anchors='the anchors or removed PDs changed (6.2)', cc='the coordinate (7.1) changed', states='STATES changed',
            landscape='the landscape (7.2) changed', width='PATH_WIDTH changed', trajectory='the trajectories (7.3) changed',
            volumes='the volumes (8.1) changed')
_STEP_NAME = dict(embedding='section 4 (4.1)', cc='7.1', landscape='7.2', trajectory='7.3', volumes='8.1')


def _digest(*parts):
    import hashlib
    h = hashlib.sha1()
    for p in parts:
        h.update(np.ascontiguousarray(p).tobytes() if isinstance(p, np.ndarray)
                 else json.dumps(p, sort_keys=True, default=str).encode())
    return h.hexdigest()[:12]


def _ledger_file(params):
    return os.path.join(params.out_dir, 'notebook_ledger.json')


def _read_ledger(params):
    try:
        with open(_ledger_file(params)) as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def _write_ledger(params, led):
    tmp = _ledger_file(params) + '.part'
    with open(tmp, 'w') as f:
        json.dump(led, f, indent=1)
    os.replace(tmp, _ledger_file(params))


def _binning_id(params, prds):
    return _digest(np.asarray(prds.thres_ids, dtype=np.int64), int(prds.thres_high), int(params.aperture_index),
                   params.tess_hemisphere_type, [float(v) for v in params.tess_hemisphere_vec])


def _anchors_id(prds):
    return _digest(sorted((int(k), int(a.CC), int(a.sense.value)) for k, a in prds.anchors.items()),
                   sorted(int(t) for t in prds.trash_ids))


def _stage_inputs(params, prds, led, stage):
    """What a stage depends on right now."""
    if stage == 'embedding':
        return dict(binning=_binning_id(params, prds))
    up = _UPSTREAM[stage]
    inputs = {up: (led.get(up) or {}).get('id')}
    if stage == 'cc':
        inputs['anchors'] = _anchors_id(prds)
    elif stage == 'landscape':
        inputs['states'] = int(params.states_per_coord)
    elif stage == 'trajectory':
        inputs['width'] = int(params.width_1D)
    return inputs


def _stage_files(params, stage):
    """Files a finished stage must have produced (number of volumes etc. must match)."""
    P = _paths(params)
    stars = glob.glob(os.path.join(P['bin_dir'], '*.star'))
    vols = glob.glob(os.path.join(P['vols_dir'], '*.mrc'))
    if stage == 'cc':
        return os.path.isfile(P['CC_file'])
    if stage == 'landscape':
        return os.path.isfile(P['traj'])
    if stage == 'trajectory':
        return bool(stars) and all(os.path.getsize(s) > 0 for s in stars)
    if stage == 'volumes':
        return bool(stars) and len(vols) == len(stars)
    if stage == 'denoise':
        den = glob.glob(os.path.join(params.out_dir, 'postproc', 'denoise_notebook', '*.mrc'))
        return bool(vols) and len(den) == len(vols)
    return True


SECTION4 = ('calc-distance', 'manifold-analysis', 'psi-analysis', 'nlsa-movie')


def _pd_done(P, num_psi, p, since=None):
    """Which section-4 steps are complete for PD p (0-based), each checked against its direct input only: an output
    must exist and be at least as new as the output it is computed from (and as `since[step]`). _status chains them."""
    since = since or {}
    mt = lambda f: os.path.getmtime(f) if os.path.isfile(f) else None
    ok = lambda t, after, step: t is not None and after is not None and t >= max(after, since.get(step, 0))
    dist, psi, eig = mt(P['dist'].format(p=p)), mt(P['psi'].format(p=p)), mt(P['eig_spec'].format(p=p + 1))
    psi2 = [mt(P['psi2'].format(p=p, k=k)) for k in range(num_psi)]
    newest_psi2 = None if None in psi2 else max(psi2)
    return {'calc-distance': ok(dist, 0, 'calc-distance'),
            'manifold-analysis': ok(psi, dist, 'manifold-analysis') and ok(eig, dist, 'manifold-analysis'),
            'psi-analysis': all(ok(t, psi, 'psi-analysis') for t in psi2),
            'nlsa-movie': ok(mt(P['class_avg'].format(p=p + 1)), newest_psi2, 'nlsa-movie')}


def _status(params, prds):
    """Which pipeline steps have complete and up-to-date results (plus why the others are out of date)."""
    P = _paths(params)
    n_pd = prds.n_thresholded
    per_pd = [_pd_done(P, params.num_psi, p) for p in range(n_pd)]
    for d in per_pd:  # a result only counts while the step before it is current too
        for a, b in zip(SECTION4, SECTION4[1:]):
            d[b] = d[b] and d[a]
    done = dict(init=True, binning=os.path.isfile(P['pd_file']),
                **{s.replace('-', '_'): n_pd > 0 and all(d[s] for d in per_pd) for s in SECTION4})
    led = _read_ledger(params)
    stale, keys = {}, {}
    upstream_ok = done['calc_distance'] and done['manifold_analysis'] and done['psi_analysis']
    for stage in STAGES:
        rec = led.get(stage)
        want = _stage_inputs(params, prds, led, stage)
        ok = bool(upstream_ok and rec and rec.get('inputs') == want and _stage_files(params, stage))
        if rec and not ok:
            changed = [k for k in want if rec.get('inputs', {}).get(k) != want[k]]
            up = _UPSTREAM.get(stage)
            keys[stage] = changed[0] if changed else 'upstream' if not upstream_ok else 'files'
            stale[stage] = (_WHY.get(changed[0], changed[0]) if changed else
                            'an earlier step is not up to date' if not upstream_ok else 'some of its files are missing')
            if changed and changed[0] == up and want[up] is None:  # the step before was stopped, not changed
                keys[stage] = 'upstream'
                stale[stage] = f'{_STEP_NAME[up]} has no finished result (stopped or not run yet): run it again'
        done[stage] = ok
        upstream_ok = ok
    done = dict(done, find_ccs=done.pop('cc'), probability_landscape=done.pop('landscape'),
                volumes=len(glob.glob(os.path.join(P['vols_dir'], '*.mrc'))) if done['volumes'] else 0)
    return done, dict(stale, _keys=keys)


@command
def ledger(toml):
    """The stage ledger (what each stage was computed from)."""
    return _read_ledger(_load(toml))


@command
def stage_begin(toml, stage):
    """Forget a stage's record before (re)computing it."""
    params = _load(toml)
    led = _read_ledger(params)
    if led.pop(stage, None) is not None:
        _write_ledger(params, led)
    return {}


@command
def stage_end(toml, stage, info=None):
    """Record that a stage finished, with the identity of its inputs."""
    from ManifoldEM import myio
    params = _load(toml)
    prds = _prds()
    led = _read_ledger(params)
    inputs = _stage_inputs(params, prds, led, stage)
    if stage == 'embedding':
        rid = _digest(inputs, time.time())  # recomputed: everything downstream must be redone
    elif stage == 'landscape':
        rid = _digest(inputs, np.asarray(myio.fin1(_paths(params)['traj'])['hUn'], dtype=float))
    else:
        rid = _digest(inputs, info)
    led[stage] = dict(id=rid, inputs=inputs, info=info, time=time.time())
    _write_ledger(params, led)
    return dict(id=rid)


def _nlsa_frames(params, n_pd):
    """states_per_coord used by psi-analysis (its frame count): the largest STATES probability-landscape accepts."""
    import h5py
    frames = 0
    for p in range(n_pd):  # PDs with few images store fewer frames, so take the maximum
        try:  # missing, or truncated by a stop (incomplete() then makes 4.1 recompute it)
            with h5py.File(params.get_psi2_file(p, 0), 'r') as h:
                frames = max(frames, int(h['IMG1'].shape[1]))
        except (OSError, KeyError):
            continue
    return frames


@command
def summary(toml, out_npz=None, n_particles_show=6000):
    """Everything the notebook plots about the projection directions (bins the data on first use)."""
    import ManifoldEM
    params = _load(toml)
    prds = _prds()  # builds pd_data.pkl the first time (this is the "binning" step)
    n_pd = prds.n_thresholded
    occ = np.asarray(prds.occupancy, dtype=int)
    rng = np.random.default_rng(0)
    pos_full = prds.pos_full
    sel = rng.choice(pos_full.shape[1], min(n_particles_show, pos_full.shape[1]), replace=False)
    anchors = [[int(k), int(a.CC), int(a.sense.value)] for k, a in sorted(prds.anchors.items())]
    trash = sorted(int(t) for t in prds.trash_ids)
    edges = np.asarray(prds.neighbor_graph.get('Edges', np.zeros((0, 2))), dtype=int).reshape(-1, 2)
    done, stale = _status(params, prds)
    if out_npz:
        np.savez(out_npz, occupancy=occ, n_used=np.minimum(occ, prds.thres_high),
                 cluster_ids=np.asarray(prds.cluster_ids, dtype=int), thres_ids=np.asarray(prds.thres_ids, dtype=int),
                 component_ids=_component_ids(prds, params.num_psi),
                 pos_pd=np.asarray(prds.pos_thresholded, dtype=float),
                 theta=np.asarray(prds.theta_thresholded, dtype=float),
                 phi=np.asarray(prds.phi_thresholded, dtype=float),
                 bin_centers=np.asarray(prds.bin_centers, dtype=float),
                 occupancy_full=np.asarray(prds.occupancy_full, dtype=int),
                 pos_particles=np.asarray(pos_full[:, sel], dtype=float), edges=edges,
                 anchors=np.asarray(anchors, dtype=int).reshape(-1, 3), trash=np.asarray(trash, dtype=int))
    return dict(
        project=params.project_name, version=getattr(ManifoldEM, '__version__', '?'),
        level=params.project_level.name, n_pd=int(n_pd), n_bins=int(prds.n_bins),
        n_particles=int(pos_full.shape[1]), n_used=int(np.minimum(occ, prds.thres_high).sum()),
        n_clusters=int(len(set(np.asarray(prds.cluster_ids).tolist()))),
        thres_low=int(prds.thres_low), thres_high=int(prds.thres_high),
        aperture_index=int(params.aperture_index), shannon_angle=float(params.sh), ang_width=float(params.ang_width),
        tess_type=params.tess_hemisphere_type, hemisphere_vec=[float(v) for v in params.tess_hemisphere_vec],
        box=int(params.ms_num_pixels), pixel_size=float(params.ms_pixel_size),
        diameter=float(params.particle_diameter), resolution=float(params.ms_estimated_resolution),
        num_psi=int(params.num_psi), num_psi_truncated=int(params.num_psi_truncated),
        states_per_coord=int(params.states_per_coord), con_order_range=int(params.con_order_range),
        width_1D=int(params.width_1D), traj_name=str(params.traj_name), ncpu=int(params.ncpu),
        nlsa_fps=float(params.nlsa_fps), nlsa_tune=int(params.nlsa_tune),
        distance_filter_type=params.distance_filter_type, distance_filter_cutoff_freq=float(params.distance_filter_cutoff_freq),
        distance_filter_order=int(params.distance_filter_order), nlsa_frames=_nlsa_frames(params, n_pd),
        max_images=int(min(prds.thres_high, occ.max())) if n_pd else 0,
        anchors=anchors, trash=trash, cc_bug=_cc_bug(), paths=_paths(params), done=done, stale=stale,
        files=dict(star=params.align_param_file, stack=params.img_stack_file, volume=params.avg_vol_file,
                   mask=params.mask_vol_file),
    )


@command
def set_params(toml, values):
    """Persist parameter changes in the project's .toml.

    (`manifold-cli <step> --flag value` prints "Changing param ..." but every step reloads the .toml,
    so only `init` and `threshold` flags actually stick. Editing the .toml is the reliable way.)"""
    params = _load(toml)
    cls = type(params)
    changed = {}
    for key, value in values.items():
        if not hasattr(cls, key):
            raise UserError(f'unknown ManifoldEM parameter: {key}')
        current = getattr(params, key)
        if isinstance(current, bool):
            value = bool(value)
        elif isinstance(current, (int, float, str)):
            value = type(current)(value)
        if value != current:
            changed[key] = [current, value]
            setattr(params, key, value)
    params.save()
    return dict(changed=changed)


@command
def bin_particles(toml, low=0, high=0, aperture_index=0, tess_type='', confirm=False):
    """Set the PD occupancy thresholds (and optionally aperture/tessellation) and (re)bin.

    0 / '' keeps the project's current value. Re-binning renumbers the PDs, so if results of later steps
    exist the change is refused unless confirm=True (the caller then deletes those results)."""
    from ManifoldEM.params import ProjectLevel
    params = _load(toml)
    want = dict(prd_thres_low=int(low) or params.prd_thres_low, prd_thres_high=int(high) or params.prd_thres_high,
                aperture_index=int(aperture_index) or params.aperture_index,
                tess_hemisphere_type=tess_type or params.tess_hemisphere_type)
    current = {k: getattr(params, k) for k in want}
    changes = {k: [current[k], v] for k, v in want.items() if v != current[k]}
    resumed = os.path.isfile(os.path.join(params.out_dir, _REBIN))
    # any section 4 output or stage record counts (a stopped clean-up may have deleted some of them already)
    has_results = bool(_read_ledger(params)) or any(glob.glob(os.path.join(d, '*.h5'))
                                                    for d in (params.dist_dir, params.psi_dir, params.psi2_dir))
    if changes and has_results and not confirm:
        return dict(refused=True, changes=changes, current=current)
    if (changes or not os.path.isfile(params.pd_file)) and not os.path.isfile(params.align_param_file):
        raise UserError(f'the STAR file of this project is missing: {params.align_param_file} (see 2.2)')
    cleaned_mark = os.path.join(params.out_dir, _CLEANED)
    was_cleaned = os.path.isfile(cleaned_mark) and not has_results  # a stopped confirmed re-bin had deleted them
    before = current  # the binning that the deleted results belonged to
    if was_cleaned:
        try:
            with open(cleaned_mark) as f:
                before = json.load(f)
        except ValueError:  # stopped while the marker was written: unknown
            before = None
    if changes and has_results:  # the PDs will be renumbered: nothing computed so far stays valid
        with open(cleaned_mark, 'w') as f:  # before clean(): a stop anywhere after the deletion is reported next time
            json.dump(current, f)
        clean(toml, ['embedding', 'cc', 'landscape', 'trajectory', 'volumes'])
        params = _load(toml)
    for key, value in want.items():
        setattr(params, key, value)
    if params.project_level.value < ProjectLevel.BINNING.value:
        params.project_level = ProjectLevel.BINNING
    pkl_before = os.path.getmtime(params.pd_file) if os.path.isfile(params.pd_file) else None
    if pkl_before is not None and {'aperture_index', 'tess_hemisphere_type'} & set(changes):
        # written before the .toml: if we are stopped before pd_data.pkl is rebuilt, the next _prds() finishes it
        with open(os.path.join(params.out_dir, _REBIN + '.part'), 'w') as f:
            json.dump(want, f)
        os.replace(os.path.join(params.out_dir, _REBIN + '.part'), os.path.join(params.out_dir, _REBIN))
    params.save()
    _prds()
    if os.path.isfile(cleaned_mark):
        os.remove(cleaned_mark)
    rebuilt = pkl_before is None or os.path.getmtime(params.pd_file) != pkl_before
    return dict(refused=False, changes=changes, current=want, rebuilt=rebuilt,
                cleaned=bool(changes and has_results) or was_cleaned, renumbered=before != want,
                resumed=resumed and not changes)


@command
def clean(toml, what):
    """Delete outputs of later steps (after re-binning, or to force a step to be recomputed)."""
    params = _load(toml)
    out = params.out_dir
    # The stage records go first, and the files are deleted from the last step backwards with the distances last, so
    # a stop half-way never leaves a record of deleted files, and 3.1 (which looks for them) still finishes the job.
    stages = dict(embedding=STAGES, movies=(), cc=('cc',), landscape=('landscape',), trajectory=('trajectory',),
                  volumes=('volumes', 'denoise'))
    led = _read_ledger(params)
    for name in what:
        for stage in stages[name]:
            led.pop(stage, None)
    _write_ledger(params, led)
    groups = dict(
        volumes=[os.path.join(out, 'postproc')],
        trajectory=[params.bin_dir],
        landscape=[params.traj_dir] + glob.glob(os.path.join(out, 'ELConc*')),
        cc=[params.CC_dir],
        movies=glob.glob(os.path.join(out, 'topos', 'PrD_*', 'class_avg.png')),
        embedding=[os.path.join(out, 'topos'), params.psi2_dir, params.psi_dir, params.dist_dir],
    )
    removed = []
    for name in groups:
        if name not in what:
            continue
        for path in groups[name]:
            if os.path.isdir(path):
                shutil.rmtree(path)
                removed.append(path)
            elif os.path.isfile(path):
                os.remove(path)
                removed.append(path)
    params.create_dir()  # ManifoldEM expects its folder skeleton to exist
    return dict(removed=len(removed))


@command
def incomplete(toml, step, since=0):
    """0-based PDs whose outputs for `step` are missing, older than what they are computed from, or older than
    `since`, for resuming with `--prds`."""
    params = _load(toml)
    P = _paths(params)
    n = params.prd_n_active
    # `manifold-analysis --prds` only creates topos/PrD_1..PrD_len(prds), so create all of them here
    for p in range(n):
        os.makedirs(os.path.dirname(P['class_avg'].format(p=p + 1)), exist_ok=True)
    files = {'calc-distance': lambda p: [(P['dist'].format(p=p), 'D')],
             'manifold-analysis': lambda p: [(P['psi'].format(p=p), 'psi')],
             'psi-analysis': lambda p: [(P['psi2'].format(p=p, k=k), 'IMG1') for k in range(params.num_psi)]}.get(step)
    return dict(prds=[p for p in range(n) if not _pd_done(P, params.num_psi, p, {step: since})[step]
                      or (files and not all(_readable(f, key) for f, key in files(p)))], n_pd=n)


def _readable(path, key):
    """An HDF5 output that a stop or crash left truncated (fresh mtime, no data) cannot be opened."""
    import h5py
    try:
        with h5py.File(path, 'r') as h:
            return key in h
    except OSError:
        return False


@command
def drop_psi2(toml, from_k, since=0):
    """Delete every PD's NLSA files for psi index >= from_k (0-based). 4.1 only recomputes psi 1..num_psi, so after a
    change of the NLSA inputs, results for higher psi would otherwise come back as current when num_psi is raised.
    Files written after `since` (the change) already use the new inputs and are kept."""
    params = _load(toml)
    P = _paths(params)
    rx = re.compile(re.escape(P['psi2']).replace(re.escape('{p}'), r'\d+').replace(re.escape('{k}'), r'(\d+)') + '$')
    n = 0
    for f in glob.glob(P['psi2'].format(p='*', k='*')):
        m = rx.match(f)
        if m and int(m.group(1)) >= int(from_k) and not (since and os.path.getmtime(f) >= since):
            os.remove(f)
            n += 1
    return dict(removed=n)


@command
def remove_movies(toml, prds=None):
    """nlsa-movie skips PDs that already have class_avg.png, so delete it to force fresh movies."""
    params = _load(toml)
    P = _paths(params)
    prds = range(params.prd_n_active) if prds is None else prds
    n = 0
    for p in prds:
        f = P['class_avg'].format(p=p + 1)
        if os.path.isfile(f):
            os.remove(f)
            n += 1
    return dict(removed=n)


# ----------------------------------------------------------------------------- anchors & find-ccs
def _components(prds, num_psi):
    """Connected components of the PD graph once the removed ('trash') PDs are cut out, as find-ccs sees it."""
    from copy import deepcopy
    from ManifoldEM.FindCCGraph import prune
    if prds.n_thresholded - len(prds.trash_ids) <= 1:  # prune() then builds a graph of node 0, whatever is left
        return [[i] for i in range(prds.n_thresholded) if i not in prds.trash_ids]
    if prds.trash_ids:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            G, _ = prune(deepcopy(prds.neighbor_graph), prds.trash_ids, num_psi)
    else:
        G = prds.neighbor_graph
    comps = [[int(c) for c in np.atleast_1d(comp)] for comp in G['NodesConnComp']]
    return [c for c in comps if not any(i in prds.trash_ids for i in c)]


def _component_ids(prds, num_psi):
    """Per-PD component label after removing PDs (-1 = removed); the binning-time cluster ids if none are removed."""
    if not prds.trash_ids:
        return np.asarray(prds.cluster_ids, dtype=int)
    ids = np.full(prds.n_thresholded, -1, dtype=int)
    for label, comp in enumerate(_components(prds, num_psi)):
        ids[comp] = label
    return ids


def _anchor_has_neighbour(prds, num_psi):
    """find-ccs propagates the coordinate between neighbouring PDs: some anchor needs a connected, non-removed neighbour."""
    return any(len(c) > 1 and any(i in prds.anchors for i in c) for c in _components(prds, num_psi))


def _unanchored_components(prds, num_psi):
    """Connected components (after removing trash PDs) that contain no anchor: their PDs get no CC."""
    return [c for c in _components(prds, num_psi) if not any(i in prds.anchors for i in c)]


@command
def set_anchors(toml, anchors, trash, check=True):
    """Replace all anchors and removed ('trash') PDs.

    anchors: list of [pd (0-based), psi (1-based, i.e. psi_<psi>.gif), 'FWD' | 'REV'], or 'auto' (the most populated
    PD, psi 1, FWD; only for a single cluster once `trash` is removed); trash: 0-based PDs. check=False skips the rules
    about what find-ccs can work with (for callers that only re-save the project's own choice, e.g. after a 4.0 change).
    Anchor.CC is stored 1-based, as the GUI does; find_ccs() below handles ManifoldEM's indexing quirk."""
    from ManifoldEM.data_store import Anchor, Sense
    params = _load(toml)
    prds = _prds()
    n = prds.n_thresholded
    trash = sorted({int(t) for t in trash})
    for t in trash:
        if not 0 <= t < n:
            raise UserError(f'PD {t + 1} does not exist (valid: 1..{n})')
    if check and n - len(trash) < 2:
        raise UserError(f'only {n - len(trash)} PD(s) would be left: find-ccs (7.1) needs at least 2 PDs that are not removed')
    before = ({k: (a.CC, a.sense.value) for k, a in prds.anchors.items()}, set(prds.trash_ids))
    if anchors == 'auto':  # resolved against the PDs removed in this same call
        prds.trash_ids = set(trash)
        comps = _components(prds, params.num_psi)
        prds.trash_ids = before[1]
        if len(comps) != 1:
            raise UserError(
                f'auto can only pick the direction for a single cluster, but with these PDs removed there are {len(comps)}: '
                'the sign of ψ is arbitrary in each PD, so an automatic FWD for every cluster is a coin flip. Watch the '
                'movies in 6.1 and enter one PD:ψ:FWD|REV entry per cluster, e.g. 12:1:FWD, 30:1:REV.')
        occ = np.asarray(prds.occupancy)
        anchors = [[max(comps[0], key=lambda i: (occ[i], -i)), 1, 'FWD']]
    prds.anchors = {}
    for pd0, psi, sense in anchors:
        pd0, psi, sense = int(pd0), int(psi), str(sense).upper()
        if not 0 <= pd0 < n:
            raise UserError(f'PD {pd0 + 1} does not exist (valid: 1..{n})')
        if not 1 <= psi <= params.num_psi:
            raise UserError(f'psi {psi} is out of range (valid: 1..{params.num_psi})')
        if pd0 in trash:
            raise UserError(f'PD {pd0 + 1} cannot be both an anchor and removed')
        if sense not in ('FWD', 'REV'):
            raise UserError(f"sense must be FWD or REV, not {sense!r}")
        prds.insert_anchor(pd0, Anchor(CC=psi, sense=Sense.FWD if sense == 'FWD' else Sense.REV))
    prds.trash_ids = set(trash)
    if check and prds.anchors and len(prds.anchors) + len(trash) != n and not _anchor_has_neighbour(prds, params.num_psi):
        raise UserError('no anchor has a neighbouring PD left: find-ccs (7.1) propagates the coordinate between '
                        'neighbouring PDs, so put at least one anchor in a group of 2 or more connected PDs that are not '
                        'removed (or anchor every remaining PD)')
    changed = before != ({k: (a.CC, a.sense.value) for k, a in prds.anchors.items()}, set(prds.trash_ids))
    if changed:  # saving marks the conformational coordinate (7.1) and everything after it as out of date
        prds.save()
    missing = _unanchored_components(prds, params.num_psi)
    return dict(n_anchors=len(prds.anchors), n_trash=len(trash), changed=changed,
                unanchored_components=[[c + 1 for c in comp] for comp in missing])


@command
def find_ccs(toml, ncpu=1, bp_num_psi=0):
    """Run find-ccs (optical flow + belief propagation) with two safety nets.

    1. Anchor.CC: the GUI and the "every PD is an anchor" shortcut treat CC as 1-based, but belief
       propagation (CC/runGlobalOptimization.py) indexes states with CC directly, so Anchor(CC=1)
       pins psi_2.  When that bug is present the anchors are shifted by -1 in memory only.
    2. bp_num_psi > 0 limits belief propagation to psi_1..psi_<bp_num_psi> (1 = keep psi_1 everywhere and
       only propagate the sense).  num_psi is restored afterwards.
    Cached optical flow is discarded when it no longer matches the inputs."""
    from ManifoldEM import myio
    from ManifoldEM.data_store import Anchor
    from ManifoldEM.find_conformational_coords import op as find_ccs_op
    params = _load(toml)
    num_psi, ncpu_saved = params.num_psi, params.ncpu
    bp_psi = int(bp_num_psi) if bp_num_psi else num_psi
    if not 1 <= bp_psi <= num_psi:
        raise UserError(f'bp_num_psi must be between 1 and num_psi={num_psi}')
    prds = _prds()
    done, stale = _status(params, prds)
    if not done['embedding']:
        raise UserError('section 4 is not complete or out of date' + (f" ({stale['embedding']})" if 'embedding' in stale
                        else '') + ': run 4.1 first')
    if not prds.anchors:
        raise UserError('No anchors set. Run step 6.2 first (find-ccs would otherwise silently do nothing).')
    bp_path = len(prds.anchors) + len(prds.trash_ids) != prds.n_thresholded  # else every PD is fixed by hand
    limit = bp_psi if bp_path else num_psi
    for pd0, a in prds.anchors.items():
        if not 1 <= a.CC <= limit:
            raise UserError(f'Anchor on PD {pd0 + 1} uses psi {a.CC}, but ' + (
                f'belief propagation only considers psi 1..{bp_psi}. Change the anchor or BP_NUM_PSI.' if bp_path
                else f'the project only has psi 1..{num_psi}.'))
    if bp_path and not _anchor_has_neighbour(prds, num_psi):
        raise UserError('no anchor has a neighbouring PD left (anchors dropped by 4.0 or 4.1 can leave only isolated '
                        'ones): find-ccs propagates the coordinate between neighbouring PDs, so set the anchors again '
                        'in 6.2, with at least one in a group of 2 or more connected PDs that are not removed')
    old_id = (_read_ledger(params).get('cc') or {}).get('id')
    stage_begin(params.proj_file, 'cc')
    # cached optical flow is only valid for the same movies, number of psi and removed PDs
    psi2 = glob.glob(os.path.join(params.psi2_dir, '*.h5'))
    marker = os.path.join(params.CC_dir, 'notebook_cc_inputs.json')
    stamp = dict(bp_psi=bp_psi, newest_psi2=max((os.path.getmtime(f) for f in psi2), default=0),
                 trash=sorted(int(t) for t in prds.trash_ids))
    old = json.load(open(marker)) if os.path.isfile(marker) else None
    if os.path.isfile(marker):
        os.remove(marker)  # rewritten only after a successful run, so an interrupted run is never trusted
    if old != stamp:
        for d in (params.CC_OF_dir, params.CC_meas_dir, params.CC_dir_temp):
            shutil.rmtree(d, ignore_errors=True)
        params.create_dir()
        os.makedirs(params.CC_dir_temp, exist_ok=True)
    shifted = bp_path and _cc_bug()
    saved_anchors = dict(prds.anchors)
    if shifted:
        prds.anchors = {k: Anchor(a.CC - 1, a.sense) for k, a in saved_anchors.items()}
    # The PD graph was built with 2 x num_psi states per PD at binning time, and belief propagation only resets
    # maxState: if num_psi was smaller then, the REV states are unreachable. Patch it in memory (never saved).
    G = prds.neighbor_graph
    G['maxState'] = 2 * bp_psi
    G['nStates'] = np.full(int(G['nNodes']), 2 * bp_psi, dtype=int)
    with open(os.path.join(params.out_dir, _RESTORE), 'w') as f:  # lets _load() undo the change if we are killed
        json.dump(dict(num_psi=num_psi, ncpu=ncpu_saved), f)
    params.ncpu, params.num_psi = int(ncpu), bp_psi
    params.save()  # find-ccs reloads the .toml
    t0 = time.time()
    try:
        find_ccs_op()
    finally:
        prds.anchors = saved_anchors
        _load(params.proj_file)  # restores num_psi / ncpu and removes the sidecar
    if not os.path.isfile(params.CC_file):
        raise UserError('find-ccs did not write CC_file.pkl; see the log above.')
    with open(marker, 'w') as f:
        json.dump(stamp, f)
    cc = myio.fin1(params.CC_file)
    psinums = np.asarray(cc['psinums'][0])
    led = _read_ledger(params)
    inputs = _stage_inputs(params, prds, led, 'cc')
    new_id = _digest(inputs, np.asarray(cc['psinums']), np.asarray(cc['senses']))
    led['cc'] = dict(id=new_id, inputs=inputs, info=dict(bp_num_psi=bp_psi), time=time.time())
    _write_ledger(params, led)
    changed = old_id is not None and old_id != new_id
    return dict(seconds=round(time.time() - t0, 1), anchors_shifted=shifted, bp_num_psi=bp_psi, bp_ran=bp_path,
                n_assigned=int((psinums >= 0).sum()), n_pd=int(len(psinums)), changed=changed)


# ---------------------------------------------------------------------------------------- inputs
def _star_pixel_size(df, optics):
    """Pixel size as RELION reads it: optics rlnImagePixelSize, else rlnDetectorPixelSize * 1e4 / rlnMagnification."""
    if 'rlnImagePixelSize' in optics.columns:
        return float(optics['rlnImagePixelSize'].values[0])
    for table in (df, optics):
        if {'rlnDetectorPixelSize', 'rlnMagnification'} <= set(table.columns):
            return float(table['rlnDetectorPixelSize'].values[0]) * 1e4 / float(table['rlnMagnification'].values[0])
    return None


REQUIRED_COLUMNS = ['rlnAngleRot', 'rlnAngleTilt', 'rlnAnglePsi', 'rlnDefocusU', 'rlnDefocusV',
                    'rlnVoltage', 'rlnSphericalAberration', 'rlnAmplitudeContrast']


def _read_star(star):
    """Particles table and first optics row, read with ManifoldEM's own (strict) parser."""
    from ManifoldEM.star import parse_star, parse_star_optics
    with open(star) as f:
        new_style = any(line.startswith('data_optics') for line in f)
    if new_style:
        optics, skip = parse_star_optics(star)
        return parse_star(star, skip), optics
    df = parse_star(star, 0)
    return df, df.iloc[:1]


@command
def inspect_inputs(star, stack, n_show=16, out_npz=None, mask=''):
    """Check a STAR file + particle stack the way ManifoldEM will read them."""
    import mrcfile
    report = dict(problems=[], warnings=[])
    if out_npz and os.path.isfile(out_npz):
        os.remove(out_npz)  # never show the previous dataset's images
    try:
        df, optics = _read_star(star)
    except Exception as exc:  # pragma: no cover - reported to the user
        report['problems'].append(f'ManifoldEM cannot parse the STAR file: {exc!r}')
        return report
    cols = set(df.columns) | set(optics.columns)
    report['columns'] = sorted(cols)
    report['missing'] = [c for c in REQUIRED_COLUMNS if c not in cols]
    if report['missing']:
        report['problems'].append('missing required STAR columns: ' + ', '.join(report['missing']))
    has_origin = {'rlnOriginX', 'rlnOriginY'} <= cols or {'rlnOriginXAngst', 'rlnOriginYAngst'} <= cols
    if not has_origin:
        report['warnings'].append('no origin (shift) columns: images are assumed to be centred')
    try:
        with mrcfile.mmap(stack, mode='r', permissive=True) as m:
            if m.data is None:
                raise ValueError(f'truncated or damaged file ({os.path.getsize(stack):,} bytes)')
            shape = m.data.shape
            report['stack_is_image_stack'] = bool(m.is_image_stack())
            report['header_pixel_size'] = float(m.voxel_size.x)
            n_img = shape[0] if len(shape) == 3 else 1
            rng = np.random.default_rng(1)
            idx = np.sort(rng.choice(n_img, min(n_show, n_img), replace=False))
            imgs = np.asarray(m.data[idx], dtype=np.float32) if len(shape) == 3 else np.asarray(m.data)[None]
    except Exception as exc:
        report['problems'].append(f'cannot read the particle stack {os.path.basename(stack)} as MRC: {exc}')
        return report
    if mask:
        try:
            with mrcfile.open(mask, permissive=True, header_only=True) as m:
                mshape = (int(m.header.nz), int(m.header.ny), int(m.header.nx))
            if mshape != (shape[-1],) * 3:
                report['problems'].append(f'the mask is {mshape[2]}×{mshape[1]}×{mshape[0]} voxels but the images are '
                                          f'{shape[-1]}×{shape[-1]} px: they must have the same box')
        except Exception as exc:
            report['problems'].append(f'cannot read the mask as MRC: {exc}')
    report.update(n_particles=int(len(df)), n_images=int(n_img), box=int(shape[-1]))
    if len(shape) != 3 or shape[-1] != shape[-2]:
        report['problems'].append(f'the stack must hold square images, found shape {shape}')
    if n_img != len(df):
        report['problems'].append(f'the STAR file has {len(df)} particles but the stack has {n_img} images '
                                  '(ManifoldEM pairs STAR row i with stack image i and ignores rlnImageName)')
    if not report['stack_is_image_stack']:
        report['warnings'].append('the MRC header does not flag the file as an image stack; ManifoldEM will '
                                  'rewrite the header in place (work on a copy)')
    report['star_pixel_size'] = _star_pixel_size(df, optics)
    hdr = report['header_pixel_size']
    if report['star_pixel_size'] and hdr > 0 and abs(report['star_pixel_size'] - hdr) > 0.01 * hdr:
        report['warnings'].append(f"the STAR file says {report['star_pixel_size']:.4g} Å/px but the stack header says "
                                  f'{hdr:.4g} Å/px: enter the correct pixel size yourself in 2.1')
    for key, col in (('voltage', 'rlnVoltage'), ('cs', 'rlnSphericalAberration'), ('amp_contrast', 'rlnAmplitudeContrast')):
        if col in cols:
            src = optics if col in optics.columns else df
            report[key] = float(src[col].values[0])
    if 'rlnOpticsGroup' in df.columns and df['rlnOpticsGroup'].nunique() > 1:
        report['warnings'].append(f"{df['rlnOpticsGroup'].nunique()} optics groups found: ManifoldEM uses the "
                                  'voltage/Cs/amplitude contrast of the first one for all particles')
    if out_npz is not None and not report['missing']:
        take = np.random.default_rng(2).choice(len(df), min(20000, len(df)), replace=False)
        np.savez(out_npz, images=imgs, image_index=idx,
                 rot=df['rlnAngleRot'].values[take].astype(float), tilt=df['rlnAngleTilt'].values[take].astype(float),
                 psi=df['rlnAnglePsi'].values[take].astype(float),
                 defocus=(0.5 * (df['rlnDefocusU'].values + df['rlnDefocusV'].values)).astype(float))
    return report


def _crop_map(path, out, new_n, scale, px_old, is_mask=False):
    """Fourier-crop a cubic map to new_n**3 (a mask is re-binarised at 0.5)."""
    import mrcfile
    with mrcfile.open(path, permissive=True) as m:
        vol = np.asarray(m.data, dtype=np.float32)
    n = vol.shape[-1]
    s = n // 2 - new_n // 2
    F = np.fft.fftshift(np.fft.fftn(vol))[s:s + new_n, s:s + new_n, s:s + new_n]
    small = (np.fft.ifftn(np.fft.ifftshift(F)).real / scale ** 3).astype(np.float32)
    if is_mask:
        small = (small > 0.5).astype(np.float32)
    with mrcfile.new(out + '.part', overwrite=True) as m:
        m.set_data(small)
        if px_old:
            m.voxel_size = px_old * scale
    os.replace(out + '.part', out)
    return out


@command
def downsample(star, stack, out_dir, factor, volume='', mask=''):
    """Fourier-crop every particle image by `factor` (e.g. 2: 336 px -> 168 px) and write a matching STAR,
    plus the mask and average volume at the new box.

    Speeds up every ManifoldEM step roughly by factor**2 to factor**3 and cuts memory by factor**2; the
    new Nyquist limit is 2 x the new pixel size. Files are written as .part and renamed, and a completed
    out_dir (marker file) is reused as is."""
    import mrcfile
    done = os.path.join(out_dir, '.complete.json')
    if os.path.isfile(done):
        with open(done) as f:
            info = json.load(f)
        if all(os.path.isfile(info[k]) for k in ('star', 'stack', 'volume', 'mask') if info.get(k)):
            return dict(info, reused=True)
        os.remove(done)  # some of its files are gone (e.g. a checkpoint without the mask): make them again
    os.makedirs(out_dir, exist_ok=True)
    try:
        px_star = _star_pixel_size(*_read_star(star))
    except Exception:
        px_star = None
    with mrcfile.mmap(stack, mode='r', permissive=True) as m:
        if m.data is None:
            raise UserError(f'cannot read the particle stack {stack}: truncated or damaged')
        n_img, n = m.data.shape[0], m.data.shape[-1]
        px_hdr = float(m.voxel_size.x)
        px_old = px_star or (px_hdr if px_hdr > 0 else None)
        new_n = int(round(n / factor / 2)) * 2
        scale = n / new_n
        out_stack = os.path.join(out_dir, os.path.splitext(os.path.basename(stack))[0] + f'_bin{factor:g}.mrcs')
        with mrcfile.new_mmap(out_stack + '.part', shape=(n_img, new_n, new_n), mrc_mode=2, overwrite=True) as out:
            s = n // 2 - new_n // 2
            for i0 in range(0, n_img, 500):
                block = np.asarray(m.data[i0:i0 + 500], dtype=np.float32)
                F = np.fft.fftshift(np.fft.fft2(block), axes=(-2, -1))[:, s:s + new_n, s:s + new_n]
                out.data[i0:i0 + len(block)] = (np.fft.ifft2(np.fft.ifftshift(F, axes=(-2, -1))).real
                                                / scale ** 2).astype(np.float32)
                print(f'\r  downsampled {min(i0 + 500, n_img)}/{n_img} images', end='', flush=True)
            out.set_image_stack()
            if px_old:
                out.voxel_size = px_old * scale
    print()
    os.replace(out_stack + '.part', out_stack)
    # rewrite the STAR line by line: pixel origins shrink, optics pixel size / box change
    out_star = os.path.join(out_dir, os.path.splitext(os.path.basename(star))[0] + f'_bin{factor:g}.star')
    header, lines_out = [], []
    stack_name = os.path.basename(out_stack)
    with open(star) as f:
        lines = f.read().splitlines()
    for line in lines:
        s_line = line.strip()
        if s_line.startswith('data_'):
            header = []
        elif s_line.startswith('_rln'):
            header.append(s_line.split()[0][1:])
        elif s_line and header and not s_line.startswith(('#', 'loop_')):
            vals = s_line.split()
            if len(vals) == len(header):
                for j, col in enumerate(header):
                    if col in ('rlnOriginX', 'rlnOriginY'):
                        vals[j] = f'{float(vals[j]) / scale:.6f}'
                    elif col in ('rlnImagePixelSize', 'rlnDetectorPixelSize'):  # old STARs: DetPix * 1e4 / Mag
                        vals[j] = f'{float(vals[j]) * scale:.6f}'
                    elif col == 'rlnImageSize':
                        vals[j] = str(new_n)
                    elif col == 'rlnImageName' and '@' in vals[j]:
                        vals[j] = vals[j].split('@')[0] + '@' + stack_name
                line = ' '.join(vals)
        lines_out.append(line)
    with open(out_star + '.part', 'w') as f:
        f.write('\n'.join(lines_out) + '\n')
    os.replace(out_star + '.part', out_star)
    result = dict(star=out_star, stack=out_stack, box=new_n, scale=scale,
                  pixel_size=round(px_old * scale, 5) if px_old else None, volume='', mask='')
    for key, path in (('volume', volume), ('mask', mask)):
        if path:
            result[key] = _crop_map(path, os.path.join(out_dir, os.path.splitext(os.path.basename(path))[0]
                                                       + f'_bin{factor:g}.mrc'), new_n, scale, px_old, key == 'mask')
    with open(done, 'w') as f:  # written last: an interrupted run is redone
        json.dump(result, f)
    return dict(result, reused=False)


# -------------------------------------------------------------------------------- synthetic data
def _relion_matrix(rot, tilt, psi):
    """RELION Euler_angles2matrix (ZYZ, degrees), vectorised -> (m, 3, 3)."""
    a, b, g = (np.deg2rad(np.asarray(v, dtype=float)) for v in (rot, tilt, psi))
    ca, sa, cb, sb, cg, sg = np.cos(a), np.sin(a), np.cos(b), np.sin(b), np.cos(g), np.sin(g)
    A = np.empty(a.shape + (3, 3))
    A[..., 0, 0], A[..., 0, 1], A[..., 0, 2] = cg * cb * ca - sg * sa, cg * cb * sa + sg * ca, -cg * sb
    A[..., 1, 0], A[..., 1, 1], A[..., 1, 2] = -sg * cb * ca - cg * sa, -sg * cb * sa + cg * ca, sg * sb
    A[..., 2, 0], A[..., 2, 1], A[..., 2, 2] = sb * ca, sb * sa, cb
    return A


@command
def synthetic(src, out, n_particles=6000, box=64, levels=20, amp_angst=25.0, snr=0.1, tilt_jitter=2.0,
              seed=0, kv=300.0, cs=2.7, ac=0.1):
    """Small RELION-style dataset with a known 1-D motion, built from the RyR1 map shipped with ManifoldEM.

    The map's periphery (the cytoplasmic 'arms') moves along z by amp_angst * tau, tau ~ U[0, 1]. Particles
    are side views on one great circle, projected (Fourier slice, RELION Euler convention), multiplied by a
    RELION-convention CTF and buried in Gaussian noise (SNR = var(signal) / var(noise))."""
    import mrcfile
    import pandas as pd
    from scipy import ndimage
    t0 = time.time()
    rng = np.random.default_rng(seed)
    os.makedirs(out, exist_ok=True)
    with mrcfile.open(src, permissive=True) as m:
        vol0, px0 = m.data.astype(np.float32), float(m.voxel_size.x)
    N0 = vol0.shape[0]
    px = px0 * N0 / box
    F = np.fft.fftshift(np.fft.fftn(vol0))
    s = N0 // 2 - box // 2
    vol = (np.fft.ifftn(np.fft.ifftshift(F[s:s + box, s:s + box, s:s + box])).real * (box / N0) ** 3).astype(np.float32)
    z, y, x = np.mgrid[:box, :box, :box].astype(np.float32)
    r = np.hypot(x - box // 2, y - box // 2) / (box / 2)
    t = np.clip((r - 0.40) / 0.30, 0, 1)
    w = t * t * (3 - 2 * t)  # 0 in the core, 1 in the periphery
    taus = np.linspace(0, 1, levels)
    amp = amp_angst / px
    vols = np.stack([ndimage.map_coordinates(vol, [z - amp * tk * w, y, x], order=3, mode='nearest') for tk in taus])
    for name, data in (('avg_volume.mrc', vols[levels // 2]), ('conformations.mrcs', vols)):
        with mrcfile.new(os.path.join(out, name + '.part'), overwrite=True) as m:
            m.set_data(data.astype(np.float32))
            m.voxel_size = px
        os.replace(os.path.join(out, name + '.part'), os.path.join(out, name))
    rot, psi = rng.uniform(0, 360, n_particles), rng.uniform(0, 360, n_particles)
    tilt = 90 + rng.normal(0, tilt_jitter, n_particles)
    level = rng.integers(0, levels, n_particles)
    dU = rng.uniform(10000.0, 30000.0, n_particles)
    dV = dU + rng.normal(0, 100, n_particles)
    dang = rng.uniform(0, 180, n_particles)
    A = _relion_matrix(rot, tilt, psi)
    lam = 12.2643247 / np.sqrt(kv * 1e3 * (1 + kv * 1e3 * 0.978466e-6))
    f = np.fft.fftfreq(box, d=px)
    FX, FY = np.meshgrid(f, f)
    s2, ang = FX ** 2 + FY ** 2, np.arctan2(FY, FX)
    M = 2 * box
    k = np.arange(M) - M // 2
    KX, KY = np.meshgrid(k, k)
    clean = np.empty((n_particles, box, box), np.float32)
    for lev in range(levels):
        vp = np.zeros((M, M, M), np.float32)
        vp[M // 4:M // 4 + box, M // 4:M // 4 + box, M // 4:M // 4 + box] = vols[lev]
        F3 = np.fft.fftshift(np.fft.fftn(np.fft.ifftshift(vp))).astype(np.complex64)
        idx = np.nonzero(level == lev)[0]
        for c0 in range(0, len(idx), 250):
            j = idx[c0:c0 + 250]
            k3 = A[j, 0, :, None] * KX.ravel() + A[j, 1, :, None] * KY.ravel()  # A^T (kx, ky, 0)
            crd = np.stack([k3[:, 2], k3[:, 1], k3[:, 0]]).reshape(3, -1) + M // 2
            F2 = (ndimage.map_coordinates(F3.real, crd, order=1)
                  + 1j * ndimage.map_coordinates(F3.imag, crd, order=1)).reshape(len(j), M, M)
            img = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F2, axes=(1, 2))), axes=(1, 2)).real
            proj = img[:, M // 4:M // 4 + box, M // 4:M // 4 + box]
            df = (0.5 * (dU[j] + dV[j]))[:, None, None] + (0.5 * (dU[j] - dV[j]))[:, None, None] * np.cos(
                2 * (ang - np.deg2rad(dang[j])[:, None, None]))
            gamma = np.pi * lam * df * s2 - 0.5 * np.pi * cs * 1e7 * lam ** 3 * s2 ** 2
            clean[j] = np.fft.ifft2(np.fft.fft2(proj) * np.sin(gamma + np.arcsin(ac))).real
        print(f'\r  projected conformation {lev + 1}/{levels}', end='', flush=True)
    print()
    sigma = np.sqrt(clean.var() / snr)
    imgs = (clean + rng.normal(0, sigma, clean.shape).astype(np.float32)) / sigma
    with mrcfile.new(os.path.join(out, 'particles.mrcs.part'), overwrite=True) as m:
        m.set_data(imgs.astype(np.float32))
        m.set_image_stack()
        m.voxel_size = px
    os.replace(os.path.join(out, 'particles.mrcs.part'), os.path.join(out, 'particles.mrcs'))
    names = [f'{i + 1:06d}@particles.mrcs' for i in range(n_particles)]
    df = pd.DataFrame(dict(rlnImageName=names, rlnAngleRot=rot, rlnAngleTilt=tilt, rlnAnglePsi=psi,
                           rlnOriginXAngst=0.0, rlnOriginYAngst=0.0, rlnDefocusU=dU, rlnDefocusV=dV,
                           rlnDefocusAngle=dang, rlnOpticsGroup=1))
    with open(os.path.join(out, 'particles.star.part'), 'w') as fh:  # header lines must be contiguous for ManifoldEM
        fh.write('\n# version 30001\n\ndata_optics\n\nloop_\n')
        for i, c in enumerate(['rlnOpticsGroupName', 'rlnOpticsGroup', 'rlnVoltage', 'rlnSphericalAberration',
                               'rlnAmplitudeContrast', 'rlnImagePixelSize', 'rlnImageSize', 'rlnImageDimensionality']):
            fh.write(f'_{c} #{i + 1}\n')
        fh.write(f'opticsGroup1 1 {kv:.6f} {cs:.6f} {ac:.6f} {px:.6f} {box} 2\n\n\n# version 30001\n\n'
                 'data_particles\n\nloop_\n')
        fh.writelines(f'_{c} #{i + 1}\n' for i, c in enumerate(df.columns))
        df.to_csv(fh, sep=' ', header=False, index=False, float_format='%.6f')
    os.replace(os.path.join(out, 'particles.star.part'), os.path.join(out, 'particles.star'))
    pd.DataFrame(dict(image_index=np.arange(n_particles), tau=taus[level], level=level)).to_csv(
        os.path.join(out, 'ground_truth.csv.part'), index=False)
    os.replace(os.path.join(out, 'ground_truth.csv.part'), os.path.join(out, 'ground_truth.csv'))
    return dict(out=os.path.abspath(out), pixel_size=px, box=box, n_particles=n_particles,
                seconds=round(time.time() - t0, 1))


@command
def ground_truth(toml, gt_csv):
    """Per-PD Spearman correlation between ManifoldEM's final tau and the synthetic ground truth."""
    import pandas as pd
    from scipy.stats import spearmanr
    from ManifoldEM import myio
    params = _load(toml)
    prds = _prds()
    gt = pd.read_csv(gt_csv)['tau'].values
    v = myio.fin1(f'{params.traj_file}name{params.traj_name}_vars.pkl')
    rows = []
    for x in sorted(v['xSelect']):
        tau = np.asarray(v['trajTaus'][x]).ravel()
        pos_path, pos_psi1 = np.asarray(v['posPathAll'][x]), np.asarray(v['posPsi1All'][x])
        idx = np.asarray(prds.thresholded_image_indices[x])[pos_path[pos_psi1]]
        con_order = len(pos_path) // params.con_order_range  # same mapping as writeRelionS2
        raw = idx[con_order - 1:con_order - 1 + len(tau)]
        rows.append(dict(pd=int(x) + 1, rho=float(spearmanr(tau, gt[raw])[0]), n=int(len(tau))))
    return dict(rows=rows)


# ------------------------------------------------------------------------------ 3D reconstruction
def _backproject(images, rot, tilt, psi, pad=2, floor_frac=0.1, batch=32):
    """Direct Fourier inversion (trilinear splatting, RELION Euler convention, no CTF), like
    `relion_reconstruct --i x.star` on ManifoldEM's trajectory stacks (which carry no CTF or shifts)."""
    images = np.asarray(images, dtype=np.float32)
    n, N, _ = images.shape
    M = int(round(pad * N)) // 2 * 2
    Mh, off = M // 2 + 1, M // 2 - N // 2
    A = _relion_matrix(rot, tilt, psi)
    KY, KX = np.meshgrid(np.fft.fftfreq(M) * M, np.arange(Mh, dtype=float), indexing='ij')
    rmax = M / 2.0 - 1.0
    sel = KX ** 2 + KY ** 2 <= rmax ** 2
    kx, ky = KX[sel], KY[sel]
    cw = np.where(kx == 0, 0.5, 1.0)  # the kx = 0 column holds both Friedel mates
    D_re, D_im, W = (np.zeros(M * M * Mh) for _ in range(3))
    pimg = np.zeros((batch, M, M), np.float32)
    for b0 in range(0, n, batch):
        B = min(n, b0 + batch) - b0
        pimg[:] = 0
        pimg[:B, off:off + N, off:off + N] = images[b0:b0 + B]
        F = np.fft.rfft2(np.fft.ifftshift(pimg[:B], axes=(-2, -1)))[:, sel]
        k3 = kx[None, :, None] * A[b0:b0 + B, None, 0, :] + ky[None, :, None] * A[b0:b0 + B, None, 1, :]
        neg = k3[..., 0] < 0  # keep the x >= 0 half via Friedel symmetry
        k3[neg] *= -1
        F = np.where(neg, np.conj(F), F)
        x, y, z = (k3[..., i].ravel() for i in range(3))
        x0, y0, z0 = np.floor(x), np.floor(y), np.floor(z)
        fx, fy, fz = x - x0, y - y0, z - z0
        x0, y0, z0 = x0.astype(np.int64), y0.astype(np.int64), z0.astype(np.int64)
        Fr, Fi, wt = F.real.ravel(), F.imag.ravel(), np.broadcast_to(cw, F.shape).ravel()
        for dz in (0, 1):
            for dy in (0, 1):
                for dx in (0, 1):
                    w = (fx if dx else 1 - fx) * (fy if dy else 1 - fy) * (fz if dz else 1 - fz)
                    idx = (((z0 + dz) % M) * M + (y0 + dy) % M) * Mh + x0 + dx
                    D_re += np.bincount(idx, w * Fr, minlength=D_re.size)
                    D_im += np.bincount(idx, w * Fi, minlength=D_im.size)
                    W += np.bincount(idx, w * wt, minlength=W.size)
    D = (D_re + 1j * D_im).reshape(M, M, Mh)
    W = W.reshape(M, M, Mh)
    del D_re, D_im
    flip = (-np.arange(M)) % M  # x = 0 plane: add the Friedel mates
    D0, W0 = D[:, :, 0].copy(), W[:, :, 0].copy()
    D[:, :, 0] += np.conj(D0[np.ix_(flip, flip)])
    W[:, :, 0] += W0[np.ix_(flip, flip)]
    kk = (np.fft.fftfreq(M) * M).astype(np.float32)
    R = np.sqrt(kk[:, None, None] ** 2 + kk[None, :, None] ** 2 + np.arange(Mh, dtype=np.float32)[None, None, :] ** 2)
    inside = R <= rmax
    shell = np.minimum(np.rint(R), M).astype(np.int32)
    del R
    wmean = np.bincount(shell[inside], W[inside], minlength=M + 1) / np.maximum(
        np.bincount(shell[inside], minlength=M + 1), 1)
    W = np.maximum(W, floor_frac * wmean[shell])  # shell-adaptive floor stabilises gaps in Fourier space
    inside &= W > 0
    W[~inside] = 1.0
    D /= W
    D[~inside] = 0
    vol = np.fft.irfftn(D, s=(M, M, M), axes=(0, 1, 2))
    ii = np.arange(-(N // 2), N - N // 2) % M
    return vol[np.ix_(ii, ii, ii)].astype(np.float32)


def _reconstruct_one(job):
    import mrcfile
    from ManifoldEM.star import parse_star
    star, out, pad, px = job
    df = parse_star(star, 0)
    stack = os.path.join(os.path.dirname(star), df.rlnImageName.values[0].split('@')[1])
    imgs = mrcfile.read(stack).astype(np.float32)
    vol = _backproject(imgs, df.rlnAngleRot.values, df.rlnAngleTilt.values, df.rlnAnglePsi.values, pad=pad)
    with mrcfile.new(out, overwrite=True) as m:
        m.set_data(vol)
        m.voxel_size = px
    return os.path.basename(out), len(imgs)


@command
def reconstruct(toml, ncpu=1, pad=2):
    """NumPy stand-in for `manifold-cli utility mrcs2mrc` (which needs RELION's relion_reconstruct).
    Writes the same postproc/vols/EulerAngles_*.mrc files."""
    import multiprocessing
    params = _load(toml)
    os.makedirs(params.postproc_mrcs2mrc_dir, exist_ok=True)
    stars = sorted(glob.glob(os.path.join(params.bin_dir, 'EulerAngles_*.star')),
                   key=lambda f: int(re.findall(r'_(\d+)_of_', f)[-1]))
    if not stars:
        raise UserError('no trajectory STAR files found: run step 7.3 first')
    jobs = [(s, os.path.join(os.path.abspath(params.postproc_mrcs2mrc_dir), os.path.basename(s)[:-5] + '.mrc'),
             pad, params.ms_pixel_size) for s in stars]
    t0 = time.time()
    with multiprocessing.get_context('fork').Pool(max(1, int(ncpu))) as pool:
        for i, (name, n) in enumerate(pool.imap(_reconstruct_one, jobs)):
            print(f'\r  {i + 1}/{len(jobs)} volumes  ({name}, {n} images, {time.time() - t0:.0f}s)', end='', flush=True)
    print()
    return dict(n=len(jobs), seconds=round(time.time() - t0, 1), out_dir=os.path.abspath(params.postproc_mrcs2mrc_dir))


@command
def relion_selftest(exe='relion_reconstruct'):
    """Reconstruct a tiny phantom from a STAR file in ManifoldEM's own format with relion_reconstruct."""
    import subprocess
    import tempfile
    import mrcfile
    exe = shutil.which(exe)
    if not exe:
        raise UserError('relion_reconstruct is not on PATH')
    N, n, apix = 32, 300, 2.0
    blobs = [((0, 0, 0), 3.0, 1.0), ((6, 0, 0), 2.0, 0.8), ((0, 8, 0), 2.0, 0.6), ((0, 0, -9), 1.6, 1.2)]
    z, y, x = np.mgrid[0:N, 0:N, 0:N] - N // 2
    truth = sum(a * np.exp(-((x - p[0]) ** 2 + (y - p[1]) ** 2 + (z - p[2]) ** 2) / (2 * s * s)) for p, s, a in blobs)
    rng = np.random.default_rng(0)
    rot, tilt = rng.uniform(0, 360, n), np.rad2deg(np.arccos(rng.uniform(-1, 1, n)))
    A = _relion_matrix(rot, tilt, np.zeros(n))
    yy, xx = np.mgrid[0:N, 0:N] - N // 2
    imgs = np.zeros((n, N, N), np.float32)
    for p, s, a in blobs:  # analytic projections of Gaussian blobs
        q = A @ np.asarray(p, dtype=float)
        imgs += (a * np.sqrt(2 * np.pi) * s * np.exp(-((xx[None] - q[:, 0, None, None]) ** 2
                 + (yy[None] - q[:, 1, None, None]) ** 2) / (2 * s * s))).astype(np.float32)
    d = tempfile.mkdtemp(prefix='relion_selftest_')
    with mrcfile.new(os.path.join(d, 'imgs.mrcs')) as m:
        m.set_data(imgs)
    with open(os.path.join(d, 'test.star'), 'w') as f:  # same layout as ManifoldEM.star.write_star
        f.write('\ndata_ \n \nloop_ \n \n_rlnImageName #1 \n_rlnAnglePsi #2 \n_rlnAngleTilt #3 \n'
                '_rlnAngleRot #4 \n_rlnDetectorPixelSize #5 \n_rlnMagnification #6 \n')
        for i in range(n):
            f.write(f'{i + 1}@imgs.mrcs 0.0 {tilt[i]} {rot[i]} {apix} 10000.0\n')
    r = subprocess.run([exe, '--i', 'test.star', '--o', 'test.mrc'], cwd=d, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError((r.stdout + r.stderr)[-2000:])
    with mrcfile.open(os.path.join(d, 'test.mrc')) as m:
        vol, vs = m.data.astype(float), float(m.voxel_size.x)
    shutil.rmtree(d)
    cc = float(np.corrcoef(vol.ravel(), truth.ravel())[0, 1])
    return dict(exe=exe, cc=cc, pixel_size=vs, ok=bool(cc > 0.98 and abs(vs - apix) < 1e-3))


# -------------------------------------------------------------------------------- free Python code
@command
def pyexec(toml, code, fig_dir):
    """Run user code with ManifoldEM's API preloaded (used by the %%manifoldem cell magic)."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from ManifoldEM.data_store import data_store, Anchor, Sense
    params = _load(toml) if toml else None
    namespace = dict(np=np, plt=plt, params=params, data_store=data_store, Anchor=Anchor, Sense=Sense)
    if params is not None:
        namespace['prds'] = _prds()
    exec(compile(code, '<%%manifoldem>', 'exec'), namespace)
    os.makedirs(fig_dir, exist_ok=True)
    figs = []
    for num in plt.get_fignums():
        path = os.path.join(fig_dir, f'figure_{num}.png')
        plt.figure(num).savefig(path, dpi=100, bbox_inches='tight')
        figs.append(path)
    plt.close('all')
    return dict(figures=figs)


if __name__ == '__main__':
    cmd, args_file, result_file = sys.argv[1], os.path.abspath(sys.argv[2]), os.path.abspath(sys.argv[3])
    with open(args_file) as f:
        kwargs = json.load(f)
    try:
        result = COMMANDS[cmd](**kwargs)
    except UserError as exc:
        print(f'\nERROR: {exc}', flush=True)
        sys.exit(2)
    with open(result_file, 'w') as f:
        json.dump(result, f, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
'''
BRIDGE_PATH = ROOT / 'mem_bridge.py'
BRIDGE_PATH.write_text(BRIDGE_SOURCE)
print(f'✔ wrote {BRIDGE_PATH}')

In [ ]:
#@title 0.4 · Load the notebook helpers { display-mode: "form" }
#@markdown Plotting and bookkeeping functions used by the cells below. Expand the cell to read them.
# Notebook-side helpers. They run in Colab's own Python and never import ManifoldEM: pipeline steps run
# as `manifold-cli` subprocesses in the ManifoldEM environment, project queries go through mem_bridge.py,
# and plots read ManifoldEM's output files (h5 / pickle / png / gif / mrc) directly.
import base64
import glob
import io
import json
import os
import pickle
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import h5py
import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, Markdown, display

IN_COLAB = 'google.colab' in sys.modules


def _read_pickle(path):
    with open(path, 'rb') as f, warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)  # numpy-1 pickles opened with numpy 2
        return pickle.load(f)


def _h5(path, *keys):
    with h5py.File(path, 'r') as f:
        return {k: f[k][()] for k in (keys or f.keys()) if k in f}


def _human(nbytes):
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if nbytes < 1024 or unit == 'TB':
            return f'{nbytes:.1f} {unit}'
        nbytes /= 1024


def _du(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) if Path(path).exists() else 0


class Session:
    """State shared by all notebook cells: paths, the current project and its latest summary."""

    def __init__(self, root, env_dir, bridge_path):
        self.root = Path(root)
        self.env = Path(env_dir)
        self.py = self.env / 'bin' / 'python'
        self.cli_exe = self.env / 'bin' / 'manifold-cli'
        self.bridge_path = Path(bridge_path)
        self.work = self.root / 'work'  # all projects live here: params_<name>.toml + output/<name>/
        self.data = self.root / 'data'
        self.tmp = self.root / 'tmp'
        for d in (self.work, self.data, self.tmp, self.work / '.notebook'):
            d.mkdir(parents=True, exist_ok=True)
        self.ncpu = os.cpu_count() or 1
        self.project = None
        self._S, self._A, self._P = {}, {}, {}
        try:
            import psutil
            self.ram_gb = psutil.virtual_memory().total / 1e9
        except ImportError:
            self.ram_gb = float('nan')

    def _loaded(self, data):
        if not self.project or not data:
            raise RuntimeError('No project selected: run step 2.1 (create) or 2.2 (re-open) first.')
        return data

    S = property(lambda self: self._loaded(self._S))  # latest project summary (from mem_bridge.py summary)
    A = property(lambda self: self._loaded(self._A))  # its arrays (occupancy, cluster ids, directions, ...)
    P = property(lambda self: self._loaded(self._P))  # paths and path templates of ManifoldEM's output files

    def workers_for(self, gb_per_worker, ncpu=None, what='this step'):
        """How many parallel workers fit in RAM (75 %); warns when even one may not."""
        ncpu = ncpu or self.ncpu
        if not np.isfinite(self.ram_gb):
            return ncpu
        n = max(1, min(ncpu, int(0.75 * self.ram_gb / max(gb_per_worker, 1e-3))))
        if gb_per_worker > 0.8 * self.ram_gb:
            print(f'⚠️ {what} needs about {gb_per_worker:.0f} GB per PD but the runtime has {self.ram_gb:.0f} GB: it will '
                  'probably crash. Downsample the images (1.5), lower PRD_THRES_HIGH (3.1) or use a high-RAM runtime.')
        elif n < ncpu:
            print(f'ℹ️  {what}: {n} parallel worker(s) instead of {ncpu}, to fit in {self.ram_gb:.0f} GB of RAM')
        return n

    # ----------------------------------------------------------------------------- processes
    def run(self, cmd, cwd=None, echo=True, check=True, hide=()):
        """Run a command, streaming its output live; the stop button kills it and all its worker processes.

        Lines starting with one of the `hide` prefixes are not shown."""
        cmd = [str(c) for c in cmd]
        if echo:
            shown = ' '.join(shlex.quote(c) for c in cmd).replace(str(self.env / 'bin') + '/', '')
            print(f'$ {shown}', flush=True)
        self._backup_toml()
        env = dict(os.environ, PYTHONUNBUFFERED='1', MPLBACKEND='Agg', OMP_NUM_THREADS='1')
        t0 = time.time()
        proc = subprocess.Popen(cmd, cwd=cwd or self.work, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, start_new_session=True)
        tail, pending = '', ''
        try:
            while True:
                chunk = proc.stdout.read1(65536)
                if not chunk:
                    break
                text = chunk.decode('utf-8', 'replace')
                tail = (tail + text)[-3000:]
                if hide:  # filter whole lines only; keep partial lines (e.g. progress bars) flowing
                    pending += text
                    *lines, pending = pending.split('\n')
                    text = ''.join(l + '\n' for l in lines if not l.startswith(hide))
                    if '\r' in pending:
                        text, pending = text + pending, ''
                sys.stdout.write(text)
                sys.stdout.flush()
            if pending and not pending.startswith(hide):
                sys.stdout.write(pending)
            rc = proc.wait()
        except KeyboardInterrupt:
            self._kill(proc)
            raise
        if check and rc != 0:
            last = [l for l in tail.replace('\r', '\n').splitlines() if l.strip()]
            hint = ' (killed: probably out of memory, see section 11)' if rc in (-9, 137) else ''
            raise RuntimeError(f'command failed with exit code {rc}{hint}: {last[-1] if last else ""}')
        return time.time() - t0

    @staticmethod
    def _kill(proc):
        """Stop a command and every process it started; a second press of stop forces it."""
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=10)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            pass
        finally:
            try:
                os.killpg(proc.pid, signal.SIGKILL)  # also any worker still alive in the group
            except ProcessLookupError:
                pass
            proc.wait()

    def _backup_toml(self):
        """Keep a copy of the project's .toml: a stop while ManifoldEM rewrites it could leave it empty."""
        if self.project:
            f, bak = Path(self.toml), self.work / '.notebook' / f'params_{self.project}.toml.bak'
            if f.exists() and f.stat().st_size > 0 and '[params]' in f.read_text():
                shutil.copy(f, bak)

    def _check_toml(self, name):
        f, bak = self.work / f'params_{name}.toml', self.work / '.notebook' / f'params_{name}.toml.bak'
        if (not f.exists() or f.stat().st_size == 0 or '[params]' not in f.read_text()) and bak.exists():
            shutil.copy(bak, f)
            print(f'ℹ️  {f.name} was empty or damaged (probably stopped while it was being written): restored the last copy')

    def cli(self, *args, ncpu=None):
        """`manifold-cli -n <ncpu> <args...> params_<project>.toml`, run from the work folder."""
        hide = () if args[0] == 'init' else ('Changing param',)  # step flags are not used: the .toml is
        dt = self.run([self.cli_exe, '-n', ncpu or self.ncpu, *args, self.toml], hide=hide)
        print(f'✔ {" ".join(map(str, args))} finished in {dt / 60:.1f} min\n')
        return dt

    def bridge(self, command, **kwargs):
        """Call a mem_bridge.py command inside the ManifoldEM environment and return its JSON result."""
        d = Path(tempfile.mkdtemp(dir=self.tmp))
        try:
            (d / 'args.json').write_text(json.dumps(kwargs, default=str))
            self.run([self.py, self.bridge_path, command, d / 'args.json', d / 'result.json'], echo=False)
            return json.loads((d / 'result.json').read_text())
        finally:
            shutil.rmtree(d, ignore_errors=True)

    # ------------------------------------------------------------------------------- projects
    @property
    def toml(self):
        if not self.project:
            raise RuntimeError('No project selected: run step 2.1 (create) or 2.2 (re-open) first.')
        return str(self.work / f'params_{self.project}.toml')

    def projects(self, valid_only=False):
        names = sorted(p.name[len('params_'):-len('.toml')] for p in self.work.glob('params_*.toml'))
        for n in names:
            self._check_toml(n)  # repair a .toml emptied by a stop while it was being written
        return [n for n in names if self._stored_name(n) == n] if valid_only else names

    def _stored_name(self, name):
        import tomllib
        try:
            return tomllib.loads((self.work / f'params_{name}.toml').read_text()).get('params', {}).get('project_name')
        except Exception:
            return None

    def use(self, name):
        import tomllib
        f = self.work / f'params_{name}.toml'
        self._check_toml(name)
        if not f.exists():
            raise FileNotFoundError(f'no project called {name!r} in {self.work} (found: {self.projects()})')
        marker = self.work / '.notebook' / f'{name}.restoring'
        if marker.exists():
            raise RuntimeError(f'the restore of project {name} from {marker.read_text().strip()} was interrupted, so its '
                               'files are a mix of two versions: run 2.2 again with RESTORE_FROM_DRIVE to finish it')
        stored = tomllib.loads(f.read_text()).get('params', {}).get('project_name')
        if stored != name:  # ManifoldEM writes to output/<project_name> and params_<project_name>.toml
            raise RuntimeError(f'{f.name} belongs to project {stored!r} (a copied or renamed file): ManifoldEM would '
                               f'change params_{stored}.toml and output/{stored}/. Create a new project in 2.1 instead.')
        self.project = name
        return self.refresh()

    @property
    def state_file(self):
        return self.work / '.notebook' / f'{self.project}.json'

    def state(self):
        return json.loads(self.state_file.read_text()) if self.state_file.exists() else {}

    def save_state(self, **values):
        st = self.state()
        st.update(values)
        self.state_file.write_text(json.dumps(st, indent=1, default=str))

    def refresh(self):
        """Re-read the project (after every step, since subprocesses change it on disk)."""
        self._check_toml(self.project)
        self._summary()
        now = {k: round(self._S[k], 6) if isinstance(self._S[k], float) else self._S[k] for k in self.SECTION4}
        seen = self.state().get('section4')
        if seen != now:   # recorded only once the bookkeeping is done: a stop half-way is finished by the next refresh
            if seen:
                self._section4_changed(seen, now)
                self._summary()
            self.save_state(section4=now)
        return self._S

    def _summary(self):
        npz = self.tmp / f'summary_{self.project}.npz'
        self._S = self.bridge('summary', toml=self.toml, out_npz=str(npz))
        with np.load(npz) as z:
            self._A = {k: z[k] for k in z.files}
        self._P = self._S['paths']

    # Section 4 parameters. A change, whether from 4.0 or from params.save() in a %%manifoldem cell, marks the steps
    # that depend on it for 4.1 (outputs older than the change count as missing) and clears what no longer holds.
    SECTION4 = ('num_psi', 'distance_filter_type', 'distance_filter_cutoff_freq', 'distance_filter_order',
                'nlsa_tune', 'con_order_range', 'nlsa_fps')

    def _section4_changed(self, old, new):
        changes = {k: (old[k], new[k]) for k in new if k in old and old[k] != new[k]}
        for k, (a, b) in changes.items():
            print(f'✔ {k}: {a} → {b}')
        if not changes or not glob.glob(self._P['dist'].format(p='*')):
            return                     # nothing computed yet
        redo = set()
        if {'distance_filter_type', 'distance_filter_cutoff_freq', 'distance_filter_order'} & set(changes):
            redo |= {'calc-distance', 'manifold-analysis', 'psi-analysis', 'nlsa-movie'}
        if 'nlsa_tune' in changes:
            redo |= {'manifold-analysis', 'psi-analysis', 'nlsa-movie'}
        if 'con_order_range' in changes:
            redo |= {'psi-analysis', 'nlsa-movie'}
        if 'num_psi' in changes or 'nlsa_fps' in changes:
            redo |= {'nlsa-movie'}     # NLSA of ψ that were never computed (or are out of date) is found by 4.1 itself
        new_movies = set(changes) - {'nlsa_fps', 'num_psi'}
        pending = self.state().get('redo') or {}
        pending = dict.fromkeys(pending, 0) if isinstance(pending, list) else dict(pending)
        pending.update({step: time.time() for step in redo})
        self.save_state(redo=pending)
        if 'psi-analysis' in pending:   # 4.1 recomputes ψ1..NUM_PSI only: NLSA results of higher ψ must not survive
            self.bridge('drop-psi2', toml=self.toml, from_k=new['num_psi'], since=pending['psi-analysis'])
        keep = None
        if new_movies or 'num_psi' in changes:
            self.bridge('clean', toml=self.toml, what=['cc', 'landscape', 'trajectory', 'volumes'])
            old_anchors = [[int(a), int(k), 'FWD' if s > 0 else 'REV'] for a, k, s in self._A['anchors']]
            keep = [a for a in old_anchors if not new_movies and a[1] <= new['num_psi']]
            res = self.bridge('set-anchors', toml=self.toml, anchors=keep, trash=[int(t) for t in self._A['trash']],
                              check=False)
            dropped = [f'{a + 1}:{k}:{s}' for a, k, s in old_anchors if [a, k, s] not in keep]
            if new_movies:
                print('⚠️ the ψ movies will change: the sign (and sometimes the order) of eigenvectors is arbitrary, so a '
                      'movie may now run the other way. The anchors were cleared; after 4.1, watch the movies again (6.1) '
                      'and enter the anchors in 6.2. Results of sections 7–8 were deleted.')
            else:
                print('ℹ️  results of sections 7–8 were deleted (they depend on NUM_PSI)'
                      + (f"; anchors dropped because their ψ > {new['num_psi']}: {', '.join(dropped)}" if dropped else ''))
                for comp in res['unanchored_components']:
                    print(f'⚠️ no anchor left among PDs {comp}: set one in 6.2')
        print(f'ℹ️  4.1 will recompute all PDs for: {self._steps(redo)}')
        if 'num_psi' in changes and new['num_psi'] > old['num_psi'] and 'psi-analysis' not in pending:
            need = self.bridge('incomplete', toml=self.toml, step='psi-analysis')['prds']
            if need:
                k0, k1 = old['num_psi'], new['num_psi']
                print(f'ℹ️  4.1 will also run NLSA for {len(need)} PD(s) that have no current ψ{k0 + 1}–ψ{k1}. ManifoldEM '
                      'recomputes every ψ of such a PD, so on real data this is the long step'
                      + (f' (ψ1–ψ{k0} come out the same, so their anchors are kept).' if keep else '.'))

    STEPS4 = ('calc-distance', 'manifold-analysis', 'psi-analysis', 'nlsa-movie')   # = mem_bridge.SECTION4

    def _steps(self, steps):
        """Section 4 step names in pipeline order."""
        return ', '.join(sorted(steps, key=lambda s: self.STEPS4.index(s) if s in self.STEPS4 else len(self.STEPS4)))

    NEEDED_FOR = dict(star='re-binning in 3.1', stack='distances and NLSA in 4.1, and 7.2', mask='distances in 4.1')

    def missing_inputs(self, keys=('star', 'stack', 'mask')):
        """The project's input files among `keys` that are not on disk (e.g. after a restore without them)."""
        return [f for f in (self.S['files'][k] for k in keys) if f and not os.path.exists(f)]

    def inputs_hint(self):
        ds = self.state().get('dataset', {})
        cells = ['1.1', {'synthetic': '1.2', 'ryr1': '1.3', 'own': '1.4'}.get(ds.get('kind'), 'the dataset cell')]
        if ds.get('downsampled_by'):
            cells.append(f"1.5 (DOWNSAMPLE_BY = {ds['downsampled_by']:g})")
        hint = f"run {', '.join(cells[:-1])} and {cells[-1]} again with the same settings: that re-creates them in the same place"
        if ds.get('kind') == 'own':
            hint += ' (files on Google Drive: tick USE_GOOGLE_DRIVE in 1.1; uploaded files: upload the same files again)'
        return hint + '. Or restore a checkpoint saved with INCLUDE_INPUT_DATA (10.1).'


    def require_inputs(self, *keys):
        missing = self.missing_inputs(keys or ('star', 'stack', 'mask'))
        if missing:
            raise RuntimeError(f"the project's input files are missing ({', '.join(missing)}): {self.inputs_hint()}")

    # stage -> (key in S['done'], key in S['stale'], label)
    STEPS = [('binning', None, 'Bin particles into PDs (3)'), ('calc_distance', None, 'Distances (4)'),
             ('manifold_analysis', None, 'Diffusion-map embedding (4)'), ('psi_analysis', None, 'NLSA (4)'),
             ('nlsa_movie', None, 'Movies (4)'), ('find_ccs', 'cc', 'Conformational coordinate (7.1)'),
             ('probability_landscape', 'landscape', 'Energy landscape (7.2)'),
             ('trajectory', 'trajectory', 'Per-state image stacks (7.3)'), ('volumes', 'volumes', 'Volumes (8.1)'),
             ('denoise', 'denoise', 'Denoised volumes (8.2)')]

    def status(self):
        d, stale = self.S['done'], {k: v for k, v in self.S['stale'].items() if k != '_keys'}
        rows = ''
        for key, stage, label in self.STEPS:
            mark = '✅' if d.get(key) else ('⚠️' if stage in stale else '⬜')
            note = f' — out of date: {stale[stage]}' if stage in stale else (f' — {d[key]}' if key == 'volumes' and d[key] else '')
            rows += f'<tr><td>{mark}</td><td>{label}{note}</td></tr>'
        redo = self.state().get('redo')
        if redo:
            rows += (f'<tr><td>⚠️</td><td>pending after a change of the section 4 parameters (4.0 or params.save()): 4.1 '
                     f'must recompute {self._steps(redo)}</td></tr>')
        display(HTML(f"<b>Project <code>{self.project}</code></b> — {self.S['n_pd']} PDs, "
                     f"ManifoldEM {self.S['version']}<table>{rows}</table>"))

    def require(self, key, what):
        """Stop a cell early, with the reason, when an earlier step is missing or out of date."""
        stage = dict(embedding='embedding', find_ccs='cc', probability_landscape='landscape', trajectory='trajectory',
                     volumes='volumes', denoise='denoise').get(key)
        if key == 'embedding' and self.state().get('redo'):
            raise RuntimeError(f"the section 4 (embedding) parameters changed: run 4.1 first (pending: {self._steps(self.state()['redo'])})")
        d = self.S['done']
        if (key == 'embedding' and not d.get('embedding') and 'embedding' not in self.S['stale']
                and all(d.get(k) for k in ('calc_distance', 'manifold_analysis', 'psi_analysis', 'nlsa_movie'))):
            raise RuntimeError(f'{what} first: its results are all on disk but not recorded as current (a step that checks '
                               'them was stopped, e.g. a re-binning in 3.1). With ONLY_MISSING_PDS ticked, 4.1 only checks '
                               'and records them, in seconds')
        if not self.S['done'].get(key):
            why = self.S['stale'].get(stage)
            raise RuntimeError(f'{what} first' + (f' (its results are out of date: {why})' if why else ''))

    def stage(self, stage, info=None, begin=False):
        """Ledger bookkeeping for a stage: forget it before computing (begin=True), record it after."""
        self.bridge('stage-begin' if begin else 'stage-end', toml=self.toml, stage=stage,
                    **({} if begin else dict(info=info)))

    # ------------------------------------------------------------------------------ geometry
    def _disk(self, vecs):
        """Lambert azimuthal equal-area map of unit vectors around the hemisphere axis (edge = 90°)."""
        axis = np.asarray(self.S['hemisphere_vec'], dtype=float)
        axis /= np.linalg.norm(axis)
        ref = np.array([0.0, 0.0, 1.0]) if abs(axis[2]) < 0.9 else np.array([0.0, 1.0, 0.0])
        e1 = np.cross(ref, axis)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(axis, e1)
        v = np.asarray(vecs, dtype=float)
        c = np.clip(axis @ v, -1, 1)
        r = np.sqrt(np.maximum(2 * (1 - c), 0))
        az = np.arctan2(e2 @ v, e1 @ v)
        return r * np.cos(az), r * np.sin(az)

    def _disk_axes(self, ax, title=''):
        for deg in (30, 60, 90):
            r = np.sqrt(2 * (1 - np.cos(np.deg2rad(deg))))
            ax.add_patch(plt.Circle((0, 0), r, fill=False, lw=0.6, color='0.75', ls='-' if deg == 90 else ':'))
            ax.text(r * 0.71, r * 0.71, f'{deg}°', fontsize=7, color='0.55')
        ax.set_aspect('equal')
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-1.5, 1.5)
        ax.axis('off')
        ax.set_title(title, fontsize=10)

    # ---------------------------------------------------------------------------- 1. inputs
    @staticmethod
    def show_inputs(report, npz=None):
        ok = not report.get('problems')
        lines = [f"**{'✅ Inputs look usable' if ok else '❌ ManifoldEM will not be able to use these inputs'}**", '']
        if 'n_particles' in report:
            lines.append(f"- {report['n_particles']:,} particles in the STAR file, {report['n_images']:,} images of "
                         f"{report['box']}×{report['box']} px in the stack")
        px = [f'{v:.4g} Å ({k})' for k, v in (('STAR', report.get('star_pixel_size')),
                                               ('MRC header', report.get('header_pixel_size'))) if v]
        if px:
            lines.append('- pixel size: ' + ', '.join(px))
        if 'voltage' in report:
            lines.append(f"- {report['voltage']:g} kV, Cs {report.get('cs', float('nan')):g} mm, "
                         f"amplitude contrast {report.get('amp_contrast', float('nan')):g}")
        lines += [f'- ❌ {p}' for p in report.get('problems', [])] + [f'- ⚠️ {w}' for w in report.get('warnings', [])]
        display(Markdown('\n'.join(lines)))
        if not npz or not os.path.exists(npz):
            return
        with np.load(npz) as z:
            imgs, idx = z['images'], z['image_index']
            rot, tilt, defocus = z['rot'], z['tilt'], z['defocus']
        n = len(imgs)
        fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(1.6 * ((n + 1) // 2), 3.4))
        for ax in axes.ravel():
            ax.axis('off')
        for ax, im, i in zip(axes.ravel(), imgs, idx):
            lo, hi = np.percentile(im, [1, 99])
            ax.imshow(im, cmap='gray', vmin=lo, vmax=hi)
            ax.set_title(f'#{i + 1}', fontsize=8)
        fig.suptitle('Random particle images', fontsize=10)
        plt.tight_layout()
        plt.show()
        fig = plt.figure(figsize=(12, 3.6))
        ax = fig.add_subplot(1, 2, 1, projection='mollweide')
        lon = np.deg2rad((rot + 180) % 360 - 180)
        lat = np.deg2rad(90 - tilt)
        ax.scatter(lon, lat, s=1, alpha=0.25, color='tab:blue', rasterized=True)
        ax.grid(True, lw=0.4)
        ax.set_title('Viewing directions (rot, tilt)', fontsize=10)
        ax.tick_params(labelsize=7)
        ax2 = fig.add_subplot(1, 2, 2)
        ax2.hist(defocus / 1e4, bins=60, color='tab:gray')
        ax2.set_xlabel('mean defocus (µm)')
        ax2.set_ylabel('particles')
        ax2.set_title('Defocus distribution', fontsize=10)
        plt.tight_layout()
        plt.show()

    # --------------------------------------------------------------------------- 3. binning
    def show_binning(self):
        S, A = self.S, self.A
        occ_full, centres = A['occupancy_full'], A['bin_centers']
        low, high = S['thres_low'], S['thres_high']
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1.1, 1]))
        x, y = self._disk(centres)
        nz = occ_full > 0
        sc = ax.scatter(x[nz], y[nz], c=np.log10(occ_full[nz]), s=14, cmap='viridis', vmin=0)
        keep = occ_full >= low
        ax.scatter(x[keep], y[keep], s=60, facecolors='none', edgecolors='crimson', lw=1.2,
                   label=f'PD (≥ {low} particles)')
        cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        cb.set_label('log10(particles per bin)')
        ax.legend(loc='lower left', fontsize=8, frameon=False)
        self._disk_axes(ax, 'Orientation bins on the half-sphere')
        ax2.hist(occ_full[nz], bins=min(80, max(10, nz.sum() // 3)), color='tab:gray')
        ax2.axvline(low, color='crimson', lw=1.5, label=f'low threshold = {low}')
        top = occ_full.max()
        if high <= 1.5 * top:
            ax2.axvline(high, color='tab:green', lw=1.5, ls='--', label=f'high threshold (cap) = {high}')
        else:  # keep the histogram readable; the cap is far to the right
            ax2.plot([], [], ' ', label=f'high threshold (cap) = {high}, off the scale')
            ax2.set_xlim(0, 1.15 * top)
        ax2.set_xlabel('particles per orientation bin')
        ax2.set_ylabel('bins')
        ax2.legend(fontsize=8)
        ax2.set_title('Bin occupancy', fontsize=10)
        plt.tight_layout()
        plt.show()
        msg = (f"**{S['n_pd']} projection directions (PDs)** out of {S['n_bins']} bins · "
               f"{S['n_used']:,} of {S['n_particles']:,} particles used · {S['n_clusters']} connected "
               f"component(s) · Shannon angle {S['shannon_angle']:.4f} rad · bin width {S['ang_width']:.4f} rad "
               f"(aperture index {S['aperture_index']})")
        warn = []
        if S['n_pd'] < 3:
            warn.append('fewer than 3 PDs: lower the low threshold or raise the aperture index')
        if low < 2 * S['con_order_range']:
            warn.append(f"the low threshold ({low}) is below 2 × con_order_range ({2 * S['con_order_range']}): "
                        'NLSA needs more images per PD; raise it or lower con_order_range in step 4.0')
        if S['n_clusters'] > 1:
            warn.append(f"{S['n_clusters']} separate groups of PDs: each group needs its own anchor in step 6")
        display(Markdown(msg + ''.join(f'\n- ⚠️ {w}' for w in warn)))

    def show_sphere_3d(self):
        import plotly.graph_objects as go
        A = self.A
        u, v = np.mgrid[0:2 * np.pi:40j, 0:np.pi:20j]
        sphere = go.Surface(x=np.cos(u) * np.sin(v), y=np.sin(u) * np.sin(v), z=np.cos(v), opacity=0.15,
                            showscale=False, colorscale=[[0, 'lightgrey'], [1, 'lightgrey']], hoverinfo='skip')
        p = A['pos_pd']
        comp = self._clusters()   # as in 5.x / 6.x: removing PDs can split a cluster (-1 = removed)
        cmap = plt.get_cmap('tab10')
        col = ['rgb(%d,%d,%d)' % tuple(int(255 * v) for v in cmap(c % 10)[:3]) if c >= 0 else 'lightgrey' for c in comp]
        pds = go.Scatter3d(x=p[0], y=p[1], z=p[2], mode='markers+text', text=[str(i + 1) for i in range(p.shape[1])],
                           textfont=dict(size=9), marker=dict(size=6, color=col),
                           hovertext=[f'PD {i + 1}: {o} particles, ' + (f'cluster {c}' if c >= 0 else 'removed (6.2)')
                                      for i, (o, c) in enumerate(zip(A['occupancy'], comp))], hoverinfo='text', name='PDs')
        q = A['pos_particles']
        parts = go.Scatter3d(x=q[0], y=q[1], z=q[2], mode='markers', marker=dict(size=1.5, color='grey', opacity=0.3),
                             hoverinfo='skip', name='particles (sample)')
        fig = go.Figure([sphere, parts, pds])
        fig.update_layout(height=550, margin=dict(l=0, r=0, t=30, b=0), title='PDs on the sphere (hover for details)',
                          scene=dict(aspectmode='cube', xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
        fig.show()

    # ----------------------------------------------------------------------------- 5. PDs
    def pd_table(self, top=None):
        import pandas as pd
        A, S = self.A, self.S
        anchors = {int(a[0]): a for a in A['anchors']}
        trash = set(int(t) for t in A['trash'])
        role = []
        for i in range(S['n_pd']):
            if i in anchors:
                role.append(f"anchor ψ{anchors[i][1]} {'FWD' if anchors[i][2] > 0 else 'REV'}")
            else:
                role.append('removed' if i in trash else '')
        # cluster = connected group of PDs as 7.1 sees it: removing PDs can split a cluster (-1 = removed)
        df = pd.DataFrame(dict(PD=np.arange(1, S['n_pd'] + 1), particles=A['occupancy'], used=A['n_used'],
                               cluster=self._clusters(), theta=np.round(A['theta'], 1), phi=np.round(A['phi'], 1),
                               role=role))
        df = df.sort_values('particles', ascending=False)
        return df.head(top) if top else df

    def _clusters(self):
        return self.A['component_ids'] if 'component_ids' in self.A else self.A['cluster_ids']

    def _gif_html(self, path, width=150, label=''):
        """Re-encode ManifoldEM's GIF (which plays once) as a looping GIF, downscaled for display."""
        from PIL import Image as PILImage, ImageSequence
        if not os.path.exists(path):
            return f'<div style="width:{width}px;font-size:11px;color:#999">{label}<br>(missing)</div>'
        with PILImage.open(path) as im:
            frames = [f.convert('L') for f in ImageSequence.Iterator(im)]
            duration = im.info.get('duration', 200)
        if frames[0].width > width:
            frames = [f.resize((width, width), PILImage.LANCZOS) for f in frames]
        frames = [f.point(lambda v: v // 4 * 4) for f in frames]  # 64 grey levels: about half the size
        buf = io.BytesIO()
        frames[0].save(buf, format='GIF', save_all=True, append_images=frames[1:], loop=0, duration=duration)
        b64 = base64.b64encode(buf.getvalue()).decode()
        return (f'<div style="display:inline-block;margin:2px;text-align:center;font-size:11px">'
                f'<img src="data:image/gif;base64,{b64}" width="{width}"><br>{label}</div>')

    def movies_html(self, pd1, psis=None, width=150):
        psis = psis or range(1, self.S['num_psi'] + 1)
        return ''.join(self._gif_html(self.P['gif'].format(p=pd1, k=k), width, f'ψ{k}') for k in psis)

    def _pd_header(self, pd1):
        A = self.A
        i = pd1 - 1
        anchors = {int(a[0]): a for a in A['anchors']}
        role = ('anchor ψ%d %s' % (anchors[i][1], 'FWD' if anchors[i][2] > 0 else 'REV')) if i in anchors else (
            'removed' if i in set(A['trash'].tolist()) else '')
        return (f"<h4 style='margin:4px 0'>PD {pd1} · {A['occupancy'][i]} particles ({A['n_used'][i]} used) · "
                f"cluster {self._clusters()[i]} · θ={A['theta'][i]:.1f}° φ={A['phi'][i]:.1f}°"
                f"{' · <b>' + role + '</b>' if role else ''}</h4>")

    def _pd_figure(self, pd1):
        """Class average, eigenvalue spectrum, kernel-bandwidth fit and diffusion-map embedding of one PD."""
        S, P = self.S, self.P
        i = pd1 - 1
        fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
        ca = P['class_avg'].format(p=pd1)
        if os.path.exists(ca):
            axes[0].imshow(plt.imread(ca).T, cmap='gray')  # transposed to match the movie frames
        axes[0].set_title('class average', fontsize=9)
        axes[0].axis('off')
        es = P['eig_spec'].format(p=pd1)
        if os.path.exists(es):
            e = np.atleast_2d(np.loadtxt(es))
            axes[1].bar(e[:, 0], e[:, 1], color=['tab:blue' if j <= S['num_psi'] else 'lightgrey' for j in e[:, 0]])
            axes[1].set_xticks(e[:, 0].astype(int))
            axes[1].set_xlabel('eigenvector')
        axes[1].set_title(f'eigenvalues (blue: ψ1–ψ{S["num_psi"]} analysed)', fontsize=9)
        axes[2].axis('off')
        axes[3].axis('off')
        psi_file = P['psi'].format(p=i)
        if os.path.exists(psi_file):
            d = _h5(psi_file)
            if {'logEps', 'logSumWij', 'popt'} <= set(d):  # absent after a manual re-embedding
                popt = d['popt']
                axes[2].axis('on')
                axes[2].plot(d['logEps'], d['logSumWij'], '.', ms=2, color='tab:gray')
                axes[2].plot(d['logEps'], popt[3] + popt[2] * np.tanh(popt[0] * d['logEps'] + popt[1]), 'tab:red')
                axes[2].axvline(-popt[1] / popt[0], color='tab:orange', ls='--', lw=1)   # fitted midpoint
                if 'sigma' in d:   # the kernel width used: ε = σ²/2 = NLSA_TUNE² × the midpoint's ε
                    axes[2].axvline(np.log(float(np.ravel(d['sigma'])[0]) ** 2 / 2), color='tab:green', ls=':', lw=1.2)
                a, b = float(popt[0]), float(popt[1])
                if np.isfinite(a) and np.isfinite(b) and abs(a) > 1e-6:   # the grid spans ±150: zoom on the bend
                    mid, half = -b / a, 6 / abs(a)
                    lo, hi = max(mid - half, d['logEps'].min()), min(mid + half, d['logEps'].max())
                    sel = (d['logEps'] >= lo) & (d['logEps'] <= hi)
                    if sel.any():
                        axes[2].set_xlim(lo, hi)
                        yy = d['logSumWij'][sel]
                        axes[2].set_ylim(yy.min() - 0.5, yy.max() + 0.5)
                axes[2].set_xlabel('ln ε')
                axes[2].set_ylabel('ln Σ W')
                axes[2].set_title(f'kernel bandwidth (dim ≈ {2 * popt[0] * popt[2]:.1f})', fontsize=9)
            psi = d['psi']
            rot = None
            dist_file = P['dist'].format(p=i)
            if os.path.exists(dist_file) and 'posPath' in d:
                rot = _h5(dist_file, 'rotations').get('rotations')
                rot = None if rot is None else (rot[np.asarray(d['posPath']).ravel()] + 180) % 360 - 180
            axes[3].axis('on')
            cyc = dict(cmap='twilight', vmin=-180, vmax=180) if rot is not None else {}   # cyclic: closes at ±180°
            s = axes[3].scatter(psi[:, 0], psi[:, 1], c=rot, s=6, **cyc)
            if rot is not None:
                plt.colorbar(s, ax=axes[3], fraction=0.05, pad=0.02).set_label('in-plane rotation (°)', fontsize=8)
            axes[3].set_xlabel('ψ1')
            axes[3].set_ylabel('ψ2')
            axes[3].set_title('diffusion map embedding', fontsize=9)
        plt.tight_layout()
        return fig

    def _pd_movies(self, pd1, width=150):
        return f'<div><b>NLSA movies</b> (one per eigenvector ψ)</div>{self.movies_html(pd1, width=width)}'

    def show_pd(self, pd1, diagnostics=True, width=150):
        """Everything ManifoldEM computed for one PD (1-based, as in topos/PrD_<n>)."""
        if not 1 <= pd1 <= self.S['n_pd']:
            print(f'PD must be between 1 and {self.S["n_pd"]}')
            return
        display(HTML(self._pd_header(pd1)))
        if diagnostics:
            self._pd_figure(pd1)
            plt.show()
        display(HTML(self._pd_movies(pd1, width)))

    def pd_browser(self, width=150):
        """Dropdown + slider over PDs. Built from HTML/Image widgets (not an Output widget), which update by value."""
        import ipywidgets as w
        order = self.pd_table()
        options = [(f"PD {r.PD} · {r.particles} particles · cluster {r.cluster}{' · ' + r.role if r.role else ''}",
                    int(r.PD)) for r in order.itertuples()]
        dropdown = w.Dropdown(options=options, description='Jump to', layout=w.Layout(width='520px'))
        slider = w.IntSlider(value=options[0][1], min=1, max=self.S['n_pd'], description='PD',
                             continuous_update=False, layout=w.Layout(width='520px'))
        header, movies = w.HTML(), w.HTML()
        figure = w.Image(format='png', layout=w.Layout(max_width='100%'))

        def render(_=None):
            header.value = self._pd_header(slider.value)
            fig = self._pd_figure(slider.value)
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=80)
            plt.close(fig)
            figure.value = buf.getvalue()
            movies.value = self._pd_movies(slider.value, width)

        def jump(change):
            slider.value = change['new']

        dropdown.observe(jump, names='value')
        slider.observe(render, names='value')
        render()
        display(w.VBox([dropdown, slider, header, figure, movies]))

    # ----------------------------------------------------------------------------- 6. anchors
    def anchor_candidates(self, per_cluster=2, psis=(1, 2, 3), width=120, max_clusters=6):
        df = self.pd_table()
        df = df[df.role != 'removed']
        sizes = df.cluster.value_counts()
        shown = list(sizes.index[:max_clusters])
        html = ['<table style="border-collapse:collapse">']
        if len(sizes) > max_clusters:
            rest = ', '.join(f'cluster {c} ({n} PDs, top PD {int(df[df.cluster == c].iloc[0].PD)})'
                             for c, n in sizes.iloc[max_clusters:].items())
            html.insert(0, f'<p>Movies for the {max_clusters} largest clusters. Others (use the 5.2 browser): {rest}</p>')
        for cluster in shown:
            sub = df[df.cluster == cluster].head(per_cluster)
            html.append(f'<tr><td colspan="{len(psis) + 1}" style="padding-top:8px"><b>Cluster {cluster}</b> '
                        f'({(df.cluster == cluster).sum()} PDs) — most populated PDs:</td></tr>')
            for r in sub.itertuples():
                cells = ''.join(f'<td>{self._gif_html(self.P["gif"].format(p=r.PD, k=k), width, f"PD {r.PD} · ψ{k}")}</td>'
                                for k in psis)
                html.append(f'<tr><td style="padding-right:8px;font-size:12px">PD <b>{r.PD}</b><br>{r.particles} '
                            f'particles</td>{cells}</tr>')
        display(HTML(''.join(html) + '</table>'))

    def anchor_string(self):
        """The project's anchors in the PD:ψ:sense form used by 6.2."""
        return ', '.join(f"{a + 1}:{k}:{'FWD' if s > 0 else 'REV'}" for a, k, s in self.A['anchors'])

    def parse_anchors(self, text):
        """'keep', 'auto' or 'PD:psi:FWD|REV, ...' (1-based PD and ψ) -> [[pd0, psi, sense], ...] or 'auto'.

        'auto' is resolved by set-anchors, which knows the PDs removed in the same call."""
        text = text.strip()
        if text.lower() == 'keep':
            if len(self.A['anchors']):
                return [[int(a), int(k), 'FWD' if s > 0 else 'REV'] for a, k, s in self.A['anchors']]
            text = 'auto'  # nothing to keep yet
        if text.lower() == 'auto':
            return 'auto'
        out = []
        for item in filter(None, (s.strip() for s in text.replace(';', ',').split(','))):
            parts = [p.strip() for p in item.split(':')]
            if len(parts) == 2:
                parts.append('FWD')
            if len(parts) != 3:
                raise ValueError(f'cannot read anchor {item!r}: use PD:psi:FWD or PD:psi:REV, e.g. 12:1:FWD')
            out.append([int(parts[0]) - 1, int(parts[1]), parts[2].upper()])
        return out

    @staticmethod
    def parse_pd_list(text):
        out = []
        for item in filter(None, (s.strip() for s in text.replace(';', ',').split(','))):
            if '-' in item:
                a, b = item.split('-')
                out += list(range(int(a), int(b) + 1))
            else:
                out.append(int(item))
        return [p - 1 for p in out]

    def show_anchor_map(self, title='PDs, anchors and removed PDs'):
        A, S = self.A, self.S
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
        x, y = self._disk(A['pos_pd'])
        for a, b in A['edges']:
            ax.plot([x[a], x[b]], [y[a], y[b]], color='0.85', lw=0.8, zorder=0)
        comp = self._clusters()
        cmap = plt.get_cmap('tab10')
        size = 40 + 160 * A['occupancy'] / A['occupancy'].max()
        ax.scatter(x, y, c=[cmap(c % 10) if c >= 0 else '0.85' for c in comp], s=size, edgecolors='k', lw=0.4, zorder=1)
        anchored = {comp[int(a[0])] for a in A['anchors']}
        lonely = [i for i in range(S['n_pd']) if comp[i] >= 0 and comp[i] not in anchored] if len(A['anchors']) else []
        if lonely:   # these PDs would get no conformational coordinate
            ax.scatter(x[lonely], y[lonely], s=size[lonely] + 120, facecolors='none', edgecolors='red', lw=1.5, zorder=1)
        for i in range(S['n_pd']):
            ax.text(x[i], y[i], str(i + 1), fontsize=7, ha='center', va='center', zorder=3)
        for pd0, psi, sense in A['anchors']:
            ax.scatter(x[pd0], y[pd0], marker='*', s=420, facecolors='gold', edgecolors='k', zorder=2)
            ax.annotate(f"PD {pd0 + 1}: ψ{psi} {'FWD' if sense > 0 else 'REV'}", (x[pd0], y[pd0]), xytext=(6, 18),
                        textcoords='offset points', fontsize=8, weight='bold', zorder=5,
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='0.6', alpha=0.9),
                        arrowprops=dict(arrowstyle='-', color='0.4', lw=0.6))
        for t in A['trash']:
            ax.scatter(x[t], y[t], marker='x', s=160, color='red', lw=2, zorder=4)
        self._disk_axes(ax, title + '\n(colour = cluster, ★ = anchor, ✕ = removed, size = particles'
                        + (', red ring = cluster without anchor)' if lonely else ')'))
        plt.show()

    # ------------------------------------------------------------------------- 7. CC & landscape
    def show_cc(self):
        A, S, P = self.A, self.S, self.P
        cc = _read_pickle(P['CC_file'])
        psinums, senses = np.asarray(cc['psinums'][0]), np.asarray(cc['senses'][0])
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw=dict(width_ratios=[1, 1.2]))
        x, y = self._disk(A['pos_pd'])
        for a, b in A['edges']:
            ax.plot([x[a], x[b]], [y[a], y[b]], color='0.88', lw=0.8, zorder=0)
        cmap = plt.get_cmap('tab10')
        for i in range(S['n_pd']):
            if psinums[i] < 0:
                ax.scatter(x[i], y[i], marker='x', color='0.5', s=60)
            else:
                ax.scatter(x[i], y[i], marker='^' if senses[i] > 0 else 'v', s=120, color=cmap(psinums[i] % 10),
                           edgecolors='k', lw=0.4)
            ax.text(x[i], y[i] - 0.09, str(i + 1), fontsize=6, ha='center', va='top', color='0.45' if psinums[i] < 0 else 'k',
                    path_effects=[patheffects.withStroke(linewidth=2, foreground='w')])   # readable over a neighbour
        for pd0 in {int(a[0]) for a in A['anchors']}:
            ax.scatter(x[pd0], y[pd0], s=420, facecolors='none', edgecolors='goldenrod', lw=1.8, zorder=2)
        self._disk_axes(ax, 'Chosen eigenvector per PD (colour)\nsense ▲ FWD / ▼ REV, ✕ unassigned, gold ring = anchor')
        k = S['num_psi']
        counts = [(np.sum((psinums == j) & (senses > 0)), np.sum((psinums == j) & (senses < 0))) for j in range(k)]
        ax2.bar(np.arange(1, k + 1), [c[0] for c in counts], color=[cmap(j % 10) for j in range(k)], label='FWD')
        ax2.bar(np.arange(1, k + 1), [c[1] for c in counts], bottom=[c[0] for c in counts],
                color=[cmap(j % 10) for j in range(k)], alpha=0.45, hatch='//', label='REV')
        ax2.set_xticks(np.arange(1, k + 1))
        ax2.set_xticklabels([f'ψ{j}' for j in range(1, k + 1)])
        ax2.set_ylabel('PDs')
        ax2.legend()
        ax2.set_title(f'{np.sum(psinums >= 0)} of {len(psinums)} PDs assigned a conformational coordinate', fontsize=10)
        plt.tight_layout()
        plt.show()
        trash = set(int(t) for t in A['trash'])
        lost = [i + 1 for i in range(len(psinums)) if psinums[i] < 0 and i not in trash]
        if lost:
            print(f"⚠️ no coordinate for PD(s) {', '.join(map(str, lost))}: give their cluster an anchor in 6.2 "
                  '(or remove junk PDs with REMOVE_PDS)')
        return psinums, senses

    def show_landscape(self, gt_rows=None):
        P = self.P
        hUn = np.asarray(_read_pickle(P['traj'])['hUn'], dtype=float)
        p = hUn / hUn.sum()
        with np.errstate(divide='ignore'):
            dG = -np.log(p)
        dG -= np.nanmin(dG[np.isfinite(dG)])
        states = np.arange(1, len(hUn) + 1)
        ncol = 3 if gt_rows else 2
        fig, axes = plt.subplots(1, ncol, figsize=(5.2 * ncol, 3.8))
        axes[0].bar(states, hUn, color='tab:blue', width=0.85)
        if gt_rows:  # the synthetic conformations are sampled uniformly: the true occupancy is flat
            axes[0].axhline(hUn.mean(), color='k', ls='--', lw=1, label='truth (uniform)')
            axes[0].legend(fontsize=8)
        axes[0].set_xlabel('conformational state (τ bin)')
        axes[0].set_ylabel('particles')
        axes[0].set_title(f'Occupancy along the coordinate ({int(hUn.sum()):,} images)', fontsize=10)
        axes[1].plot(states, dG, 'o-', color='crimson', ms=3)
        if gt_rows:  # spread a flat profile would show from counting noise alone (±2 sigma)
            noise = 2 / np.sqrt(max(hUn.mean(), 1))
            ref = -np.log(hUn.mean() / hUn.max())
            axes[1].axhspan(ref - noise, ref + noise, color='0.85', zorder=0, label='flat ± counting noise')
            axes[1].legend(fontsize=8)
        axes[1].set_xlabel('conformational state (τ bin)')
        axes[1].set_ylabel('ΔG = −ln P  (k$_B$T)')
        axes[1].set_title('Free-energy profile', fontsize=10)
        if gt_rows:
            rho = np.array([r['rho'] for r in gt_rows])
            axes[2].bar([r['pd'] for r in gt_rows], rho, color=np.where(rho > 0, 'tab:green', 'tab:orange'))
            axes[2].axhline(0, color='k', lw=0.6)
            axes[2].set_ylim(-1, 1)
            axes[2].set_xlabel('PD')
            axes[2].set_ylabel('Spearman ρ (recovered τ vs truth)')
            axes[2].set_title(f'Synthetic ground truth: median |ρ| = {np.median(np.abs(rho)):.2f}', fontsize=10)
        plt.tight_layout()
        plt.show()
        if gt_rows:
            same = max(np.sum(rho > 0), np.sum(rho < 0))
            display(Markdown(f'{same} of {len(rho)} PDs agree on the direction of the motion. The overall sign is '
                             'arbitrary; a PD with the minority sign got the wrong sense in 7.1.'))
        return states, hUn, dG

    # ---------------------------------------------------------------------------- 8. volumes
    @staticmethod
    def volume_files(folder):
        import re
        return sorted(glob.glob(os.path.join(folder, '*.mrc')),
                      key=lambda f: int((re.findall(r'_(\d+)_of_', f) or ['0'])[-1]))


def read_mrc(path):
    import mrcfile
    with mrcfile.open(path, permissive=True) as m:
        return np.array(m.data, dtype=np.float32, copy=True), float(m.voxel_size.x) or 1.0  # copy: read-only arrays


def needs_flip(files, mode='auto'):
    """ManifoldEM writes the trajectory images with inverted contrast, so volumes often come out negative."""
    if mode in ('yes', 'no'):
        return mode == 'yes'
    vol, _ = read_mrc(files[len(files) // 2])
    return abs(np.percentile(vol, 0.05)) > abs(np.percentile(vol, 99.95))


def volume_projections(files, flip=False):
    """(states, 3, N, N) projections along z, y and x, reading one volume at a time."""
    out = []
    for f in files:
        v, _ = read_mrc(f)
        v = -v if flip else v
        out.append(np.stack([v.sum(axis=a) for a in (0, 1, 2)]))
    return np.stack(out)


def show_volume_strip(proj, n=8, title=''):
    idx = np.unique(np.linspace(0, len(proj) - 1, min(n, len(proj))).round().astype(int))
    fig, axes = plt.subplots(4, len(idx), figsize=(1.8 * len(idx), 7.4))
    axes = np.atleast_2d(axes).reshape(4, len(idx))
    moving = int(np.argmax([proj[:, r].std(axis=0).sum() for r in range(3)]))  # view in which the motion is largest
    diff = proj[idx, moving] - proj[idx[0], moving]
    lim = np.percentile(np.abs(diff), 99.8) or 1.0
    for col in range(len(idx)):
        axes[3, col].imshow(diff[col], cmap='RdBu_r', vmin=-lim, vmax=lim)
        axes[3, col].axis('off')
    axes[3, 0].text(-0.15, 0.5, f"change vs state {idx[0] + 1}\n({('z', 'y', 'x')[moving]})",
                    transform=axes[3, 0].transAxes, rotation=90, va='center', ha='right', fontsize=8)
    for row, name in enumerate(('projection along z', 'along y', 'along x')):
        lo, hi = np.percentile(proj[idx, row], [1, 99.8])
        for col, j in enumerate(idx):
            axes[row, col].imshow(proj[j, row], cmap='gray', vmin=lo, vmax=hi)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(f'state {j + 1}', fontsize=8)
        axes[row, 0].text(-0.15, 0.5, name, transform=axes[row, 0].transAxes, rotation=90, va='center',
                          ha='right', fontsize=8)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


def volume_movie(proj, out_gif, fps=8, size=128, max_states=24):
    """Looping GIF of the three projections, played forward and back along the conformational coordinate."""
    from PIL import Image as PILImage
    limits = [np.percentile(proj[:, r], [1, 99.8]) for r in range(3)]
    frames = []
    keep = np.unique(np.linspace(0, len(proj) - 1, min(max_states, len(proj))).round().astype(int)).tolist()
    for j in keep + keep[-2:0:-1]:
        canvas = PILImage.new('L', (3 * size + 8, size), 255)
        for r, (lo, hi) in enumerate(limits):
            t = np.clip((proj[j, r] - lo) / (hi - lo + 1e-12), 0, 1)
            canvas.paste(PILImage.fromarray((255 * t).astype(np.uint8)).resize((size, size), PILImage.LANCZOS),
                         (r * (size + 4), 0))
        frames.append(canvas)
    frames[0].save(out_gif, save_all=True, append_images=frames[1:], loop=0, duration=int(1000 / fps))
    b64 = base64.b64encode(open(out_gif, 'rb').read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{b64}"><br><small>{out_gif} — projections along z, y and x, '
                 'played along the conformational coordinate and back</small>'))


def isosurface_movie(files, states, flip=False, sigma=3.0, step=2):
    """Plotly isosurfaces (marching cubes) with a slider over states; renders with plain fig.show()."""
    import plotly.graph_objects as go
    from skimage.measure import marching_cubes
    ref, apix = read_mrc(files[len(files) // 2])
    ref = -ref if flip else ref
    level = float(ref.mean() + sigma * ref.std())
    meshes = []
    for j in states:
        v, _ = read_mrc(files[j])
        v = -v if flip else v
        try:
            verts, faces, _, _ = marching_cubes(v, level=level, step_size=step, spacing=(apix,) * 3)
        except (ValueError, RuntimeError):  # no surface at this level
            verts, faces = np.zeros((3, 3)), np.array([[0, 1, 2]])
        z, y, x = verts.T
        meshes.append(go.Mesh3d(x=x, y=y, z=z, i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], color='#9ab',
                                lighting=dict(ambient=0.45, diffuse=0.8, specular=0.25)))
    size = ref.shape[-1] * apix
    axis = dict(range=[0, size], visible=False)
    fig = go.Figure(data=[meshes[0]], frames=[go.Frame(data=[m], name=str(j + 1)) for m, j in zip(meshes, states)])
    fig.update_layout(
        height=600, margin=dict(l=0, r=0, t=40, b=0),
        title=f'Isosurface at mean + {sigma:g}σ — drag the slider (or press play) to move along the coordinate',
        scene=dict(aspectmode='cube', xaxis=axis, yaxis=axis, zaxis=axis),
        sliders=[dict(currentvalue=dict(prefix='state '), pad=dict(t=10), steps=[
            dict(method='animate', label=str(j + 1), args=[[str(j + 1)], dict(mode='immediate', frame=dict(duration=0))])
            for j in states])],
        updatemenus=[dict(type='buttons', x=0.05, y=0.05, buttons=[dict(label='▶ play', method='animate', args=[
            None, dict(frame=dict(duration=300, redraw=True), fromcurrent=True, mode='immediate')])])])
    fig.show()


def write_chimerax_script(files, folder):
    """open_in_chimerax.cxc: opens the volumes as a series in numeric state order (a *.mrc wildcard sorts 1, 10, 11, ...)."""
    cxc = os.path.join(folder, 'open_in_chimerax.cxc')
    with open(cxc, 'w') as fh:
        fh.write('open ' + ' '.join(os.path.relpath(f, folder) for f in files) + ' vseries true\n')
        fh.write('volume #1 style surface\n')
    return cxc


def denoise_volumes(files, out_dir, sigma=1.0, svd_rank=0):
    """Gaussian-filter each volume and/or keep only the leading SVD components across the trajectory."""
    import mrcfile
    from scipy import ndimage
    os.makedirs(out_dir, exist_ok=True)
    X = proj = mean = None
    if svd_rank:  # one float32 stack; the SVD comes from the small (states x states) Gram matrix
        first, _ = read_mrc(files[0])
        need_gb = len(files) * first.size * 4 / 1e9
        try:
            import psutil
            if need_gb > 0.7 * psutil.virtual_memory().available / 1e9:
                raise MemoryError(f'SVD denoising needs ~{need_gb:.1f} GB of RAM for {len(files)} volumes of '
                                  f'{first.shape[0]}³; use SVD_COMPONENTS = 0 or downsample')
        except ImportError:
            pass
        X = np.empty((len(files), first.size), np.float32)
        for i, f in enumerate(files):
            X[i] = read_mrc(f)[0].ravel()
        mean = X.mean(0)
        X -= mean
        w, U = np.linalg.eigh((X @ X.T).astype(np.float64))
        w, U = np.clip(w[::-1], 0, None), U[:, ::-1]
        k = min(int(svd_rank), len(w))
        proj = (U[:, :k] @ U[:, :k].T).astype(np.float32)  # projector onto the leading k components
        print(f'kept {k} SVD components ({100 * w[:k].sum() / max(w.sum(), 1e-30):.1f}% of the variance)')
    paths = []
    for i, f in enumerate(files):
        v, apix = read_mrc(f)
        if proj is not None:
            v = (mean + proj[i] @ X).reshape(v.shape)
        if sigma:
            v = ndimage.gaussian_filter(v, sigma).astype(np.float32)
        p = os.path.join(out_dir, os.path.basename(f))
        with mrcfile.new(p, overwrite=True) as m:
            m.set_data(v.astype(np.float32))
            m.voxel_size = apix  # keep the pixel size (ManifoldEM's own `utility denoise` drops it)
        paths.append(p)
    return paths


def register_manifoldem_magic(session):
    """%%manifoldem: run a cell inside the ManifoldEM environment with params/data_store/prds preloaded."""
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def manifoldem(line, cell):
        fig_dir = tempfile.mkdtemp(dir=session.tmp)
        if not session.project:
            print('ℹ️  no project is open (2.1 / 2.2): params and prds are not loaded')
        try:
            res = session.bridge('pyexec', toml=session.toml if session.project else '', code=cell, fig_dir=fig_dir)
            for f in res.get('figures', []):
                display(Image(filename=f))
        finally:
            shutil.rmtree(fig_dir, ignore_errors=True)
            if session.project:
                session.refresh()  # the cell may have saved changes (prds.save(), params.save())


mem = Session(ROOT, ENV_DIR, BRIDGE_PATH)
register_manifoldem_magic(mem)
_dataset_file = mem.data / 'current_dataset.json'
DATA = json.loads(_dataset_file.read_text()) if _dataset_file.exists() else None
print(f'✔ helpers loaded · {mem.ncpu} CPUs will be used')
if mem.projects():
    print(f'existing projects: {", ".join(mem.projects())} — use 2.2 to re-open one')

## 1 · Data

ManifoldEM starts from a **3D refinement** of your particles (RELION, cryoSPARC → RELION STAR, ...). It needs:

* a **STAR file** with, for every particle, `rlnAngleRot`, `rlnAngleTilt`, `rlnAnglePsi`, `rlnDefocusU`,
  `rlnDefocusV` and (optionally) origins `rlnOriginX/Y[Angst]`, plus `rlnVoltage`, `rlnSphericalAberration` and
  `rlnAmplitudeContrast` (per particle or in a `data_optics` block);
* **one** `.mrcs` particle stack whose *i*-th image is the *i*-th STAR row (combine several stacks with
  `relion_stack_create`). Ideally re-extract the particles centred, with origins set to 0;
* optionally a **3D mask** (`.mrc`, same box and pixel size as the images). ManifoldEM projects it along each PD and
  uses that footprint instead of its default circular mask, both for the image distances and for the movies. A mask
  around the whole particle removes solvent noise; a mask around one moving region focuses the analysis on it;
* optionally an **average volume**, which is stored in the project for the desktop GUI; this notebook does not need it.

Choose a dataset in **1.1**, then run the matching cell (1.2, 1.3 or 1.4). The other two will skip themselves.

In [ ]:
#@title 1.1 · Choose a dataset { display-mode: "form" }
DATASET = "Synthetic demo (generated here, ~1 min)"  #@param ["Synthetic demo (generated here, ~1 min)", "RyR1 tutorial data (download, several GB)", "My own data (Google Drive, URL or upload)"]
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#@markdown Tick **USE_GOOGLE_DRIVE** to mount your Drive at `/content/drive` if your data is there. (The checkpoint
#@markdown cells 10.1 and 2.2 mount Drive by themselves when you use them.)
DATASET_KIND = {'Synthetic': 'synthetic', 'RyR1 tuto': 'ryr1', 'My own da': 'own'}[DATASET[:9]]
if USE_GOOGLE_DRIVE:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Google Drive can only be mounted inside Colab; skipping.')
print(f"Dataset: {DATASET} → now run cell {dict(synthetic='1.2', ryr1='1.3', own='1.4')[DATASET_KIND]}.")

In [ ]:
#@title 1.2 · Synthetic demo dataset { display-mode: "form" }
#@markdown A small dataset with a **known** motion: the RyR1 map that ships with ManifoldEM, whose four cytoplasmic
#@markdown arms move up and down by up to `MOTION_AMPLITUDE_A`. The 20 conformations along the motion are equally
#@markdown likely, so the true free-energy profile is **flat**. The particles are side views on one great circle,
#@markdown 64 px at 5.25 Å/px, with CTF and Gaussian noise. Section 7 compares ManifoldEM's answer with the truth.
N_PARTICLES = 6000  #@param {type:"slider", min:3000, max:20000, step:1000}
SNR = 0.1  #@param {type:"number"}
MOTION_AMPLITUDE_A = 25  #@param {type:"number"}
SEED = 0  #@param {type:"integer"}
import urllib.request
if DATASET_KIND != 'synthetic':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    src = mem.data / 'RyR1_AvgVol_Binned.mrc'
    if not src.exists():
        base = MANIFOLDEM_REPO.strip().rstrip('/')
        urls = [base.replace('github.com', 'raw.githubusercontent.com') + f'/{MANIFOLDEM_VERSION.strip() or "main"}/demo/RyR1_AvgVol_Binned.mrc',
                'https://raw.githubusercontent.com/flatironinstitute/ManifoldEM/main/demo/RyR1_AvgVol_Binned.mrc']
        for url in urls:
            try:
                urllib.request.urlretrieve(url, str(src) + '.part')
                os.replace(str(src) + '.part', src)
                break
            except Exception as exc:
                print(f'could not download {url}: {exc}')
        else:
            raise RuntimeError('could not download the RyR1 map that the synthetic data is built from')
    # one folder (and one project) per parameter set, so existing projects never see their inputs change
    tag = f'n{int(N_PARTICLES)}_snr{float(SNR):g}_amp{float(MOTION_AMPLITUDE_A):g}_seed{int(SEED)}'
    default_tag = 'n6000_snr0.1_amp25_seed0'
    out = mem.data / 'synthetic' / tag
    complete = out / '.complete.json'
    if complete.exists() and all((out / n).exists() for n in ('particles.star', 'particles.mrcs', 'avg_volume.mrc', 'ground_truth.csv')):
        info = json.loads(complete.read_text())
        print(f'· reusing the dataset generated earlier with these settings: {out}')
    else:
        info = mem.bridge('synthetic', src=str(src), out=str(out), n_particles=int(N_PARTICLES), box=64,
                          amp_angst=float(MOTION_AMPLITUDE_A), snr=float(SNR), seed=int(SEED))
        complete.write_text(json.dumps(info))   # written last: an interrupted run is redone
    px = info['pixel_size']
    DATA = dict(kind='synthetic', star=str(out / 'particles.star'), stack=str(out / 'particles.mrcs'),
                volume=str(out / 'avg_volume.mrc'), mask='', ground_truth=str(out / 'ground_truth.csv'),
                project='synthetic_demo' if tag == default_tag else f'synthetic_{tag}',
                suggest=dict(pixel_size=px, diameter=300.0, resolution=round(2 * px, 3), aperture_index=4, bp_num_psi=1))
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f"✔ {info['n_particles']:,} particles ({info['box']} px, {px:.3g} Å/px) → {out}")

In [ ]:
#@title 1.3 · RyR1 tutorial dataset { display-mode: "form" }
#@markdown The RyR1 "great circles" dataset from the [ManifoldEM tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md)
#@markdown and Ojha *et al.* (2025): 14,491 particles, 1.255 Å/px, diameter 360 Å, resolution 5 Å, aperture index 4.
#@markdown The download is several GB; downsampling (1.5) is recommended on the free tier.
RYR1_URL = "https://users.flatironinstitute.org/~rblackwell/manifold/RyR1GCs_demo.tar.gz"  #@param {type:"string"}
KEEP_ARCHIVE = False  #@param {type:"boolean"}
import tarfile
if DATASET_KIND != 'ryr1':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    dest = mem.data / 'ryr1'
    dest.mkdir(parents=True, exist_ok=True)
    archive = dest / RYR1_URL.rstrip('/').split('/')[-1]
    if not (dest / '.extracted').exists():
        downloaded = dest / '.download_complete'
        if not downloaded.exists():          # wget -c resumes an interrupted download
            mem.run(['wget', '-c', '--progress=dot:giga', '-O', archive, RYR1_URL])
            downloaded.touch()
        print('· extracting ...')
        with tarfile.open(archive) as tf:
            tf.extractall(dest, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
        (dest / '.extracted').touch()
        if not KEEP_ARCHIVE:
            archive.unlink()
    stars = sorted(dest.rglob('*.star'))
    stacks = sorted(dest.rglob('*.mrcs'))
    if not stars or not stacks:
        raise RuntimeError(f'no .star / .mrcs found in {dest}')
    star = next((s for s in stars if 'clustRem' in s.name), stars[0])
    stack = next((s for s in stacks if s.stem == star.stem), stacks[0])
    DATA = dict(kind='ryr1', star=str(star), stack=str(stack), volume='', mask='', ground_truth='',
                project='ryr1_tutorial',
                suggest=dict(pixel_size=1.255, diameter=360.0, resolution=5.0, aperture_index=4, downsample=2))
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f'✔ STAR: {star}\n  stack: {stack} ({stack.stat().st_size / 1e9:.1f} GB)')

In [ ]:
#@title 1.4 · Your own data { display-mode: "form" }
#@markdown Paths on Google Drive (mount it in 1.1, e.g. `/content/drive/MyDrive/cryoem/particles.star`), paths on the
#@markdown Colab disk, or `https://` links. Leave the volume and mask empty if you have none.
STAR_FILE = ""  #@param {type:"string"}
PARTICLE_STACK = ""  #@param {type:"string"}
AVERAGE_VOLUME = ""  #@param {type:"string"}
MASK_VOLUME = ""  #@param {type:"string"}
COPY_TO_COLAB_DISK = True  #@param {type:"boolean"}
#@markdown Copying from Drive to the Colab disk makes every step faster and protects the originals (ManifoldEM may
#@markdown rewrite the header of the particle stack). Each version of a file gets its own folder, so projects made
#@markdown from earlier versions are never affected.
UPLOAD_FROM_THIS_COMPUTER = False  #@param {type:"boolean"}
#@markdown Upload files with the browser instead: fine for the STAR file or a mask, slow for stacks of many GB (use Drive).
#@markdown Empty fields above are filled with the uploaded `.star` / `.mrcs` files.
import hashlib
if DATASET_KIND != 'own':
    print(f'skipped (1.1 selected: {DATASET})')
else:
    dest = mem.data / 'own'
    dest.mkdir(parents=True, exist_ok=True)
    uploaded = {}
    if UPLOAD_FROM_THIS_COMPUTER:
        if not IN_COLAB:
            raise RuntimeError('uploading only works inside Colab')
        from google.colab import files
        incoming = Path(tempfile.mkdtemp(dir=dest, prefix='.upload-'))
        names = [Path(p).name for p in files.upload(target_dir=str(incoming))]   # Colab writes the files there itself
        # the folder is named after the files' content: uploading the same files again (e.g. after a disconnect) gives
        # the same paths, so an existing project finds its inputs again
        h = hashlib.sha1()
        for n in sorted(names):
            with open(incoming / n, 'rb') as fh:
                h.update(f'{n}|'.encode() + hashlib.file_digest(fh, 'sha1').digest())
        up = dest / f'upload-{h.hexdigest()[:10]}'
        if up.exists():
            shutil.rmtree(incoming)
            print(f'· the same files were uploaded before: reusing {up}')
        else:
            incoming.rename(up)
        for n in names:
            uploaded[n] = up / n
            (up / f'.{n}.version').write_text(up.name)   # the version 1.5 uses (the folder is named after the content)
            print(f'uploaded {n} ({uploaded[n].stat().st_size / 1e6:.1f} MB)')
        if not STAR_FILE.strip():
            STAR_FILE = next((str(p) for n, p in uploaded.items() if n.endswith('.star')), '')
        if not PARTICLE_STACK.strip():   # prefer .mrcs, then the largest file (a .mrc may be the mask)
            stacks = sorted((p for n, p in uploaded.items() if n.endswith(('.mrcs', '.mrc'))),
                            key=lambda p: (p.suffix != '.mrcs', -p.stat().st_size))
            PARTICLE_STACK = str(stacks[0]) if stacks else ''
            if stacks:
                print(f'using {stacks[0].name} as the particle stack')

    def fetch(path, role):
        """Local copy of a path or URL, in a folder unique to that exact file version."""
        path = path.strip()
        if not path:
            return ''
        if path.startswith(('http://', 'https://')):
            name = path.split('?')[0].rstrip('/').split('/')[-1]
            if not name.endswith(('.star', '.mrcs', '.mrc')):    # e.g. .../download or .../view links
                name = role + ('.star' if role == 'star' else '.mrcs' if role == 'stack' else '.mrc')
            target = dest / hashlib.sha1(path.encode()).hexdigest()[:10] / name
            if target.exists():
                print(f'· reusing {target} (downloaded earlier; delete that folder to download again)')
                return str(target)
            target.parent.mkdir(exist_ok=True)
            part = target.with_name(name + '.part')
            mem.run(['wget', '-c', '--progress=dot:giga', '-O', part, path])   # -c resumes a partial download
            with open(part, 'rb') as fh:
                head = fh.read(512).lstrip().lower()
            if head.startswith((b'<!doctype', b'<html')):
                part.unlink()
                raise ValueError(f'{path} returned a web page, not the file (e.g. a Drive "view" link or a login '
                                 'page): use a direct download link, or put the file on Drive and give its path')
            with open(part, 'rb') as fh:   # the version 1.5 uses (servers do not always send a date)
                target.with_name(f'.{name}.version').write_text(hashlib.file_digest(fh, 'sha1').hexdigest())
            os.replace(part, target)
            return str(target)
        p = Path(path).expanduser()
        if not p.exists() and p.name in uploaded:
            p = uploaded[p.name]                      # an uploaded file given by name
        if not p.exists():
            raise FileNotFoundError(f'{path} not found' + (' (is Google Drive mounted? see 1.1)' if '/drive/' in path else ''))
        if p.is_dir():
            inside = sorted(x.name for x in p.iterdir() if x.suffix in ('.star', '.mrcs', '.mrc'))
            raise ValueError(f'{path} is a folder: give the file inside it' + (f" ({', '.join(inside)})" if inside else ''))
        if not COPY_TO_COLAB_DISK or mem.data.resolve() in p.resolve().parents:
            return str(p.resolve())
        st = p.stat()
        target = dest / hashlib.sha1(f'{p.resolve()}|{st.st_size}|{st.st_mtime_ns}'.encode()).hexdigest()[:10] / p.name
        if target.exists():
            print(f'· reusing {target}')
        else:
            target.parent.mkdir(exist_ok=True)
            print(f'· copying {p.name} ({st.st_size / 1e9:.2f} GB) ...')
            shutil.copy2(p, str(target) + '.part')
            os.replace(str(target) + '.part', target)
        # the version 1.5 uses: the folder is named after the source's path, size and time (ManifoldEM may rewrite the
        # copy's header, which changes its own modification time but not the images)
        target.with_name(f'.{p.name}.version').write_text(target.parent.name)
        return str(target)

    if not STAR_FILE.strip() or not PARTICLE_STACK.strip():
        raise ValueError('please fill in STAR_FILE and PARTICLE_STACK (or upload a .star and a .mrcs file)')
    DATA = dict(kind='own', star=fetch(STAR_FILE, 'star'), stack=fetch(PARTICLE_STACK, 'stack'),
                volume=fetch(AVERAGE_VOLUME, 'volume'), mask=fetch(MASK_VOLUME, 'mask'), ground_truth='',
                project='my_project', suggest={})
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print('✔ ' + '\n  '.join(f'{k}: {DATA[k]}' for k in ('star', 'stack', 'volume', 'mask') if DATA[k]))

In [ ]:
#@title 1.5 · (Optional) Downsample the particles { display-mode: "form" }
#@markdown Fourier-crops every image (and the mask and average volume) by `DOWNSAMPLE_BY`, e.g. 336 → 168 px. Every
#@markdown ManifoldEM step becomes roughly 4–8× faster and needs 4× less memory; information beyond the new Nyquist
#@markdown limit (2 × new pixel size) is lost. `0` = automatic (2 for the RyR1 tutorial data, otherwise keep the
#@markdown original size). The smaller copy is always written to the Colab disk. An existing project keeps using its
#@markdown own images: create a new one in 2.1.
DOWNSAMPLE_BY = 0  #@param {type:"number"}
import hashlib
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
original = DATA.get('original') or {k: DATA[k] for k in ('star', 'stack', 'volume', 'mask')}
factor = DOWNSAMPLE_BY or DATA['suggest'].get('downsample', 1)
if factor <= 1:
    if DATA.get('downsampled_by'):
        DATA.update(original, downsampled_by=None)   # back to the original images
        DATA['suggest'] = DATA.get('suggest_original', DATA['suggest'])
        _dataset_file.write_text(json.dumps(DATA, indent=1))
        print('✔ using the original images again')
    else:
        print('keeping the original images')
elif DATA.get('downsampled_by') == factor:
    print(f'already downsampled by {factor:g}: {DATA["stack"]}')
else:
    # One folder per (input versions, factor): projects made from other inputs never see their files change, and the
    # same inputs re-created after a disconnect find their downsampled images again. 1.4 records the version of what
    # it copied, downloaded or received; the synthetic and RyR1 folders hold one fixed dataset each; for other files
    # (used in place) the modification time identifies the version.
    def _version(p):
        p, st = Path(p).resolve(), Path(p).stat()
        tag = p.with_name(f'.{p.name}.version')
        if tag.exists():
            return f'{p}|{st.st_size}|{tag.read_text()}'
        fixed = any((mem.data / d).resolve() in p.parents for d in ('synthetic', 'ryr1'))
        return f'{p}|{st.st_size}' + ('' if fixed else f'|{st.st_mtime_ns}')
    key = '|'.join(_version(p) for p in (original['star'], original['stack'], original['volume'], original['mask']) if p)
    out_dir = mem.data / DATA['kind'] / 'downsampled' / (hashlib.sha1(f'{key}|{factor:g}'.encode()).hexdigest()[:10])
    res = mem.bridge('downsample', star=original['star'], stack=original['stack'], factor=float(factor),
                     volume=original['volume'], mask=original['mask'], out_dir=str(out_dir))
    DATA.setdefault('suggest_original', dict(DATA['suggest']))
    DATA.update(star=res['star'], stack=res['stack'], volume=res['volume'], mask=res['mask'],
                downsampled_by=factor, original=original)
    sug = DATA['suggest'] = dict(DATA['suggest_original'])
    if res.get('pixel_size'):
        sug['pixel_size'] = res['pixel_size']
        nyquist = round(2 * res['pixel_size'], 3)
        # the resolution only sizes the orientation bins: correct a suggested value only if it is now clearly below Nyquist
        if sug.get('resolution') and sug['resolution'] < 0.95 * nyquist:
            sug['resolution'] = nyquist
    _dataset_file.write_text(json.dumps(DATA, indent=1))
    print(f"✔ {res['box']} px images{' (reused)' if res.get('reused') else ''} → {res['stack']}")
    print(f"  new pixel size: {res['pixel_size']:g} Å" if res.get('pixel_size') else
          f"  the pixel size is now {res['scale']:.3g} × the original: enter it in 2.1")

In [ ]:
#@title 1.6 · Check the inputs { display-mode: "form" }
#@markdown Reads the STAR file and the stack exactly as ManifoldEM will, and shows a few particles and the
#@markdown distribution of viewing directions and defocus.
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
INPUT_REPORT = mem.bridge('inspect-inputs', star=DATA['star'], stack=DATA['stack'], mask=DATA.get('mask', ''),
                          out_npz=str(mem.tmp / 'inputs.npz'))
mem.show_inputs(INPUT_REPORT, mem.tmp / 'inputs.npz')
DATA['checked'] = dict(star=DATA['star'], stack=DATA['stack'], mask=DATA.get('mask', ''), problems=INPUT_REPORT['problems'])
star_px, header_px = INPUT_REPORT.get('star_pixel_size'), INPUT_REPORT.get('header_pixel_size') or 0
if star_px and header_px and abs(star_px - header_px) > 0.01 * header_px:
    print('⚠️ the two pixel sizes disagree: enter the right one as PIXEL_SIZE_A in 2.1')
elif star_px or header_px:
    DATA['suggest'].setdefault('pixel_size', star_px or header_px)
_ = _dataset_file.write_text(json.dumps(DATA, indent=1))

## 2 · Project

A project is a `params_<name>.toml` file plus an `output/<name>/` folder, both in `/content/manifoldem/work`.

* **Pixel size** in Å (of the images you use, i.e. after downsampling).
* **Object diameter** in Å: the largest dimension of the particle.
* **Resolution** in Å: the resolution of your refinement. It cannot be better (smaller) than the Nyquist limit,
  2 × the pixel size. ManifoldEM uses it only to size the orientation bins, so changing it changes the PDs.
* **Aperture index** (1–4, sometimes more): scales the size of the orientation bins. The bin width is
  aperture index × Shannon angle, where Shannon angle = resolution / diameter. Larger bins collect more particles per
  PD but blur the motion with nearby views. It usually takes some trial and error (section 3 shows the effect in seconds).

In [ ]:
#@title 2.1 · Create the project { display-mode: "form" }
PROJECT_NAME = "auto"  #@param {type:"string"}
PIXEL_SIZE_A = 0  #@param {type:"number"}
OBJECT_DIAMETER_A = 0  #@param {type:"number"}
RESOLUTION_A = 0  #@param {type:"number"}
APERTURE_INDEX = 0  #@param {type:"integer"}
#@markdown `auto` / `0` = the value suggested for the selected dataset (or read from the STAR file / stack header).
USE_MASK = True  #@param {type:"boolean"}
OVERWRITE_EXISTING = False  #@param {type:"boolean"}
#@markdown A project that already exists with the same inputs, pixel size, diameter and resolution is simply re-opened
#@markdown (it keeps its aperture index: change that in 3.1). Tick only to start a project of the same name again from
#@markdown scratch: its results are deleted.
import re
assert DATA, 'choose and prepare a dataset first (1.1–1.4)'
sug = DATA.get('suggest', {})
def _pick(value, key, what):
    value = value or sug.get(key)
    if not value:
        raise ValueError(f'please enter {what} (no suggestion available for this dataset)')
    return value
name = DATA['project'] if PROJECT_NAME.strip().lower() in ('', 'auto') else PROJECT_NAME.strip()
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', name) or 'params_' in name:
    raise ValueError('use letters, digits, "_", "-" or "." for the project name (and not "params_")')
px = float(_pick(PIXEL_SIZE_A, 'pixel_size', 'PIXEL_SIZE_A'))
diameter = float(_pick(OBJECT_DIAMETER_A, 'diameter', 'OBJECT_DIAMETER_A'))
resolution = float(_pick(RESOLUTION_A, 'resolution', 'RESOLUTION_A'))
aperture = int(APERTURE_INDEX or sug.get('aperture_index', 1))
mask = DATA.get('mask', '') if USE_MASK else ''
if (mem.work / f'params_{name}.toml').exists() and not OVERWRITE_EXISTING:
    previous = mem.project
    S = mem.use(name)
    # the aperture is not compared: 3.1 may have changed it on purpose
    same = ([S['files'][k] or '' for k in ('star', 'stack', 'mask')] == [DATA['star'], DATA['stack'], mask or '']
            and all(abs(S[k] - v) <= 1e-6 * v for k, v in (('pixel_size', px), ('diameter', diameter), ('resolution', resolution))))
    if not same:
        mem.project = previous
        if previous:
            mem.refresh()
        raise RuntimeError(f'project {name!r} already exists with other inputs or settings: re-open it with 2.2 (then '
                           'Runtime ▸ Run after to continue), pick another name, or tick OVERWRITE_EXISTING')
    print(f'✔ project {name} already exists with these inputs, pixel size, diameter and resolution: re-opened it '
          '(tick OVERWRITE_EXISTING to start again from scratch)')
    if APERTURE_INDEX and int(APERTURE_INDEX) != S['aperture_index']:
        print(f"⚠️ APERTURE_INDEX {int(APERTURE_INDEX)} was not applied: the project uses aperture index "
              f"{S['aperture_index']}. Change it in 3.1 (APERTURE_INDEX_OVERRIDE, re-bins) or tick OVERWRITE_EXISTING")
    mem.status()
else:
    checked = DATA.get('checked', {})
    if [checked.get(k) for k in ('star', 'stack', 'mask')] != [DATA['star'], DATA['stack'], DATA.get('mask', '')]:
        checked = dict(problems=mem.bridge('inspect-inputs', star=DATA['star'], stack=DATA['stack'],
                                           mask=DATA.get('mask', ''))['problems'])
    if checked['problems']:
        raise ValueError('ManifoldEM cannot use these inputs (see 1.6): ' + '; '.join(checked['problems']))
    cmd = [mem.cli_exe, 'init', '-o', '-p', name, '-a', DATA['star'], '-i', DATA['stack'],
           '-s', px, '-d', diameter, '-r', resolution, '-x', aperture]
    if DATA.get('volume'):
        cmd += ['-v', DATA['volume']]
    if mask:
        cmd += ['-m', mask]
    existed = (mem.work / f'params_{name}.toml').exists()
    mem.run(cmd)
    for left in (mem.work / 'output' / f'{name}.replaced', mem.work / '.notebook' / f'{name}.json.replaced',
                 mem.work / f'params_{name}.toml.replaced', mem.work / '.notebook' / f'{name}.restoring'):
        shutil.rmtree(left, ignore_errors=True) if left.is_dir() else left.unlink(missing_ok=True)   # no restore to finish
    mem.project = name
    try:
        mem.state_file.unlink(missing_ok=True)   # a new project: forget the notebook's records of an earlier one
        mem.save_state(dataset=DATA)
        mem.refresh()
        mem.just_created = (name, os.path.getmtime(mem.work / f'params_{name}.toml'))   # for 2.2 (this session only)
    except Exception:
        if not existed:   # do not leave a half-made project behind that blocks the next attempt
            (mem.work / f'params_{name}.toml').unlink(missing_ok=True)
            shutil.rmtree(mem.work / 'output' / name, ignore_errors=True)
            mem.state_file.unlink(missing_ok=True)
        mem.project = None
        raise
    print(f"✔ project {name}: {px:g} Å/px, diameter {diameter:g} Å, resolution {resolution:g} Å, aperture {aperture} "
          f"→ Shannon angle {resolution / diameter:.4f} rad, bin width {aperture * resolution / diameter:.4f} rad")
    print(f"  {mem.S['n_pd']} PDs with the default thresholds — tune them in section 3")

In [ ]:
#@title 2.2 · Re-open a project (after a restart or disconnect) { display-mode: "form" }
#@markdown Not needed right after 2.1 (in Run all this cell keeps the project that is open).
PROJECT_TO_OPEN = ""  #@param {type:"string"}
#@markdown Empty = the open project, else the most recently modified one (or, with RESTORE_FROM_DRIVE, the newest
#@markdown checkpoint).
RESTORE_FROM_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/ManifoldEM"  #@param {type:"string"}
CHECKPOINT = "newest"  #@param {type:"string"}
#@markdown With **RESTORE_FROM_DRIVE**, a checkpoint saved by 10.1 is unpacked first (`newest`, or a file name such
#@markdown as `synthetic_demo_20261028-120000.tar.gz`). Drive is mounted if needed.
OVERWRITE_LOCAL_COPY = False  #@param {type:"boolean"}
#@markdown Only replace a project that already exists on the Colab disk if this is ticked (an empty project that 2.1
#@markdown has just made, e.g. in Run all after a disconnect, is replaced anyway).
import errno, re, tarfile
want = PROJECT_TO_OPEN.strip()

def _swap_in(staging, proj):
    """Move an unpacked project into place. Every part is swapped before any old copy is deleted; the marker file
    (removed last) makes the project refuse to open until a re-run of the restore has finished the job."""
    parts = [mem.root / rel for rel in (f'work/output/{proj}', f'work/.notebook/{proj}.json', f'work/params_{proj}.toml')]
    olds = [dst.with_name(dst.name + '.replaced') for dst in parts]
    for dst, old in zip(parts, olds):                 # the .toml last
        new = staging / dst.relative_to(mem.root)
        if not new.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        if old.exists():                              # an interrupted restore: `old` is the local copy, `dst` is not
            shutil.rmtree(dst, ignore_errors=True) if dst.is_dir() else dst.unlink(missing_ok=True)
        elif dst.exists():
            dst.rename(old)
        shutil.move(str(new), dst)
    for old in olds:
        shutil.rmtree(old, ignore_errors=True) if old.is_dir() else old.unlink(missing_ok=True)

if RESTORE_FROM_DRIVE:
    if IN_COLAB and not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    stamp = re.compile(r'(.+)_(\d{8}-\d{6})\.tar\.gz')
    found = [(m.group(2), m.group(1), p) for p in Path(DRIVE_FOLDER).glob('*.tar.gz')
             if (m := stamp.fullmatch(p.name)) and (not want or m.group(1) == want)]
    if CHECKPOINT.strip() not in ('', 'newest'):
        found = [f for f in found if f[2].name == CHECKPOINT.strip()]
        if not found:
            raise FileNotFoundError(f'no checkpoint called {CHECKPOINT.strip()} in {DRIVE_FOLDER}' + (f' for project {want}' if want else ''))
    if not found:
        raise FileNotFoundError(f"no checkpoints{' of project ' + want if want else ''} in {DRIVE_FOLDER} (10.1 saves them there)")
    for _, proj, archive in sorted(found, reverse=True):          # newest first; skip damaged archives
        marker = mem.work / '.notebook' / f'{proj}.restoring'
        if (mem.work / f'params_{proj}.toml').exists() and not OVERWRITE_LOCAL_COPY and not marker.exists():
            if (glob.glob(str(mem.work / 'output' / proj / 'distances' / '*.h5')) or getattr(mem, 'just_created', None)
                    != (proj, os.path.getmtime(mem.work / f'params_{proj}.toml'))):   # 3.1 / 4.0 change the .toml's time
                print(f'project {proj} already exists on the Colab disk: kept it (tick OVERWRITE_LOCAL_COPY to replace it with {archive.name})')
                want = proj
                break
            print(f'· project {proj} was just made by 2.1 and has no results yet: replacing it')
        staging = Path(tempfile.mkdtemp(dir=mem.tmp))
        need = 0
        try:
            try:
                print(f'· checking and unpacking {archive.name} ...')
                mem.run(['gzip', '-t', archive], echo=False)            # full CRC check before touching anything
                with tarfile.open(archive) as tf:
                    members = [m for m in tf.getmembers() if m.name.startswith((f'work/params_{proj}.toml', f'work/output/{proj}/',
                                                                                f'work/.notebook/{proj}.json', 'data/'))]
                    need = sum(m.size for m in members)
                    tf.extractall(staging, members=members, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
            except (RuntimeError, tarfile.TarError, EOFError, OSError) as exc:
                if getattr(exc, 'errno', None) in (errno.ENOSPC, errno.EDQUOT):   # the disk is full, not the archive bad
                    raise RuntimeError(f'not enough free space on the Colab disk to unpack {archive.name}'
                                       + (f' (it needs about {_human(need)} free)' if need else '') + ': delete files you '
                                       f'no longer need (e.g. other projects in {mem.work}) and run 2.2 again') from None
                print(f'⚠️ skipping unreadable checkpoint {archive.name}: {exc}')
                continue
            if marker.exists():
                print(f'· finishing the restore of {proj} that was interrupted')
            marker.write_text(archive.name)
            _swap_in(staging, proj)
            # input files; a dataset's .complete.json last, so it never marks a half-moved dataset as complete
            for f in sorted((staging / 'data').rglob('*'), key=lambda f: f.name == '.complete.json') if (staging / 'data').exists() else []:
                dst = mem.root / f.relative_to(staging)
                if f.is_file() and not dst.exists():  # never overwrite input files another project may use
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(f), dst)
            marker.unlink()
            print(f'✔ restored project {proj} from {archive.name}')
            want = proj
            break
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    else:
        raise RuntimeError('none of the checkpoints could be read')
names = mem.projects(valid_only=True)
if not names:
    stuck = sorted(m.stem for m in (mem.work / '.notebook').glob('*.restoring'))
    if stuck:
        raise RuntimeError(f"the restore of {', '.join(stuck)} was interrupted: run 2.2 again with RESTORE_FROM_DRIVE to finish it")
    raise RuntimeError(f'no projects in {mem.work}: the Colab disk was reset (a disconnect or a new runtime). Tick '
                       'RESTORE_FROM_DRIVE to unpack a checkpoint saved by 10.1, or start again in section 1.')
if not want and mem.project in names:
    want = mem.project
    print(f'· {want} is already open (enter PROJECT_TO_OPEN to open another project)')
name = want or max(names, key=lambda n: os.path.getmtime(mem.work / f'params_{n}.toml'))
mem.use(name)
DATA = mem.state().get('dataset', DATA)
S = mem.S
mem.status()
print(f"settings: thresholds {S['thres_low']}/{S['thres_high']} (3.1) · aperture {S['aperture_index']} · NUM_PSI {S['num_psi']}, "
      f"filter {S['distance_filter_type']} {S['distance_filter_cutoff_freq']:g}/{S['distance_filter_order']}, "
      f"NLSA_TUNE {S['nlsa_tune']}, CON_ORDER_RANGE {S['con_order_range']}, FPS {S['nlsa_fps']:g} (4.0) · "
      f"STATES {S['states_per_coord']} (7.2) · PATH_WIDTH {S['width_1D']} (7.3)")
if S['anchors']:
    print(f"anchors (6.2): {mem.anchor_string()}" + (f" · removed: {', '.join(str(t + 1) for t in S['trash'])}" if S['trash'] else ''))
gone = [f'{f} (needed for {why})' for k, why in mem.NEEDED_FOR.items() for f in mem.missing_inputs((k,))]
if gone:
    print(f"⚠️ input files are missing: {'; '.join(gone)}. To get them back, {mem.inputs_hint()}")

## 3 · Projection directions

Particles are grouped by viewing direction into bins on a half-sphere. Directions on the other half are mirrored
onto it. Every bin with at least `PRD_THRES_LOW` particles becomes a **projection direction (PD)**, and at most
`PRD_THRES_HIGH` particles per PD are used. Re-run this cell until the map looks reasonable: PDs should cover the
populated views, each with enough particles. Neighbouring PDs form connected groups (*clusters*); each cluster
later needs at least one anchor.

In [ ]:
#@title 3.1 · Bin the particles and set the PD thresholds { display-mode: "form" }
PRD_THRES_LOW = 0  #@param {type:"integer"}
PRD_THRES_HIGH = 0  #@param {type:"integer"}
#@markdown Minimum particles for a bin to become a PD, and the maximum used per PD. `0` keeps the project's current
#@markdown values (100 and 2000 for a new project).
APERTURE_INDEX_OVERRIDE = 0  #@param {type:"integer"}
#@markdown `0` keeps the project's current aperture index (the one 2.1 created it with, unless it was changed here).
TESSELLATION = "keep"  #@param ["keep", "lovisolo_silva", "fibonacci"]
CONFIRM_REBINNING = False  #@param {type:"boolean"}
#@markdown Re-binning renumbers the PDs, so once section 4 has results a change is only made with this box ticked,
#@markdown and the results of sections 4–8 are then deleted.
SHOW_3D_SPHERE = False  #@param {type:"boolean"}
res = mem.bridge('bin-particles', toml=mem.toml, low=int(PRD_THRES_LOW), high=int(PRD_THRES_HIGH),
                 aperture_index=int(APERTURE_INDEX_OVERRIDE), tess_type='' if TESSELLATION == 'keep' else TESSELLATION,
                 confirm=bool(CONFIRM_REBINNING))
changes = ', '.join(f'{k} {a} → {b}' for k, (a, b) in res['changes'].items())
if res['refused']:
    print(f'⚠️ not changed: {changes}. This project already has results from section 4; tick CONFIRM_REBINNING to '
          're-bin anyway (sections 4–8 are then deleted and must be run again). Showing the current binning.')
elif res['cleaned']:
    mem.save_state(redo={})
    if res.get('renumbered', True):
        print(f"ℹ️  {changes or 'finished a re-binning that had been stopped'}: the PDs were renumbered, so the results of "
              'sections 4–8 were deleted')
    else:
        print(f"ℹ️  {changes + ': ' if changes else ''}the binning is the one from before a re-binning that was stopped, but "
              'that re-binning had already deleted the results of sections 4–8: run section 4 again')
elif changes:
    print(f'✔ {changes}')
elif res.get('resumed'):
    print('ℹ️  finished a re-binning that had been stopped')
mem.refresh()
mem.show_binning()
if SHOW_3D_SPHERE:
    mem.show_sphere_3d()

## 4 · Manifold embedding of every PD

This is the compute-heavy part, run independently for every PD:

1. **Distances** (`calc-distance`): CTF-aware distances between all images of the PD, after in-plane alignment and filtering.
2. **Embedding** (`manifold-analysis`): a diffusion map of those distances. Its leading eigenvectors (ψ1, ψ2, …) are
   candidate conformational coordinates.
3. **NLSA** (`psi-analysis`): nonlinear Laplacian spectral analysis along each ψ denoises the images and orders them.
4. **Movies** (`nlsa-movie`): one 2D movie per ψ, which you inspect in sections 5–6.

The steps resume where they stopped: after a crash or a runtime restart, re-run 4.1 with `ONLY_MISSING_PDS`
ticked. On real data this section takes hours, so save a checkpoint (10.1) when it finishes.

In [ ]:
#@title 4.0 · Embedding parameters (optional) { display-mode: "form" }
#@markdown The defaults work for most data. Brackets list what a change makes 4.1 recompute.
NUM_PSI = 8  #@param {type:"slider", min:1, max:8, step:1}
#@markdown Number of eigenvectors ψ analysed and turned into movies [movies, sections 7–8; NLSA too, of every ψ, for
#@markdown PDs that have no current results for the added ψ].
DISTANCE_FILTER = "Butter"  #@param ["Butter", "Gauss"]
FILTER_CUTOFF = 0.5  #@param {type:"number"}
FILTER_ORDER = 8  #@param {type:"integer"}
#@markdown Low-pass filter applied before computing distances; cutoff as a fraction of Nyquist [all of 4, anchors, 7–8].
NLSA_TUNE = 3  #@param {type:"integer"}
#@markdown Diffusion-map kernel tuning: the kernel width is NLSA_TUNE × the automatic one, so larger = wider kernel
#@markdown [embedding, NLSA, movies, anchors, 7–8].
CON_ORDER_RANGE = 50  #@param {type:"integer"}
#@markdown NLSA window ("concatenation order") = images in the PD ÷ CON_ORDER_RANGE, so 50 means 1/50 of the images and
#@markdown a larger value a shorter window. Keep PRD_THRES_LOW ≥ 2 × CON_ORDER_RANGE [NLSA, movies, anchors, 7–8].
MOVIE_FPS = 5  #@param {type:"number"}
#@markdown Frame rate of the ψ movies [movies only].
CONFIRM_CHANGES = False  #@param {type:"boolean"}
#@markdown Once section 4 has results, changes are only applied with this box ticked (so re-running the cell on a
#@markdown re-opened project never replaces its settings with the defaults by accident).
values = dict(num_psi=int(NUM_PSI), distance_filter_type=DISTANCE_FILTER, distance_filter_cutoff_freq=float(FILTER_CUTOFF),
              distance_filter_order=int(FILTER_ORDER), nlsa_tune=int(NLSA_TUNE), con_order_range=int(CON_ORDER_RANGE),
              nlsa_fps=float(MOVIE_FPS))
S = mem.refresh()
changes = {k: (S[k], v) for k, v in values.items() if (round(S[k], 6) if isinstance(v, float) else S[k]) != v}
has_results = bool(glob.glob(mem.P['dist'].format(p='*')))
if int(CON_ORDER_RANGE) * 2 > S['thres_low']:
    print(f"⚠️ CON_ORDER_RANGE {CON_ORDER_RANGE} is more than half the PD threshold {S['thres_low']} (3.1): NLSA may fail")
if not changes:
    print('✔ parameters unchanged')
elif has_results and not CONFIRM_CHANGES:
    print('⚠️ not changed (tick CONFIRM_CHANGES to apply): ' + ', '.join(f'{k} {a} → {b}' for k, (a, b) in changes.items()))
else:
    # The new values are saved first. The refresh then notices the change, marks what 4.1 must recompute and clears
    # what no longer holds (anchors, sections 7–8); if the cell is stopped in between, the next refresh does that.
    mem.bridge('set-params', toml=mem.toml, values={k: b for k, (a, b) in changes.items()})
    mem.refresh();

In [ ]:
#@title 4.1 · Run distances → embedding → NLSA → movies { display-mode: "form" }
RUN_DISTANCES = True  #@param {type:"boolean"}
RUN_EMBEDDING = True  #@param {type:"boolean"}
RUN_NLSA = True  #@param {type:"boolean"}
RUN_MOVIES = True  #@param {type:"boolean"}
ONLY_MISSING_PDS = True  #@param {type:"boolean"}
#@markdown Skip PDs whose results already exist (resume after an interruption). Untick to recompute everything.
NCPU = 0  #@param {type:"integer"}
#@markdown `0` = all CPUs (automatically reduced for the distance and NLSA steps if memory is short).
ncpu = NCPU or mem.ncpu
S = mem.refresh()
redo = mem.state().get('redo') or {}             # step -> time of the 4.0 change that requires it
redo = dict.fromkeys(redo, 0) if isinstance(redo, list) else dict(redo)
rebinned = S['stale']['_keys'].get('embedding') == 'binning'   # PDs renumbered since
steps = [('calc-distance', RUN_DISTANCES), ('manifold-analysis', RUN_EMBEDDING), ('psi-analysis', RUN_NLSA),
         ('nlsa-movie', RUN_MOVIES)]
dirty = set()            # PDs recomputed by an earlier step in this run: later steps must follow
recomputed = set()       # PDs whose embedding / NLSA (and so their ψ movies) were recomputed
ran = []                 # (step index, PDs) of the steps run so far
warned = set()           # unticked steps already reported as not current
def _anchor_nlsa_current(pd0, psi):   # NLSA re-run only because other ψ were added gives this ψ's movie unchanged
    f, base = mem.P['psi2'].format(p=pd0, k=psi - 1), mem.P['psi'].format(p=pd0)
    return (os.path.exists(f) and os.path.exists(base)
            and os.path.getmtime(f) >= max(os.path.getmtime(base), redo.get('psi-analysis', 0)))
anchors_current = {int(a) for a, k, s in mem.A['anchors'] if _anchor_nlsa_current(int(a), int(k))}
t_start = time.time()
for i, (step, wanted) in enumerate(steps):
    if not wanted:
        continue
    if not ONLY_MISSING_PDS or rebinned:
        todo = set(range(S['n_pd']))
    else:   # outputs older than a pending 4.0 change count as missing, so an interrupted recompute resumes
        todo = set(mem.bridge('incomplete', toml=mem.toml, step=step, since=redo.get(step, 0))['prds']) | dirty
    behind = {}              # unticked earlier steps whose results are not current (PDs recomputed before them count)
    for j, (b, w) in enumerate(steps[:i]):
        if not w and b not in warned:
            outdated = set(mem.bridge('incomplete', toml=mem.toml, step=b, since=redo.get(b, 0))['prds'])
            outdated |= set().union(*[t for k, t in ran if k < j])
            if outdated:
                behind[b] = outdated
    if not todo:
        warned |= set(behind)
        print(f'✔ {step}: all {S["n_pd"]} PDs already done' + (
            ' for their current input, but ' + ', '.join(f'{b} ({len(o)} PD(s))' for b, o in behind.items())
            + (' have no current result (their boxes are unticked)' if len(behind) > 1 else
               ' has no current result (its box is unticked)') + ': run 4.1 with every box ticked' if behind else ''))
        redo.pop(step, None)
        mem.save_state(redo=redo)
        continue
    if step == 'calc-distance':   # the particle images (and mask) are read by the distances and by NLSA
        mem.require_inputs('stack', 'mask')
    elif step == 'psi-analysis' or (step == 'manifold-analysis' and RUN_NLSA):   # before an embedding that NLSA follows
        mem.require_inputs('stack')
    hit = {b: o & todo for b, o in behind.items() if o & todo}
    warned |= set(hit)
    if hit:
        print(f'⚠️ {step}: ' + ', '.join(f'{len(o)} PD(s) have no current {b} result' for b, o in hit.items())
              + (' (their boxes are unticked)' if len(hit) > 1 else ' (its box is unticked)')
              + ', so this is computed from outdated input and will be redone by the next full 4.1')
    if step != 'nlsa-movie' and not dirty:
        mem.stage('embedding', begin=True)       # results of 7–8 no longer match until they are recomputed
    if step == 'nlsa-movie':
        mem.bridge('remove-movies', toml=mem.toml, prds=sorted(todo))   # nlsa-movie skips PDs that have a class_avg.png
    n_workers = ncpu
    if step in ('calc-distance', 'psi-analysis'):         # peak memory ≈ 64 bytes × images × pixels² per worker
        n_workers = mem.workers_for(64 * S['max_images'] * S['box'] ** 2 / 1e9, ncpu, step)
    args = [step] + (['--prds', ','.join(map(str, sorted(todo)))] if len(todo) < S['n_pd'] else [])
    print(f'▶ {step}: {len(todo)} PD(s)')
    mem.cli(*args, ncpu=n_workers)
    ran.append((i, todo))
    if step != 'nlsa-movie':
        dirty |= todo
    if step == 'manifold-analysis':
        recomputed |= todo
    elif step == 'psi-analysis':
        recomputed |= todo - anchors_current
    redo.pop(step, None)
    mem.save_state(redo=redo)
S = mem.refresh()
if S['done']['calc_distance'] and S['done']['manifold_analysis'] and S['done']['psi_analysis'] and (dirty or not S['done']['embedding']):
    mem.stage('embedding')
    S = mem.refresh()
stale_anchors = [a for a in mem.A['anchors'].tolist() if a[0] in recomputed]
if stale_anchors:   # their movies may now run the other way
    keep = [[int(a), int(k), 'FWD' if s > 0 else 'REV'] for a, k, s in mem.A['anchors'] if a not in recomputed]
    mem.bridge('set-anchors', toml=mem.toml, anchors=keep, trash=[int(t) for t in mem.A['trash']], check=False)
    print('⚠️ the ψ movies of anchor PD(s) ' + ', '.join(str(a + 1) for a, _, _ in stale_anchors) + ' were recomputed and '
          'may now run the other way: those anchors were removed. Check the movies and set them again in 6.2.')
    S = mem.refresh()
mem.status()
print(f'total {(time.time() - t_start) / 60:.1f} min')
if S['done'].get('nlsa_movie'):
    top = int(mem.pd_table().iloc[0].PD)
    display(HTML(f'<b>ψ movies of the most populated PD ({top})</b><br>' + mem.movies_html(top, width=120)))
    if time.time() - t_start > 1800:
        print('💾 that took a while: save a checkpoint now with 10.1 (it can be run at any time)')

## 5 · Explore the projection directions

For each PD, look for a ψ whose movie shows a **smooth, physically plausible motion** (domains moving, not just
noise flickering or the whole particle shifting or rotating). The class average shows what the PD looks like. The
eigenvalue spectrum shows how many ψ carry signal. The bandwidth plot is the diffusion-map kernel fit, zoomed on its
bend: a good plot shows one smooth S-shaped rise between two flat plateaus. The orange line is the fitted midpoint;
the green dotted line is the kernel width ManifoldEM uses (NLSA_TUNE² × the midpoint's ε). The tanh is only an
approximate model, so offsets of a few tenths at the knees are normal (every PD of the demo has them); be suspicious
of a missing plateau, several steps or a bend at the edge. The embedding ψ1–ψ2 is coloured by in-plane rotation: if the colour follows the curve, the "motion"
is just residual rotation.

In [ ]:
#@title 5.1 · Overview of all PDs { display-mode: "form" }
SHOW_TOP = 15  #@param {type:"integer"}
mem.refresh()
display(HTML(mem.pd_table(top=SHOW_TOP or None).to_html(index=False)))
mem.show_anchor_map(title='PDs on the half-sphere')

In [ ]:
#@title 5.2 · PD browser { display-mode: "form" }
SHOW_PD = 0  #@param {type:"integer"}
#@markdown `0` = interactive browser (slider + list sorted by particle count); a number shows that PD only.
mem.refresh()
if SHOW_PD:
    mem.show_pd(int(SHOW_PD))
else:
    mem.pd_browser()

## 6 · Anchors: tell ManifoldEM which motion to follow

ManifoldEM propagates *one* conformational coordinate across all PDs with optical flow and belief propagation. It
needs **anchors**: PDs where you state which ψ movie shows the motion of interest and in which direction it runs.

* Give **at least one anchor per cluster** (clusters are the colours on the map). PDs in a cluster without an
  anchor get no coordinate. Removing PDs can split a cluster: the 6.2 map then colours the pieces separately and
  rings every PD of a piece without an anchor in red.
* The **sense** must be consistent: if a domain moves *up* in your `FWD` anchor's movie, every other `FWD` anchor
  must show the same motion running the same way. Mark an anchor `REV` if its movie shows the motion backwards. The
  sign of each PD's ψ is arbitrary, which is why only you can decide this across clusters.
* Good anchors have many particles and a clean, unambiguous movie. More anchors make the propagation more reliable.
* You can also **remove** PDs whose movies are junk; they are excluded from the landscape and the volumes.

For the RyR1 tutorial data, the tutorial's anchors (0-based PDs 52 FWD and 45 REV in its text) are `53:1:FWD, 46:1:REV`
here. That only holds with the tutorial's binning: 3.1 must report 53 PDs out of 2005 bins (resolution 5 Å,
aperture 4, thresholds 100/2000). Otherwise pick your own in 6.1, and check the ψ movies either way.

In [ ]:
#@title 6.1 · Candidate anchors: the most populated PDs of every cluster { display-mode: "form" }
PDS_PER_CLUSTER = 2  #@param {type:"integer"}
PSI_TO_SHOW = "1, 2, 3"  #@param {type:"string"}
MAX_CLUSTERS = 6  #@param {type:"integer"}
#@markdown Movies are shown for the largest clusters only (to keep the notebook small); browse the others in 5.2.
mem.refresh()
psis = [int(p) for p in PSI_TO_SHOW.replace(' ', '').split(',') if p]
mem.anchor_candidates(per_cluster=int(PDS_PER_CLUSTER), psis=[p for p in psis if 1 <= p <= mem.S['num_psi']],
                      max_clusters=int(MAX_CLUSTERS))

In [ ]:
#@title 6.2 · Set anchors and removed PDs { display-mode: "form" }
ANCHORS = "keep"  #@param {type:"string"}
#@markdown Comma-separated `PD:ψ:sense` entries, e.g. `53:1:FWD, 46:1:REV` (PD and ψ numbers as shown in this notebook).
#@markdown `keep` = the project's current anchors (or `auto` if it has none yet). `auto` = the most populated PD,
#@markdown ψ1, `FWD`: fine for one cluster (like the demo); with several clusters you have to choose the senses yourself.
REMOVE_PDS = "keep"  #@param {type:"string"}
#@markdown PDs to exclude, e.g. `7, 12-14`; empty = none; `keep` = the project's current choice.
mem.refresh()
mem.require('embedding', 'run section 4 (4.1)')
if not mem.S['done']['nlsa_movie']:
    raise RuntimeError('the ψ movies are missing: run 4.1 (with RUN_MOVIES) first')
anchors = mem.parse_anchors(ANCHORS)
trash = [int(t) for t in mem.A['trash']] if REMOVE_PDS.strip().lower() == 'keep' else mem.parse_pd_list(REMOVE_PDS)
res = mem.bridge('set-anchors', toml=mem.toml, anchors=anchors, trash=trash)
mem.refresh()
mem.show_anchor_map()
print(('✔ ' if res['changed'] else '✔ unchanged · ') + f'anchors: {mem.anchor_string()}'
      + (f" · removed PDs: {', '.join(str(t + 1) for t in trash)}" if trash else ''))
for comp in res['unanchored_components']:
    print(f'⚠️ no anchor among PDs {comp}: they will get no conformational coordinate')
if res['changed'] and 'cc' in mem.S['stale']:
    print('ℹ️  the anchors changed: run 7.1 again (sections 7–8 are now out of date)')

## 7 · Conformational coordinate and energy landscape

* **7.1** finds, for every PD, the ψ and the sense that match the anchors. It computes optical flow between the
  movies of neighbouring PDs and runs belief propagation over the graph of PDs. A mix of ▲ FWD and ▼ REV is normal:
  the sign of each ψ is arbitrary, and the sense only records whether it runs with or against the common direction.
  What matters is that few PDs are left unassigned (✕).
* **7.2** re-runs NLSA with those choices, places every image on the common coordinate τ ∈ [0, 1] and histograms the
  occupancy. Its negative logarithm is the free-energy profile, in units of k<sub>B</sub>T.
* **7.3** sorts the images into `STATES` bins along τ and writes one image stack + STAR file per state. These are
  ManifoldEM's NLSA-reconstructed (denoised, CTF-corrected) snapshots with the PD's average orientation, not subsets
  of your raw particles. Section 8 back-projects them into one volume per state to visualise the motion.

**How to read the landscape.** The demo's true profile is flat. Counting noise alone gives a few tenths of k<sub>B</sub>T,
and the extra structure you will see (about 1 k<sub>B</sub>T, a pile-up in the first and last states) comes from NLSA
edge effects and from ManifoldEM's τ not being exactly linear in the real motion. On real data, trust features that are
clearly larger than that, away from the ends, and that persist when you change the anchors or `STATES`.

In [ ]:
#@title 7.1 · Find the conformational coordinate (belief propagation) { display-mode: "form" }
BP_NUM_PSI = "auto"  #@param ["auto", "all", "1", "2", "3", "4", "5", "6", "7", "8"]
#@markdown How many eigenvectors belief propagation may choose from in each PD: `all` is ManifoldEM's standard
#@markdown behaviour; `1` keeps ψ1 everywhere and only propagates the sense (robust when ψ1 clearly shows the motion in
#@markdown most PDs). `auto` = `1` for the synthetic demo, `all` otherwise.
RECOMPUTE = False  #@param {type:"boolean"}
#@markdown Run again even if the coordinate is up to date for the current anchors.
if BP_NUM_PSI == 'auto':
    bp_num_psi = int(mem.state().get('dataset', {}).get('suggest', {}).get('bp_num_psi', 0))
else:
    bp_num_psi = 0 if BP_NUM_PSI == 'all' else int(BP_NUM_PSI)     # 0 = all of ψ1..ψ<NUM_PSI>
mem.refresh()
mem.require('embedding', 'run section 4 (4.1)')
led_bp = mem.S['done']['find_ccs'] and mem.bridge('ledger', toml=mem.toml).get('cc', {}).get('info', {}).get('bp_num_psi')
if mem.S['done']['find_ccs'] and not RECOMPUTE and led_bp == (bp_num_psi or mem.S['num_psi']):
    print('✔ already up to date for these anchors (tick RECOMPUTE to run it again)')
else:
    res = mem.bridge('find-ccs', toml=mem.toml, ncpu=mem.ncpu, bp_num_psi=bp_num_psi)
    how = f"belief propagation over ψ1–ψ{res['bp_num_psi']}" if res['bp_ran'] else 'every PD is an anchor, no propagation needed'
    print(f"✔ {res['n_assigned']} of {res['n_pd']} PDs assigned in {res['seconds']:.0f} s ({how})")
    if res['anchors_shifted']:
        print('ℹ️  this ManifoldEM version reads Anchor.CC as 0-based during belief propagation; the notebook compensated, '
              'so your anchors refer to the ψ numbers shown here')
    if res['changed']:
        print('ℹ️  the coordinate changed: run 7.2–8.3 again')
mem.refresh()
mem.show_cc();

In [ ]:
#@title 7.2 · Probability and free-energy landscape { display-mode: "form" }
STATES = 0  #@param {type:"integer"}
#@markdown Number of bins along the coordinate. Section 8 makes STATES − PATH_WIDTH + 1 volumes (STATES with the
#@markdown default PATH_WIDTH of 1). `0` keeps the project's current value (50 for a new project). It cannot exceed
#@markdown the number of movie frames computed in section 4 (normally 50).
RECOMPUTE = False  #@param {type:"boolean"}
S = mem.refresh()
mem.require('find_ccs', 'run 7.1')
states = int(STATES) or S['states_per_coord']
if S['nlsa_frames'] and states > S['nlsa_frames']:
    raise ValueError(f"STATES can be at most {S['nlsa_frames']}, the number of NLSA frames computed in section 4")
if states < S['width_1D']:
    print(f"⚠️ STATES {states} is below the current PATH_WIDTH {S['width_1D']}: choose PATH_WIDTH ≤ {states} in 7.3")
if S['done']['probability_landscape'] and states == S['states_per_coord'] and not RECOMPUTE:
    print('✔ already up to date (tick RECOMPUTE to run it again)')
else:
    mem.require_inputs('stack')   # probability-landscape re-runs NLSA, which reads the particle images
    mem.stage('landscape', begin=True)
    mem.bridge('set-params', toml=mem.toml, values=dict(states_per_coord=states))
    mem.cli('probability-landscape', ncpu=mem.workers_for(64 * S['max_images'] * S['box'] ** 2 / 1e9, what='probability-landscape'))
    mem.stage('landscape')
mem.refresh()
gt = mem.state().get('dataset', {}).get('ground_truth')
GT_ROWS = mem.bridge('ground-truth', toml=mem.toml, gt_csv=gt)['rows'] if gt and os.path.exists(gt) else None
mem.show_landscape(GT_ROWS);

In [ ]:
#@title 7.3 · Write per-state image stacks (3D trajectories) { display-mode: "form" }
PATH_WIDTH = 0  #@param {type:"slider", min:0, max:5, step:1}
#@markdown Number of neighbouring bins pooled into each state: more images per volume, less resolution along τ, and
#@markdown one state fewer per extra bin (STATES − PATH_WIDTH + 1 states, so PATH_WIDTH can be at most STATES). `0`
#@markdown keeps the project's current value (1 for a new project).
RECOMPUTE = False  #@param {type:"boolean"}
S = mem.refresh()
mem.require('probability_landscape', 'run 7.2')
width = int(PATH_WIDTH) or S['width_1D']
if width > S['states_per_coord']:
    raise ValueError(f"PATH_WIDTH {width} is larger than STATES {S['states_per_coord']} (7.2): ManifoldEM makes STATES − "
                     f"PATH_WIDTH + 1 states, so set PATH_WIDTH to at most {S['states_per_coord']} or raise STATES in 7.2")
if S['done']['trajectory'] and width == S['width_1D'] and not RECOMPUTE:
    print('✔ already up to date (tick RECOMPUTE to run it again)')
else:
    mem.stage('trajectory', begin=True)
    mem.bridge('set-params', toml=mem.toml, values=dict(width_1D=width))
    had_volumes = S['done']['volumes']
    mem.bridge('clean', toml=mem.toml, what=['trajectory', 'volumes'])   # stale states would be mixed in otherwise
    if had_volumes:
        print('ℹ️  the volumes of section 8 were deleted: run 8.1 (then 8.2–8.3) to rebuild them from the new stacks')
    mem.cli('trajectory', ncpu=mem.workers_for(2 * 20 * 4 * S['max_images'] * S['box'] ** 2 * width / 1e9,
                                               what='trajectory'))
    mem.stage('trajectory')
mem.refresh()
stars = sorted(glob.glob(os.path.join(mem.P['bin_dir'], '*.star')))
counts = [sum(1 for l in open(s) if '@' in l) for s in stars]
print(f'✔ {len(stars)} states (STATES {mem.S["states_per_coord"]}, PATH_WIDTH {mem.S["width_1D"]}), '
      f'{min(counts)}–{max(counts)} images each → {mem.P["bin_dir"]}')

## 8 · 3D volumes along the coordinate

ManifoldEM's `utility mrcs2mrc` reconstructs every state with RELION's `relion_reconstruct`. Colab can install a
CPU build of RELION from Ubuntu's package archive in about a minute. The built-in NumPy reconstructor (direct
Fourier inversion, validated against RELION) is a fallback that needs no installation. The volumes are made from the
NLSA snapshots of 7.3, so they show the motion at modest resolution; they are not refined maps.

In [ ]:
#@title 8.1 · Reconstruct one volume per state { display-mode: "form" }
METHOD = "RELION relion_reconstruct (apt install, ~1 min)"  #@param ["RELION relion_reconstruct (apt install, ~1 min)", "NumPy (built in, no installation)"]
RECOMPUTE = False  #@param {type:"boolean"}
S = mem.refresh()
mem.require('trajectory', 'run 7.3')
stars = sorted(glob.glob(os.path.join(mem.P['bin_dir'], '*.star')))
box = S['box']
use_relion = METHOD.startswith('RELION')
if S['done']['volumes'] and not RECOMPUTE:
    print(f"✔ already up to date: {S['done']['volumes']} volumes (tick RECOMPUTE to rebuild them, e.g. with the other METHOD)")
else:
    mem.stage('volumes', begin=True)
    mem.bridge('clean', toml=mem.toml, what=['volumes'])
    if use_relion and not shutil.which('relion_reconstruct'):
        print('· installing RELION from the Ubuntu archive ...')
        try:
            apt = [] if os.geteuid() == 0 else ['sudo']
            mem.run(apt + ['apt-get', '-qq', 'update'], echo=False)
            mem.run(apt + ['env', 'DEBIAN_FRONTEND=noninteractive', 'apt-get', '-qq', 'install', '-y', 'relion'], echo=False)
        except Exception as exc:
            print(f'⚠️ RELION could not be installed ({exc}); using the NumPy reconstructor instead')
            use_relion = False
    if use_relion:
        try:
            test = mem.bridge('relion-selftest')
        except RuntimeError as exc:
            test = dict(ok=False, cc=float('nan'), error=str(exc))
        if test['ok']:
            print(f"✔ relion_reconstruct self-test: correlation with the phantom {test['cc']:.4f}")
        else:
            print(f"⚠️ relion_reconstruct failed its self-test ({test.get('error', test)}); using the NumPy reconstructor instead")
            use_relion = False
    if use_relion:   # measured: ~0.6 GB per job at 128 px, ~4.7 GB at 256 px
        mem.cli('utility', 'mrcs2mrc', ncpu=mem.workers_for(0.15 + 4.7 * (box / 256) ** 3, what='reconstruction'))
        missing = [s for s in stars if not os.path.exists(os.path.join(mem.P['vols_dir'], os.path.basename(s)[:-5] + '.mrc'))]
        for s in missing:   # manifold-cli hides RELION's messages; retry visibly
            print(f'· retrying {os.path.basename(s)}')
            mem.run(['relion_reconstruct', '--i', os.path.basename(s), '--o',
                     os.path.join(mem.P['vols_dir'], os.path.basename(s)[:-5] + '.mrc')], cwd=mem.P['bin_dir'], check=False)
    else:            # measured: ~1 GB per job at 128 px, ~6.3 GB at 256 px (pad 2); pad 1 above 256 px
        pad = 2 if box <= 256 else 1
        per_job = 0.2 + (9.5 if pad == 2 else 1.6) * (box / 256) ** 3
        mem.bridge('reconstruct', toml=mem.toml, ncpu=mem.workers_for(per_job, what='reconstruction'), pad=pad)
    if len(mem.volume_files(mem.P['vols_dir'])) == len(stars):
        mem.stage('volumes', info=dict(method='relion' if use_relion else 'numpy'))
mem.refresh()
VOLUME_FILES = mem.volume_files(mem.P['vols_dir'])
print(f'{"✔" if len(VOLUME_FILES) == len(stars) else "⚠️"} {len(VOLUME_FILES)} of {len(stars)} volumes in {mem.P["vols_dir"]}')

In [ ]:
#@title 8.2 · Denoise the volumes { display-mode: "form" }
GAUSSIAN_SIGMA_VOXELS = 1.0  #@param {type:"number"}
SVD_COMPONENTS = 0  #@param {type:"integer"}
#@markdown `SVD_COMPONENTS > 0` keeps only the leading components of the whole series of volumes (a strong denoiser for
#@markdown smooth motions; try 3–5). Results go to `postproc/denoise_notebook/`; the raw volumes are kept.
#@markdown (ManifoldEM's `utility denoise` only filters the first and last 5 states.)
mem.refresh()
mem.require('volumes', 'run 8.1')
VOLUME_FILES = mem.volume_files(mem.P['vols_dir'])
DENOISED_DIR = os.path.join(mem.P['out_dir'], 'postproc', 'denoise_notebook')
mem.stage('denoise', begin=True)
movie_dir = Path(mem.P['out_dir']) / 'postproc' / 'notebook'
if (movie_dir / 'open_in_chimerax.cxc').exists() and '../denoise_notebook/' in (movie_dir / 'open_in_chimerax.cxc').read_text():
    for f in ('open_in_chimerax.cxc', 'trajectory_projections.gif'):   # 8.3 made them from the denoised volumes
        (movie_dir / f).unlink(missing_ok=True)
    print('ℹ️  the movie and ChimeraX script of 8.3 showed the previous denoised volumes and were deleted: run 8.3 again')
shutil.rmtree(DENOISED_DIR, ignore_errors=True)
DENOISED_FILES = denoise_volumes(VOLUME_FILES, DENOISED_DIR, sigma=float(GAUSSIAN_SIGMA_VOXELS), svd_rank=int(SVD_COMPONENTS))
mem.stage('denoise', info=dict(sigma=float(GAUSSIAN_SIGMA_VOXELS), svd=int(SVD_COMPONENTS)))
print(f'✔ {len(DENOISED_FILES)} denoised volumes → {DENOISED_DIR}')

In [ ]:
#@title 8.3 · Watch the conformational change { display-mode: "form" }
SHOW = "denoised"  #@param ["denoised", "raw"]
FLIP_CONTRAST = "auto"  #@param ["auto", "yes", "no"]
#@markdown ManifoldEM's trajectory images have inverted contrast, so volumes often come out negative; `auto` fixes that.
ISO_LEVEL_SIGMA = 3.0  #@param {type:"number"}
ISOSURFACE_STATES = 10  #@param {type:"integer"}
mem.refresh()
mem.require('volumes', 'run 8.1')
kind = SHOW
if SHOW == 'denoised' and not mem.S['done']['denoise']:
    print('no up-to-date denoised volumes (8.2): showing the raw ones')
    kind = 'raw'
folder = os.path.join(mem.P['out_dir'], 'postproc', 'denoise_notebook') if kind == 'denoised' else mem.P['vols_dir']
files = mem.volume_files(folder)
flip = needs_flip(files, FLIP_CONTRAST)
proj = volume_projections(files, flip)
show_volume_strip(proj, title=f'{len(files)} states along the conformational coordinate ({kind} volumes)')
movie_dir = os.path.join(mem.P['out_dir'], 'postproc', 'notebook')
os.makedirs(movie_dir, exist_ok=True)
for f in ('open_in_chimerax.cxc', 'trajectory_projections.gif'):   # a stopped run must not leave a mismatched pair
    Path(movie_dir, f).unlink(missing_ok=True)
volume_movie(proj, os.path.join(movie_dir, 'trajectory_projections.gif'))
states = sorted(set(np.linspace(0, len(files) - 1, max(2, min(ISOSURFACE_STATES, len(files)))).round().astype(int)))
isosurface_movie(files, states, flip=flip, sigma=float(ISO_LEVEL_SIGMA), step=2 if mem.S['box'] <= 160 else 3)
cxc = write_chimerax_script(files, movie_dir)
print(f'Tip — the results zip (10.2) contains {os.path.relpath(cxc, mem.P["out_dir"])}: open it in ChimeraX to play '
      'the volumes as a series in the right order.')

## 9 · Python API (advanced)

Everything ManifoldEM computes is available through its documented `data_store` API (see `help(data_store)` and the
main README). The API lives in ManifoldEM's own environment, so start a cell with **`%%manifoldem`** to run it there,
with `np`, `plt`, `params`, `data_store`, `prds` (the projection directions), `Anchor` and `Sense` already loaded.
Printed output and matplotlib figures appear below the cell.

* Each `%%manifoldem` cell runs in a **fresh process**: variables do not carry over to the next cell, and changes to
  `prds` / `params` are kept only if the cell calls `prds.save()` / `params.save()`. To pass results on, write them to
  a file (e.g. `np.save` in the project folder).
* Indices in the API are **0-based** (PD *n* in this notebook is index *n − 1*), except `Anchor.CC`, which is the
  1-based ψ number (`Anchor(CC=1)` = ψ1, as in 6.2 and the GUI).
* Section 4 parameters saved here (`params.nlsa_tune = 4; params.save()`) are handled like a change in 4.0: the steps
  that depend on them are marked for 4.1, and results that no longer hold are cleared.

In [ ]:
%%manifoldem
top = int(np.argmax(prds.occupancy))                  # 0-based index of the most populated PD
prd = data_store.get_prd_data(top)
print(f'PD {top + 1} (index {top})')
print(prd)
print('embedding keys:', sorted(prd.psi_data.keys()))
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, img, title in zip(ax, (prd.raw_images[0], prd.transformed_images[0], np.fft.fftshift(prd.ctf_images[0])),
                         ('raw image', 'filtered + aligned', 'CTF (centred)')):   # CTFs are stored in FFT layout
    a.imshow(img, cmap='gray'); a.set_title(title); a.axis('off')

## 10 · Save and download

The Colab disk is wiped when the runtime is recycled. **10.1** packs the project (and optionally the input data) into
one archive on Google Drive, which **2.2** can restore in a later session. Tick its box and run it whenever you want a
checkpoint (Run all skips it). **10.2** downloads the results to your computer. Browser downloads are fine up to a
GB or two; use 10.1 for anything larger.

In [ ]:
#@title 10.1 · Save a checkpoint to Google Drive { display-mode: "form" }
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown Tick and run to save (Drive is mounted if needed). Unticked, the cell does nothing, so Run all never asks
#@markdown for access to your Drive.
DRIVE_FOLDER = "/content/drive/MyDrive/ManifoldEM"  #@param {type:"string"}
INCLUDE_INPUT_DATA = False  #@param {type:"boolean"}
#@markdown Include the STAR file and particle stack (and the synthetic ground truth), if they are on the Colab disk.
#@markdown Needed to continue the analysis in a new runtime.
WAIT_FOR_UPLOAD = False  #@param {type:"boolean"}
#@markdown Drive uploads in the background. Tick before closing or deleting the runtime: the cell then waits until
#@markdown Drive has the whole file (and unmounts Drive).
if not SAVE_TO_DRIVE:
    print('skipped: tick SAVE_TO_DRIVE to save a checkpoint to Google Drive (10.2 downloads the results instead)')
else:
    toml = mem.toml                                       # fails clearly if no project is open
    if IN_COLAB and not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
    members = [f'work/params_{mem.project}.toml', f'work/output/{mem.project}', f'work/.notebook/{mem.project}.json']
    if INCLUDE_INPUT_DATA:
        inputs = {mem.S['files'][k] for k in ('star', 'stack', 'volume', 'mask') if mem.S['files'][k]}
        inputs.add(mem.state().get('dataset', {}).get('ground_truth') or '')
        marker = Path(mem.S['files']['stack']).parent / '.complete.json'   # so 1.2 / 1.5 reuse the images after a restore
        if marker.exists():   # with every file it vouches for (e.g. a downsampled mask the project does not use)
            info = json.loads(marker.read_text())
            inputs |= {str(marker)} | {info[k] for k in ('star', 'stack', 'volume', 'mask') if info.get(k)}
        inputs |= {str(r) for r in (Path(i).resolve().with_name(f'.{Path(i).resolve().name}.version') for i in list(inputs) if i)
                   if r.exists()}   # 1.4's version records: 1.5 names its folders after them
        for f in sorted(i for i in inputs if i):
            if Path(f).resolve().is_relative_to(mem.root.resolve()):
                members.append(str(Path(f).resolve().relative_to(mem.root.resolve())))
            else:
                print(f'not included (outside {mem.root}): {f}')
    members = [m for m in members if (mem.root / m).exists()]
    archive = Path(DRIVE_FOLDER) / f'{mem.project}_{time.strftime("%Y%m%d-%H%M%S")}.tar.gz'
    part = archive.with_name(archive.name + '.part')       # 2.2 never picks up a half-written archive
    tar = ['tar', '-I', 'pigz'] if shutil.which('pigz') else ['tar', '-z']
    try:
        mem.run(tar + ['-cf', part, f'--exclude=work/output/{mem.project}/*.part', '-C', mem.root] + members)
        mem.run(['gzip', '-t', part], echo=False)
        part.rename(archive)
    finally:
        part.unlink(missing_ok=True)
    print(f'✔ {archive} ({_human(archive.stat().st_size)})')
    if IN_COLAB and WAIT_FOR_UPLOAD:
        from google.colab import drive
        drive.flush_and_unmount()
        print('✔ uploaded to Google Drive (Drive is now unmounted)')
    elif IN_COLAB:
        print('· Drive finishes the upload in the background: tick WAIT_FOR_UPLOAD before deleting the runtime')

In [ ]:
#@title 10.2 · Download the results { display-mode: "form" }
WHAT = "results (volumes, landscape, tables, movies)"  #@param ["results (volumes, landscape, tables, movies)", "whole project (large)"]
import zipfile, pandas as pd
S = mem.refresh()
done = S['done']
out = Path(mem.P['out_dir'])
export = out / 'postproc' / 'notebook'
export.mkdir(parents=True, exist_ok=True)
for name in ('conformational_coordinates.csv', 'landscape.csv'):
    (export / name).unlink(missing_ok=True)
mem.pd_table().to_csv(export / 'projection_directions.csv', index=False)
if done['find_ccs']:
    cc = _read_pickle(mem.P['CC_file'])
    pd.DataFrame(dict(PD=np.arange(1, S['n_pd'] + 1), psi=cc['psinums'][0] + 1, sense=cc['senses'][0])).to_csv(
        export / 'conformational_coordinates.csv', index=False)   # psi 0 = unassigned
if done['probability_landscape']:
    h = np.asarray(_read_pickle(mem.P['traj'])['hUn'], dtype=float)
    with np.errstate(divide='ignore'):
        pd.DataFrame(dict(state=np.arange(1, len(h) + 1), particles=h, free_energy_kT=-np.log(h / h.max()))).to_csv(
            export / 'landscape.csv', index=False)
skipped = [label for key, stage, label in mem.STEPS if stage and not done.get(key)]
zip_path = mem.root / f'{mem.project}_{"results" if WHAT.startswith("results") else "project"}.zip'
if WHAT.startswith('results'):   # only up-to-date results; traj_*_group_*.pkl are large intermediate image lists
    keep_dirs = {'vols': done['volumes'], 'denoise_notebook': done['denoise'], 'notebook': True}
    picks = [Path(mem.toml)] + [p for p in out.rglob('*') if p.is_file() and (
        ('postproc' in p.parts and keep_dirs.get(p.parent.name, False)
         and (done['volumes'] or p.suffix not in ('.gif', '.cxc')))     # the movie and script show the volumes
        or (p.parent.name == 'traj' and '_group_' not in p.name and done['probability_landscape'])
        or (p.name in ('comp_psi_sense_nodes.txt', 'CC_file.pkl') and done['find_ccs']))]
    cxc, gif = export / 'open_in_chimerax.cxc', export / 'trajectory_projections.gif'
    line = next((l for l in cxc.read_text().splitlines() if l.startswith('open ')), '') if cxc.exists() else ''
    shown = [(export / t).resolve() for t in line.split()[1:] if t.endswith('.mrc')]
    in_zip = {p.resolve() for p in picks}
    if (cxc.exists() or gif.exists()) and not (shown and gif.exists() and all(f in in_zip for f in shown)
                                               and gif.stat().st_mtime >= max(f.stat().st_mtime for f in shown)):
        picks = [p for p in picks if p not in (cxc, gif)]    # they show volumes that are gone or out of date
        skipped.append('movie and ChimeraX script of 8.3 (run 8.3 again)')
    if skipped:
        print('not included (missing or out of date): ' + ', '.join(skipped))
else:
    picks = [Path(mem.toml)] + [p for p in out.rglob('*') if p.is_file()]
total = sum(p.stat().st_size for p in picks)
if IN_COLAB and total > 2e9:
    raise RuntimeError(f'these files add up to {_human(total)}, too much for a browser download: save them '
                       'with 10.1 (Google Drive), or download single files from the Files panel on the left')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in picks:
        z.write(p, p.relative_to(mem.work))
print(f'✔ {zip_path} ({_human(zip_path.stat().st_size)})')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

## 11 · Troubleshooting

| Symptom | What to do |
|---|---|
| A cell says *run … first* | Run the cells in order. After a restart, run section 0 and then 2.2. |
| `NameError: mem` | Section 0 has not been run in this session. |
| ManifoldEM reports a missing STAR column | See 1.6. Export the STAR file again from RELION / cryoSPARC with angles, defocus and optics. |
| *the STAR file has N particles but the stack has M images* | ManifoldEM pairs STAR row *i* with stack image *i*. Write a single stack in STAR order (`relion_stack_create`). |
| Very few PDs | Lower `PRD_THRES_LOW` or increase the aperture index (3.1). Keep `PRD_THRES_LOW ≥ 2 × CON_ORDER_RANGE`. |
| Session crashes ("used all available RAM") | Run section 0 and 2.2, set `NCPU` to 1 in 4.1 and resume with `ONLY_MISSING_PDS`. If one PD alone does not fit: lower `PRD_THRES_HIGH` (3.1, re-bins: section 4 starts over), or downsample (1.5) and create a new project (2.1). |
| Kernel restarted or crashed during section 4 | Files are kept: run section 0, 2.2, then 4.1 with `ONLY_MISSING_PDS` ticked (finished PDs are kept). |
| Runtime disconnected and 2.2 says *no projects* | The Colab disk was reset. Run section 0, then 2.2 with `RESTORE_FROM_DRIVE` (your last 10.1 checkpoint). If it reports missing inputs, do what it says (1.1, the dataset cell and, if you downsampled, 1.5, all with the same settings), then continue (e.g. 4.1 with `ONLY_MISSING_PDS`). Run all also works with `RESTORE_FROM_DRIVE` ticked in 2.2: it replaces the empty project that 2.1 makes. Work after the last checkpoint is lost. |
| A cell says *run … first (its results are out of date: …)* | Something it depends on changed since: re-run the step it names (the status table in 2.2 / 4.1 lists what is out of date). |
| A cell says *already up to date* | Nothing it depends on changed; tick `RECOMPUTE` to run it anyway (e.g. 8.1 with the other `METHOD`). |
| A cell says *not changed … tick CONFIRM_…* | 3.1 and 4.0 protect existing results from accidental changes (e.g. form defaults on a re-opened project). Tick the box only if you really want the new values. |
| Many PDs unassigned after 7.1 | Add an anchor to every cluster (6.2); remove junk PDs; try `BP_NUM_PSI = 1` if ψ1 shows the motion almost everywhere. |
| Landscape looks like noise | Check the anchors' senses: one inverted anchor scrambles the coordinate. Look at more PDs in 5.2. |
| RELION fails to install or to run | 8.1 falls back to the NumPy reconstructor automatically. |
| Volumes look inverted or empty | `FLIP_CONTRAST` / `ISO_LEVEL_SIGMA` in 8.3; for full control open `postproc/notebook/open_in_chimerax.cxc` (in the 10.2 zip) in ChimeraX. |

**What the notebook does differently from typing the CLI by hand** (all ManifoldEM v0.4.0 behaviour; check
again on newer versions):
* ManifoldEM runs in a separate Python 3.12 environment with NumPy < 2 (its `nptyping` dependency and some code do not
  support NumPy 2).
* Parameters are written into the project's `.toml` (`manifold-cli <step> --param` flags other than for `init` and
  `threshold` are not persistent).
* Belief propagation in v0.4.0 reads `Anchor.CC` as 0-based while the GUI and the tutorial use 1-based ψ numbers; 7.1
  compensates, so `Anchor(CC=1)` means ψ1 here as in the GUI.
* `find-ccs` with no anchors silently does nothing. 6.2 / 7.1 check for that, and they warn about clusters without
  an anchor.
* `utility mrcs2mrc` hides RELION errors, so 8.1 checks that every volume was written. `utility denoise` only
  filters the first and last states, so 8.2 denoises all of them and keeps the pixel size.

**Links:** [ManifoldEM on GitHub](https://github.com/flatironinstitute/ManifoldEM) ·
[tutorial](https://github.com/flatironinstitute/ManifoldEM/blob/main/tutorial/README.md) ·
[issues](https://github.com/flatironinstitute/ManifoldEM/issues) ·
Dashti *et al.*, PNAS 2014 ([doi](https://doi.org/10.1073/pnas.1419276111)) ·
Dashti *et al.*, Nat. Commun. 2020 ([doi](https://doi.org/10.1038/s41467-020-18403-x)) ·
Seitz *et al.*, IEEE TCI 2022 ([doi](https://doi.org/10.1109/TCI.2022.3174801)) ·
Ojha *et al.*, Acta Cryst. D 2025 ([doi](https://doi.org/10.1107/S2059798325001469))